# NEXUS-II v1.9.2 — Stage 06 Confirmation Completion
## RC13C FINAL confirmation-completion amendment

### Purpose

RC13B completed and cryptographically froze discovery (3000–3019) and generated valid confirmation observations for 3100–3126, then stopped before claim evaluation when the **proposed** Q bank for 3127 violated the preregistered physical-independence support gate.

RC13C does **not** refit discovery, alter H6-B/H6-W/H6-A, inspect partial confirmation outcomes to tune the method, or rerun completed confirmation tasks. It makes one prospective protocol amendment:

> A complete matched-Q bank is sampled by deterministic whole-bank rejection sampling from the same Gaussian proposal distribution, accepting the first candidate whose maximum pairwise |cosine| across Q_A/Q_T/Q_B and all frozen target modules is ≤ 0.15.

The accepted Q-geometry plan for **all 30 confirmation task IDs is frozen before any RC13B confirmation response table is deserialized**. Existing 3100–3126 must replay as accepted candidate 0. Only missing tasks 3127–3129 are then generated. The unchanged RC13B discovery model and unchanged confirmatory estimator are evaluated once the full n=30 confirmation cohort exists.

### Primary result

RC13C produces the evaluable Stage-06 H6-B / H6-W / H6-A result. The aborted RC13B stop remains preserved as parent evidence.


In [ ]:
# 0. Environment, deterministic execution, durable persistence, and RC13C amendment identity
import os, sys, json, math, time, random, hashlib, platform, copy, gc, zipfile, shutil, pickle, subprocess, warnings, itertools, marshal, types, re
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, List, Optional, Any

# Set before the first CUDA/cuBLAS operation.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import sklearn, scipy
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

# Provenance baseline: scientific fingerprinting must be invariant to unrelated functions/classes
# already present in a reused Colab runtime. Objects subsequently defined/overwritten by this
# notebook are fingerprinted; pre-existing ambient __main__ objects are not.
_S6_AMBIENT_MAIN_OBJECTS={
    name:obj for name,obj in globals().items()
    if ((isinstance(obj,types.FunctionType) or isinstance(obj,type)) and getattr(obj,'__module__',None)=='__main__')
}

PROJECT='NEXUS-II'
VERSION='1.9.2'
PROTOCOL_REVISION='NEXUS-II-v1.9.2-stage06-confirmation-completion-RC13C-FINAL-20260927'
IMPLEMENTATION_REVISION='NEXUS-II-v1.9.2-stage06-RC13C-r3-FULL-PARENT-EQUIVALENCE-NONZERO-Q-QA-MERGED-PROVENANCE-AND-REPORTING-HARDENED-20260927'
MASTER_SEED=20260926
PROFILE=os.environ.get('NEXUS_PROFILE','research').lower()
assert PROFILE in {'smoke','research'}
STRICT_DETERMINISM=bool(int(os.environ.get('NEXUS_STRICT_DETERMINISM','1')))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed:int):
    os.environ['PYTHONHASHSEED']=str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(MASTER_SEED)
torch.use_deterministic_algorithms(True,warn_only=not STRICT_DETERMINISM)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    if hasattr(torch.backends.cuda,'enable_flash_sdp'): torch.backends.cuda.enable_flash_sdp(False)
    if hasattr(torch.backends.cuda,'enable_mem_efficient_sdp'): torch.backends.cuda.enable_mem_efficient_sdp(False)
    if hasattr(torch.backends.cuda,'enable_math_sdp'): torch.backends.cuda.enable_math_sdp(True)
torch.set_float32_matmul_precision('highest')

REQUIRE_A100=bool(int(os.environ.get('NEXUS_REQUIRE_A100','1' if PROFILE=='research' else '0')))
if PROFILE=='research' and REQUIRE_A100:
    if not torch.cuda.is_available():
        raise RuntimeError('RC13C research requires CUDA/A100. Use NEXUS_PROFILE=smoke for CPU validation.')
    gpu=torch.cuda.get_device_name(0)
    if 'A100' not in gpu.upper():
        raise RuntimeError(f'RC13C research requires A100 by default; detected {gpu}.')

_IN_COLAB=Path('/content').exists()
USE_DRIVE=bool(int(os.environ.get('NEXUS_USE_DRIVE','1' if (PROFILE=='research' and _IN_COLAB) else '0')))
ALLOW_LOCAL_RESEARCH=bool(int(os.environ.get('NEXUS_ALLOW_LOCAL_RESEARCH','0')))
if _IN_COLAB and USE_DRIVE and not Path('/content/drive/MyDrive').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive',force_remount=False)
    except Exception as e:
        if PROFILE=='research' and not ALLOW_LOCAL_RESEARCH:
            raise RuntimeError(f'Paid RC13C research refused without durable Drive: {e}') from e

if Path('/content/drive/MyDrive').exists() and USE_DRIVE:
    ROOT=Path('/content/drive/MyDrive/NEXUS_II_v1_9_2_STAGE06_RC13C');PERSISTENCE_ACTIVE='drive'
elif _IN_COLAB:
    if PROFILE=='research' and not ALLOW_LOCAL_RESEARCH:
        raise RuntimeError('Paid RC13C research refused on ephemeral Colab storage.')
    ROOT=Path('/content/NEXUS_II_v1_9_2_STAGE06_RC13C');PERSISTENCE_ACTIVE='local'
else:
    ROOT=Path('./NEXUS_II_v1_9_2_STAGE06_RC13C');PERSISTENCE_ACTIVE='local'
ROOT.mkdir(parents=True,exist_ok=True)

# Canonical parent evidence from the aborted-but-integrity-preserved RC13B run.
RC13B_SOURCE_ZIP_NAME='NEXUSII_v1.9.1_S6_research_e556899106_impldbddfdc8_6a38ea_STAGE06_RESULTS.zip'
RC13B_SOURCE_ZIP_SHA256='ce32e9d26899ddcb9b32669e9fe67bb730cc1972f243a0490562edc54b39c947'
RC13B_PROTOCOL_HASH='e5568991066d679c1663fc3e5e3bbdd22e51e2cb7cea71a645c112e9526f66cb'
RC13B_SCIENTIFIC_CODE_HASH='c91af12fed11f71fe23a992da1d6e67915460a4db8b66577751286f054806a75'
RC13B_DISCOVERY_MODEL_SHA256='3eb7b969a54b0004222c5b81f4344a8d91ce02b06fdf8f04dced20017222ab0b'
RC13B_DISCOVERY_ATLAS_SHA256='7603aeeb62e9c453bb0c095aac2902202d1489033035cd3a5b8ca9bcfd78ad0f'
RC13B_DISCOVERY_CURVES_SHA256='a4b467f763ac2947d9ce6801c44879e78f020cefdb5a23d714269e0dc9ad0d26'
RC13B_DISCOVERY_LOCK_SHA256='113eed25783b3ed0331fb92d5e02a3c017fe6a82a12705a0d556c8c3c31ee9d0'

# Research tables are canonical Parquet. There is no research CSV fallback.
if PROFILE=='research':
    try:
        import pyarrow as _pa
        PYARROW_VERSION=_pa.__version__
    except Exception:
        if _IN_COLAB:
            subprocess.check_call([sys.executable,'-m','pip','install','-q','pyarrow==23.0.1'])
            import pyarrow as _pa
            PYARROW_VERSION=_pa.__version__
        else:
            raise RuntimeError('RC13C research requires pyarrow/Parquet before any scientific work; install pyarrow or use smoke profile for local validation.')
else:
    try:
        import pyarrow as _pa
        PYARROW_VERSION=_pa.__version__
    except Exception:
        PYARROW_VERSION=None

try:
    _driver=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True,stderr=subprocess.DEVNULL).strip().splitlines()[0] if torch.cuda.is_available() else None
except Exception:
    _driver=None

ENV={
    'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'numpy':np.__version__,
    'pandas':pd.__version__,'sklearn':sklearn.__version__,'scipy':scipy.__version__,'pyarrow':PYARROW_VERSION,
    'device':str(DEVICE),'gpu_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_capability':list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
    'cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    'nvidia_driver':_driver,'default_dtype':str(torch.get_default_dtype()),
    'deterministic':STRICT_DETERMINISM,'persistence':PERSISTENCE_ACTIVE,
}
ENV_COMPAT={k:ENV[k] for k in ('python','torch','numpy','pandas','sklearn','scipy','pyarrow','device','gpu_name','gpu_capability','cuda','cudnn','nvidia_driver','default_dtype','deterministic')}
ENV_COMPAT_HASH=hashlib.sha256(json.dumps(ENV_COMPAT,sort_keys=True).encode()).hexdigest()
IMPLEMENTATION_REVISION_HASH=hashlib.sha256(IMPLEMENTATION_REVISION.encode()).hexdigest()
print('DEVICE',DEVICE,ENV.get('gpu_name'))
print('PERSISTENCE',PERSISTENCE_ACTIVE)


In [ ]:
# 1. Frozen RC13C Stage-06 confirmation-completion protocol
@dataclass(frozen=True)
class CoreConfig:
    # Controlled clean-learning system: intentionally held fixed from RC12D.
    train_steps:int=48 if PROFILE=='smoke' else 500
    batch_size:int=64 if PROFILE=='smoke' else 128
    lr:float=1.2e-3
    weight_decay:float=1e-4
    adam_beta1:float=0.9
    adam_beta2:float=0.999
    adam_eps:float=1e-8
    adam_amsgrad:bool=False
    adam_maximize:bool=False
    adam_foreach:bool=False
    adam_capturable:bool=False
    adam_differentiable:bool=False
    adam_fused:bool=False
    donor_frac:float=0.40
    recipient_frac:float=0.70

    # Q measurement. h=0 is never used to construct either Stage-06 representation.
    horizons:Tuple[int,...]=(0,1,2,4,8) if PROFILE=='smoke' else (0,1,2,4,8,16)
    context_horizons:Tuple[int,...]=(1,2) if PROFILE=='smoke' else (1,2,4)
    primary_horizon:int=8 if PROFILE=='smoke' else 16
    target_positions:Tuple[float,...]=(1/3,1.0) if PROFILE=='smoke' else (0.0,1/3,2/3,1.0)
    qA_noise_replicate:int=0
    qT_noise_replicate:int=1
    qB_noise_replicate:int=2

    # Local, functionally calibrated physical Q pulses.
    functional_target_rmse:float=5e-4
    calibration_rel_tol:float=0.02
    calibration_asymmetry_max:float=0.12
    max_parameter_fraction:float=0.05
    min_functional_rmse:float=2e-5
    primary_response_metrics:Tuple[str,...]=('delta_loss_per_unit','logit_rmse_per_unit','mean_rep_displacement_per_unit')
    q_required_pass_fraction:float=1.0
    q_locality_dose_factors:Tuple[float,...]=(0.5,0.75,1.0)
    q_locality_min_secant_cosine:float=0.98
    q_locality_max_relative_norm_drift:float=0.25
    q_noise_max_abs_cosine:float=0.15

    # Hidden optimizer-state intervention locality: prospective one-step logit secants.
    hidden_locality_dose_factors:Tuple[float,...]=(0.5,0.75,1.0)
    hidden_locality_min_secant_cosine:float=0.98
    hidden_locality_max_relative_norm_drift:float=0.25

    # Function-preserving AdamW first-moment interventions.
    discovery_history_doses:Tuple[float,...]=(-1.0,-0.5,0.5,1.0)
    confirmation_history_doses:Tuple[float,...]=(-1.0,1.0)
    random_a_doses:Tuple[float,...]=(-1.0,1.0)
    random_b_doses:Tuple[float,...]=(-1.0,1.0)
    surgery_target_one_step_rmse:float=2e-4 if PROFILE=='smoke' else 5e-4
    surgery_min_one_step_rmse:float=1e-6 if PROFILE=='smoke' else 2e-5
    surgery_rel_tol:float=0.08
    surgery_sign_asymmetry_max:float=0.30
    surgery_max_m1_fraction:float=0.75
    axis_orthogonality_abs_cos_max:float=0.02 if PROFILE=='smoke' else 0.01

    # Fresh base competence, frozen before intervention outcomes.
    clean_median_acc_floor:float=0.25 if PROFILE=='smoke' else 0.65
    recipient_acc_floor:float=0.18 if PROFILE=='smoke' else 0.50
    clean_task_nontrivial_floor:float=0.20 if PROFILE=='smoke' else 0.40
    recipient_task_nontrivial_floor:float=0.15 if PROFILE=='smoke' else 0.35
    min_nontrivial_task_fraction:float=0.50 if PROFILE=='smoke' else 0.90

    # Discovery-only matched-view representation selection.
    candidate_dims:Tuple[int,...]=(1,2,4) if PROFILE=='smoke' else (1,2,4,8)
    rcca_regs:Tuple[float,...]=(1e-3,1e-2,1e-1,1.0) if PROFILE=='smoke' else (1e-4,1e-3,1e-2,1e-1,1.0,10.0)
    rcca_outer_folds:int=3 if PROFILE=='smoke' else 5
    rcca_inner_folds:int=2 if PROFILE=='smoke' else 4
    rcca_final_reg_folds:int=3 if PROFILE=='smoke' else 5
    ridge_alphas:Tuple[float,...]=(1e-4,1e-3,1e-2,1e-1,1.0,10.0,100.0)
    dimension_cv_ridge_alpha:float=1.0
    dimension_error_tolerance:float=0.15 if PROFILE=='smoke' else 0.10
    min_dimension_jackknife_agreement:float=0.50 if PROFILE=='smoke' else 0.75
    confirmation_d1_noninferiority_margin:float=0.35 if PROFILE=='smoke' else 0.10

    # H6-B: between-task predictive phenotype.
    between_min_query_corr:float=0.20 if PROFILE=='smoke' else 0.60
    between_min_query_corr_ci_lower:float=-0.20 if PROFILE=='smoke' else 0.40
    min_between_predictive_improvement:float=-0.20 if PROFILE=='smoke' else 0.05
    between_full_context_noninferiority_margin:float=0.40 if PROFILE=='smoke' else 0.10
    between_max_residual_gain_given_z:float=0.40 if PROFILE=='smoke' else 0.05

    # H6-W: baseline-centered within-task causal state.
    within_min_mean_query_corr:float=-0.20 if PROFILE=='smoke' else 0.50
    within_min_query_corr_ci_lower:float=-0.50 if PROFILE=='smoke' else 0.20
    within_min_positive_corr_fraction:float=0.30 if PROFILE=='smoke' else 0.70
    min_within_predictive_improvement:float=-0.20 if PROFILE=='smoke' else 0.05
    min_unseen_instrument_improvement:float=-0.30 if PROFILE=='smoke' else 0.05
    min_contrast_improvement_vs_zero:float=-0.30 if PROFILE=='smoke' else 0.10
    min_contrast_improvement_vs_rich:float=-0.30 if PROFILE=='smoke' else 0.05
    within_full_context_noninferiority_margin:float=0.45 if PROFILE=='smoke' else 0.10
    within_max_residual_gain_given_z:float=0.45 if PROFILE=='smoke' else 0.05
    contrast_full_context_noninferiority_margin:float=0.50 if PROFILE=='smoke' else 0.10
    contrast_max_residual_gain_given_z:float=0.50 if PROFILE=='smoke' else 0.05

    # Rich observational/microscopic challengers.
    observational_logit_sketch_dim:int=8 if PROFILE=='smoke' else 32
    observational_anchor_n:int=32 if PROFILE=='smoke' else 128
    observational_anchor_banks:int=2 if PROFILE=='smoke' else 4
    microscopic_m1_coordinate_n:int=8 if PROFILE=='smoke' else 24
    microscopic_precond_coordinate_n:int=6 if PROFILE=='smoke' else 16

    # H6-A: confirmation-only functional-potency anisotropy.
    anisotropy_random_directions:int=3 if PROFILE=='smoke' else 8
    anisotropy_median_ratio_min:float=1.2 if PROFILE=='smoke' else 10.0
    anisotropy_ratio_ci_lower_min:float=0.5 if PROFILE=='smoke' else 5.0
    anisotropy_fraction_above_floor:float=0.20 if PROFILE=='smoke' else 0.80
    anisotropy_task_ratio_floor:float=0.5 if PROFILE=='smoke' else 5.0
    anisotropy_best_random_median_min:float=0.5 if PROFILE=='smoke' else 3.0
    anisotropy_best_random_ci_lower_min:float=0.2 if PROFILE=='smoke' else 1.5

    # Replicate/task is always the independent inferential unit.
    bootstrap_n:int=256 if PROFILE=='smoke' else 10000
    signflip_mc_draws:int=20000 if PROFILE=='smoke' else 100000

CORE=CoreConfig()

class ScientificGateStop(RuntimeError): pass


# RC13C preserves the full RC13B discovery and confirmation ID sets. Only 3127–3129 are prospectively generated.
S6_DISCOVERY=tuple(range(3000,3020))
S6_CONFIRMATION=tuple(range(3100,3130))
RC13B_SEALED_CONFIRMATION=tuple(range(3100,3127))
RC13C_COMPLETION_REPLICATES=tuple(range(3127,3130))
assert set(RC13B_SEALED_CONFIRMATION).isdisjoint(RC13C_COMPLETION_REPLICATES)
assert tuple(sorted(RC13B_SEALED_CONFIRMATION+RC13C_COMPLETION_REPLICATES))==S6_CONFIRMATION

RC13C_Q_MAX_CANDIDATES=64
RC13C_Q_REJECTION_NAMESPACE='RC13C_Q_REJECTION_V1'

PARENT_EVIDENCE={
 'policy':'RC13C imports the cryptographically frozen RC13B discovery model and the sealed, unanalyzed valid confirmation blocks 3100-3126. It does not refit discovery or inspect partial H6 outcomes before the RC13C Q-plan lock. The sole scientific amendment is prospective whole-bank Q-geometry rejection qualification for completion.',
 'RC12D':{
   'protocol_hash':'54d2dd5eaf1937b2457c2e7748d2bec524a5db2ff9ed2f728f8a0d9fa53a5bf5',
   'canonical_final_zip_sha256':'02a3c95af6798348f77f35af663d0a0891af86a7e549afd36d9cdfc14dcd1b43',
   'frozen_result':'H5 failed: F/I/Q/N/P/S passed while G/Z/E/C failed. Design history only.'
 },
 'RC13B':{
   'protocol_hash':RC13B_PROTOCOL_HASH,
   'scientific_code_hash':RC13B_SCIENTIFIC_CODE_HASH,
   'canonical_aborted_results_zip_sha256':RC13B_SOURCE_ZIP_SHA256,
   'discovery_model_sha256':RC13B_DISCOVERY_MODEL_SHA256,
   'discovery_atlas_sha256':RC13B_DISCOVERY_ATLAS_SHA256,
   'discovery_dimension_curves_sha256':RC13B_DISCOVERY_CURVES_SHA256,
   'discovery_lock_sha256':RC13B_DISCOVERY_LOCK_SHA256,
   'sealed_confirmation_replicates':list(RC13B_SEALED_CONFIRMATION),
   'missing_confirmation_replicates':list(RC13C_COMPLETION_REPLICATES),
   'parent_status':'RC13B stopped before claim evaluation after 3126 because the candidate-0 Q bank for 3127 violated max |cos| <= 0.15. H6-B/H6-W/H6-A were null/not evaluated.'
 }
}

PREREG={
 'scientific_question':'Complete the frozen Stage-06 RC13B confirmation without refitting discovery or changing H6-B/H6-W/H6-A, using a prospectively frozen valid matched-Q measurement plan for the three missing confirmation tasks.',
 'headline_claims':{
   'H6_B':'Unchanged from RC13B: between-task predictive phenotype.',
   'H6_W':'Unchanged from RC13B: baseline-centered within-task causal state.',
   'H6_A':'Unchanged from RC13B: optimizer first-moment functional-potency anisotropy.'
 },
 'no_master_vote':'H6_B, H6_W, and H6_A remain independent; no rescue/majority rule.',
 'sealed_parent_confirmation':'RC13B confirmation response tables and outcome-bearing replicate metadata 3100-3126 are hash-validated as opaque bytes but must not be deserialized before the RC13C Q-plan lock is written. They remain primary confirmation observations after the lock because their candidate-0 Q banks already satisfied the same physical support condition.',
 'q_estimand':'All primary Stage-06 confirmation measurements are explicitly conditional on the preregistered valid-Q support event A: maximum pairwise absolute cosine among Q_A/Q_T/Q_B <= 0.15 at every frozen target module.',
 'sealed_reuse_justification':'Let Q^(0),Q^(1),... be iid candidate matched-Q banks independent of neural outcomes and let A denote the frozen valid-support event. An RC13B candidate-0 bank retained because A holds has law P(Q|A). The first accepted bank under prospective rejection sampling also has law P(Q|A). Therefore sealed candidate-0-valid tasks 3100-3126 and RC13C completion tasks target the same conditional Q measurement law; the amendment changes only qualification mechanics, not the H6 estimand.',
 'q_rejection_sampling_rule':f'For each confirmation task, generate whole-bank candidate index c=0,1,...,{RC13C_Q_MAX_CANDIDATES-1}. Candidate 0 is exactly the original RC13B direction seed recipe. For c>0, Gaussian noise direction seeds append namespace {RC13C_Q_REJECTION_NAMESPACE} and c. Accept the first whole bank for which the maximum pairwise absolute cosine across all three Q noise directions and all frozen targets is <=0.15. Acceptance uses Q geometry only; no trained weights, branch responses, H6 coordinates, losses, or labels.',
 'q_plan_firewall':'The complete accepted-candidate plan for all 3100-3129 is generated and cryptographically locked before any RC13B confirmation response parquet OR outcome-bearing replicate metadata JSON is deserialized and before any RC13C completion training/response measurement. Existing 3100-3126 must replay as accepted candidate 0 from architecture/seed geometry alone. Only after the lock may their sealed metadata be opened to cross-check the frozen geometry. Candidate-0 for 3127 must reproduce the documented RC13B support failure before a later candidate can be accepted.',
 'matched_query_banks':{
   'Q_A':'noise replicate 0, signed, h in context_horizons',
   'Q_B':'noise replicate 2, signed, h in context_horizons',
   'Q_T':'noise replicate 1, signed, primary_horizon only',
   'legacy_diagnostics':'gradient, orthogonal, optimizer-reset measured exactly as RC13B and excluded from primary representations/targets.'
 },
 'discovery_lock':'RC13B discovery model, atlas, dimension curves, scalers, rCCA loadings, ridge decoders, d=1 hypotheses, and d=2 challengers are imported by exact hash and never refit in RC13C.',
 'estimator':'The RC13B confirm_stage06 analysis and all supporting transforms/statistics are unchanged. RC13C only transforms the complete n=30 confirmation atlas through the imported frozen discovery bundles.',
 'dimension_rule':'Unchanged RC13B rule: d=1 is tested in confirmation against the RC13B discovery-selected d=2 challenger; confirmation never selects dimension.',
 'within_task_centering':'Unchanged RC13B exact branch-minus-baseline centering before transform; physical ΔY=0 uses the frozen discovery scaler.',
 'confirmation_only_random_b':'Unchanged: random_b is confirmation-only; existing 3100-3126 remain sealed and missing 3127-3129 generate random_b only after the RC13C Q-plan lock.',
 'causal_contrasts':'Unchanged familywise +1/-1 Q_T contrasts for history, random_a, random_b.',
 'anisotropy_estimand':'Unchanged RC13B confirmation-only history-vs-8-random functional-potency test; calibration failures are never silently dropped.',
 'inference_unit':'Task/replicate only. Final primary confirmation n=30 = sealed 3100-3126 plus prospectively completed 3127-3129.',
 'sample_size_rationale':'RC13C does not alter the RC13B planned n=30. It completes the three missing tasks and evaluates the original frozen n=30 confirmatory design exactly once.',
 'claim_boundary':'Unchanged RC13B controlled synthetic clean-learning ResNet18-GN/AdamW population and Stage-06 physical Q family, now explicitly conditioned on valid Q-support geometry.',
 'post_freeze_change_policy':'After the RC13C Q-plan lock is written, any change to accepted candidate indices, discovery objects, H6 estimator, thresholds, statistics, completion IDs, branch schema, or claim language requires a new scientific protocol hash.'
}

PROTOCOL_PAYLOAD={
 'project':PROJECT,'version':VERSION,'protocol_revision':PROTOCOL_REVISION,'profile':PROFILE,'master_seed':MASTER_SEED,
 'core':asdict(CORE),'replicates':{'discovery':S6_DISCOVERY,'confirmation':S6_CONFIRMATION,
    'sealed_parent_confirmation':RC13B_SEALED_CONFIRMATION,'completion':RC13C_COMPLETION_REPLICATES},
 'q_completion_rule':{'max_candidates':RC13C_Q_MAX_CANDIDATES,'namespace':RC13C_Q_REJECTION_NAMESPACE,
    'candidate0':'exact RC13B seed recipe','acceptance_threshold':CORE.q_noise_max_abs_cosine},
 'parent_evidence':PARENT_EVIDENCE,'prereg':PREREG,
}
PROTOCOL_HASH=hashlib.sha256(json.dumps(PROTOCOL_PAYLOAD,sort_keys=True,separators=(',',':')).encode()).hexdigest()
CONFIG_HASH=PROTOCOL_HASH
RUN_TAG=f'NEXUSII_v{VERSION}_S6C_{PROFILE}_{PROTOCOL_HASH[:10]}_impl{IMPLEMENTATION_REVISION_HASH[:8]}_{ENV_COMPAT_HASH[:6]}'
BASE=ROOT/'AUTO_CAMPAIGN'/RUN_TAG
for d in ('artifacts','blocks','manifests','logs','stage_zips','parent_rc13b'):
    (BASE/d).mkdir(parents=True,exist_ok=True)
UNLOCK_SEALED=False
RC13C_Q_PLAN=None
RC13C_Q_PLAN_LOCKED=False
print('RUN_TAG',RUN_TAG)
print('PROTOCOL_HASH',PROTOCOL_HASH)


In [ ]:
# 3. Stable identity, hashing, atomic I/O, event log

def stable_seed(*parts)->int:
    return int(hashlib.sha256('|'.join(map(str,parts)).encode()).hexdigest()[:8],16)

def tree_to_cpu(x):
    if torch.is_tensor(x): return x.detach().cpu().clone()
    if isinstance(x,dict): return {k:tree_to_cpu(v) for k,v in x.items()}
    if isinstance(x,list): return [tree_to_cpu(v) for v in x]
    if isinstance(x,tuple): return tuple(tree_to_cpu(v) for v in x)
    return copy.deepcopy(x)

def hash_tree(obj)->str:
    h=hashlib.sha256()
    def rec(x):
        if torch.is_tensor(x):
            y=x.detach().cpu().contiguous(); h.update(b'T'); h.update(str(y.dtype).encode()); h.update(str(tuple(y.shape)).encode()); h.update(y.numpy().tobytes())
        elif isinstance(x,dict):
            h.update(b'D')
            for k in sorted(x,key=lambda z:repr(z)): rec(k); rec(x[k])
        elif isinstance(x,(list,tuple)):
            h.update(type(x).__name__.encode())
            for v in x: rec(v)
        elif x is None: h.update(b'N')
        else: h.update(type(x).__name__.encode()); h.update(repr(x).encode())
    rec(obj); return h.hexdigest()

def atomic_json(path,payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=Path(str(path)+'.tmp')
    with open(tmp,'w') as f:
        f.write(json.dumps(payload,indent=2,sort_keys=True)); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)

def _durability_barrier():
    """Best-effort OS flush used only after paid checkpoint/stage commits on persistent Drive."""
    if globals().get('PERSISTENCE_ACTIVE')=='drive':
        try:os.sync()
        except Exception:
            try:subprocess.run(['sync'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,timeout=30)
            except Exception:pass


def _durable_checkpoint_commit(stage,label,step,path,spec_hash,rows):
    """Cryptographic marker proving that one paid response panel reached the persistent run tree."""
    path=Path(path)
    if not path.is_file():raise RuntimeError(f'Checkpoint commit target missing after save: {path}')
    rel=str(path.relative_to(BASE))
    marker=BASE/'manifests'/'checkpoint_commits'/stage/label/f'step_{int(step):06d}.json'
    atomic_json(marker,{'stage':stage,'label':label,'step':int(step),'relative_path':rel,'bytes':int(path.stat().st_size),'sha256':file_sha256(path),'run_spec_hash':str(spec_hash),'rows':int(rows),'committed_unix':time.time(),'persistence':PERSISTENCE_ACTIVE})
    _durability_barrier()
    # Read back marker after flush. A failed Drive commit is a hard infrastructure error before more GPU work is purchased.
    chk=json.loads(marker.read_text())
    if chk.get('sha256')!=file_sha256(path) or int(chk.get('bytes',-1))!=path.stat().st_size:
        raise RuntimeError(f'Durable checkpoint readback verification failed: {stage}/{label}/step {step}')
    print('DURABLE',stage,label,'step',step,'->',rel)
    return marker


def save_table(df,stem):
    # Exactly one canonical representation is allowed. Research is parquet-only: silently
    # changing serialization or dtype semantics mid-campaign is a reproducibility failure.
    stem=Path(stem); stem.parent.mkdir(parents=True,exist_ok=True)
    pq,cv=Path(str(stem)+'.parquet'),Path(str(stem)+'.csv')
    try:
        tmp=Path(str(pq)+'.tmp')
        df.to_parquet(tmp,index=False,compression='zstd'); os.replace(tmp,pq)
        if cv.exists():cv.unlink()
        return pq
    except Exception as e:
        try:
            if 'tmp' in locals() and tmp.exists():tmp.unlink()
        except Exception:pass
        if PROFILE=='research':
            raise RuntimeError(f'RC13B research parquet write failed for {stem}; refusing CSV fallback: {e}') from e
        tmp=Path(str(cv)+'.tmp');df.to_csv(tmp,index=False);os.replace(tmp,cv)
        if pq.exists():pq.unlink()
        return cv


def load_table(stem):
    stem=Path(stem); hits=[p for p in (Path(str(stem)+'.parquet'),Path(str(stem)+'.csv')) if p.exists()]
    if len(hits)>1: raise RuntimeError(f'Ambiguous table representations for {stem}: {hits}')
    if not hits: raise FileNotFoundError(stem)
    return pd.read_parquet(hits[0]) if hits[0].suffix=='.parquet' else pd.read_csv(hits[0])

def log_event(event,**fields):
    row={'time':time.time(),'event':event,**fields}
    with open(BASE/'logs'/'events.jsonl','a') as f: f.write(json.dumps(row,sort_keys=True)+'\n')

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def table_path(stem):
    stem=Path(stem); hits=[Path(str(stem)+s) for s in ('.parquet','.csv') if Path(str(stem)+s).exists()]
    if len(hits)>1: raise RuntimeError(f'Ambiguous table representations for {stem}: {hits}')
    if len(hits)==1:return hits[0]
    raise FileNotFoundError(stem)

# Immutable protocol/environment records are written once per run root.
atomic_json(BASE/'manifests'/'environment.json',ENV)
atomic_json(BASE/'manifests'/'core_config.json',asdict(CORE))
atomic_json(BASE/'manifests'/'protocol_payload.json',PROTOCOL_PAYLOAD)
atomic_json(BASE/'manifests'/'environment_compatibility.json',{'compat':ENV_COMPAT,'sha256':ENV_COMPAT_HASH})
atomic_json(BASE/'manifests'/'preregistered_analysis.json',PREREG)
atomic_json(BASE/'manifests'/'parent_evidence_manifest.json',PARENT_EVIDENCE)
print('IDENTITY / I-O INITIALIZED')


## Controlled task and model laboratory

Stage 06 intentionally retains the NEXUS synthetic generator and ResNet18-GN training engine so the new causal question changes **one scientific thing at a time**. The task is clean (`eta=0`) with `K=32`, `r=4`; the surgery acts only on hidden optimizer state after the recipient checkpoint is formed.


In [ ]:
# 5. Master latent suite and task variants
@dataclass(frozen=True)
class SuiteSpec:
    max_K:int=48
    image_size:int=32
    n_classes:int=4
    train_n:int=4096
    direction_n:int=192
    calibration_n:int=192
    response_n:int=192
    evaluation_n:int=768
    sealed_n:int=768
    seed:int=0

ROLE_NAMES=('train','direction','calibration','response','evaluation','sealed')

def spatial_basis(max_K,S,seed):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[-1:1:complex(S),-1:1:complex(S)]; out=[]
    for _ in range(max_K):
        cx,cy=rng.uniform(-.75,.75,2); sx,sy=rng.uniform(.15,.55,2); f=rng.uniform(.5,3.0); a=rng.uniform(0,2*np.pi)
        xr=np.cos(a)*xx+np.sin(a)*yy
        b=np.exp(-((xx-cx)**2/(2*sx*sx)+(yy-cy)**2/(2*sy*sy)))*np.cos(np.pi*f*xr)
        b=b/(np.sqrt(np.mean(b*b))+1e-8); out.append(b.astype('float32'))
    return torch.tensor(np.stack(out),dtype=torch.float32)

def make_suite(spec:SuiteSpec):
    Z={}
    sizes={r:getattr(spec,f'{r}_n') for r in ROLE_NAMES}
    for role,n in sizes.items():
        g=torch.Generator().manual_seed(stable_seed('suite_z',spec.seed,role))
        # dimension-major generation makes first k coordinates invariant as max_K changes.
        Z[role]=torch.randn(spec.max_K,n,generator=g).T.contiguous()
    basis=spatial_basis(spec.max_K,spec.image_size,stable_seed('suite_basis',spec.seed))
    # Fixed odd nonlinear coefficient bank for nested 4-class teacher.
    rng=np.random.default_rng(stable_seed('label_coeff',spec.seed))
    coeff=torch.tensor(rng.uniform(0.7,1.5,size=spec.max_K),dtype=torch.float32)
    freq=torch.tensor(rng.uniform(0.7,2.0,size=spec.max_K),dtype=torch.float32)
    suite={'spec':asdict(spec),'Z':Z,'basis':basis,'coeff':coeff,'freq':freq}
    suite['suite_hash']=hash_tree({'spec':suite['spec'],'basis':basis,'coeff':coeff,'freq':freq,'Z':Z})
    return suite

def nested_clean_labels(z,r,coeff,freq):
    assert r>=2 and r<=z.shape[1]
    # Two nested coordinate streams define two balanced-ish bits -> 4 classes.
    ia=torch.arange(0,r,2); ib=torch.arange(1,r,2)
    def score(idx):
        if len(idx)==0:return torch.zeros(len(z))
        zz=z[:,idx]; c=coeff[idx]; f=freq[idx]
        term=torch.tanh(c*zz)+0.25*torch.sin(f*zz)
        return term.sum(1)/math.sqrt(len(idx))
    a=score(ia); b=score(ib)
    return ((a>0).long()*2+(b>0).long()), torch.minimum(a.abs(),b.abs())

def nested_noisy_labels(clean,eta,seed,role):
    if eta<=0:return clean.clone(),torch.zeros(len(clean),dtype=torch.bool)
    g=torch.Generator().manual_seed(stable_seed('label_noise',seed,role))
    u=torch.rand(len(clean),generator=g); mask=u<eta
    # Replacement is guaranteed to differ from clean label.
    off=1+torch.randint(0,3,(len(clean),),generator=g)
    noisy=(clean+off)%4
    return torch.where(mask,noisy,clean),mask

def render_fixedK(z,basis,K):
    raw=torch.einsum('nk,khw->nhw',z[:,:K],basis[:K])/math.sqrt(K)
    return torch.tanh(0.85*raw).unsqueeze(1)

def render_nuisanceK(z,basis,K,r,nuisance_gain=1.0):
    assert K>r>=2
    sig=torch.einsum('nk,khw->nhw',z[:,:r],basis[:r])/math.sqrt(r)
    nui=torch.einsum('nk,khw->nhw',z[:,r:K],basis[r:K])/math.sqrt(K-r)
    raw=(sig+nuisance_gain*nui)/math.sqrt(1+nuisance_gain**2)
    return torch.tanh(0.85*raw).unsqueeze(1)

def make_task_variant(suite,K,r,eta=0.0,mode='fixedK',replicate_id=0):
    assert 2<=r<=K<=suite['spec']['max_K']
    public={}; sealed={}; clean_hashes={}; noise_masks={}
    for role in ROLE_NAMES:
        z=suite['Z'][role]
        X=render_nuisanceK(z,suite['basis'],K,r) if mode=='nuisanceK' else render_fixedK(z,suite['basis'],K)
        y_clean,margin=nested_clean_labels(z,r,suite['coeff'],suite['freq'])
        # Noise changes the training objective, not the primary measurement operator.
        # The direction role stores a noisy objective label separately so conventional objective-gradient
        # diagnostics can still be measured without changing the clean-reference causal intervention.
        if role in {'train','direction'}:
            y_objective,mask=nested_noisy_labels(y_clean,eta,suite['spec']['seed'],role)
        else:
            y_objective=y_clean.clone(); mask=torch.zeros(len(y_clean),dtype=torch.bool)
        y_primary=y_objective if role=='train' else y_clean
        item={'X':X.contiguous(),'y':y_primary.contiguous(),'y_clean':y_clean.contiguous(),'y_objective':y_objective.contiguous(),'teacher_margin':margin.contiguous()}
        if role=='sealed': sealed=item
        else: public[role]=item
        clean_hashes[role]=hash_tree(y_clean); noise_masks[role]=mask
    spec={'K':K,'r':r,'label_noise':float(eta),'mode':mode,'n_classes':4,'replicate_id':int(replicate_id),'suite_hash':suite['suite_hash'],'suite_seed':suite['spec']['seed'],'in_ch':1}
    public_hash=hash_tree({'spec':spec,'public':public})
    sealed_hash=hash_tree({'spec':spec,'sealed':sealed})
    task={'spec':spec,'roles':public,'task_hash':public_hash,'sealed_hash':sealed_hash,
          'clean_label_hashes':clean_hashes,'noise_masks':noise_masks}
    return task,sealed

def get_sealed(sealed_bundle):
    if not UNLOCK_SEALED: raise RuntimeError('Sealed holdout is locked. Set NEXUS_UNLOCK_SEALED=1 only after analysis is frozen.')
    return sealed_bundle


In [ ]:
# 7. Architecture registry

def _gn(c):
    g=min(32,max(1,c//8))
    while c%g!=0:g-=1
    return nn.GroupNorm(g,c)

class ConvCalib(nn.Module):
    def __init__(self,n_classes=4,in_ch=1,w=16):
        super().__init__(); self.stem=nn.Conv2d(in_ch,w,3,padding=1); self.b1=nn.Sequential(nn.Conv2d(w,2*w,3,stride=2,padding=1),nn.GELU()); self.b2=nn.Sequential(nn.Conv2d(2*w,4*w,3,stride=2,padding=1),nn.GELU()); self.pool=nn.AdaptiveAvgPool2d(1); self.head=nn.Linear(4*w,n_classes)
    def forward(self,x,return_features=False):
        f=[]; x=F.gelu(self.stem(x));f.append(x.mean((-2,-1)));x=self.b1(x);f.append(x.mean((-2,-1)));x=self.b2(x);f.append(x.mean((-2,-1)));y=self.head(self.pool(x).flatten(1));return (y,f) if return_features else y
    def intervention_targets(self):return ['stem','b1','b2','head']
    def target_module(self,n):return getattr(self,n)

class BasicBlockGN(nn.Module):
    expansion=1
    def __init__(self,cin,cout,stride=1):
        super().__init__(); self.c1=nn.Conv2d(cin,cout,3,stride=stride,padding=1,bias=False);self.n1=_gn(cout);self.c2=nn.Conv2d(cout,cout,3,padding=1,bias=False);self.n2=_gn(cout)
        self.skip=nn.Identity() if stride==1 and cin==cout else nn.Sequential(nn.Conv2d(cin,cout,1,stride=stride,bias=False),_gn(cout))
    def forward(self,x):
        y=F.gelu(self.n1(self.c1(x)));y=self.n2(self.c2(y));return F.gelu(y+self.skip(x))

class ResNet18GN(nn.Module):
    def __init__(self,n_classes=4,in_ch=1,base=32):
        super().__init__();self.stem=nn.Sequential(nn.Conv2d(in_ch,base,3,padding=1,bias=False),_gn(base),nn.GELU())
        self.layer1=self._layer(base,base,2,1);self.layer2=self._layer(base,2*base,2,2);self.layer3=self._layer(2*base,4*base,2,2);self.layer4=self._layer(4*base,8*base,2,2)
        self.pool=nn.AdaptiveAvgPool2d(1);self.head=nn.Linear(8*base,n_classes)
    def _layer(self,cin,cout,n,stride):return nn.Sequential(BasicBlockGN(cin,cout,stride),*[BasicBlockGN(cout,cout,1) for _ in range(n-1)])
    def forward(self,x,return_features=False):
        f=[];x=self.stem(x);f.append(x.mean((-2,-1)));x=self.layer1(x);f.append(x.mean((-2,-1)));x=self.layer2(x);f.append(x.mean((-2,-1)));x=self.layer3(x);f.append(x.mean((-2,-1)));x=self.layer4(x);f.append(x.mean((-2,-1)));z=self.pool(x).flatten(1);y=self.head(z);return (y,f) if return_features else y
    def intervention_targets(self):return ['stem','layer1','layer2','layer3','layer4','head']
    def target_module(self,n):return getattr(self,n)

class ViT(nn.Module):
    def __init__(self,n_classes=4,in_ch=1,dim=128,depth=4,heads=4,patch=4):
        super().__init__();self.patch=nn.Conv2d(in_ch,dim,patch,patch);nt=(32//patch)**2;self.cls=nn.Parameter(torch.zeros(1,1,dim));self.pos=nn.Parameter(torch.randn(1,nt+1,dim)*0.02)
        layer=nn.TransformerEncoderLayer(dim,heads,4*dim,dropout=0.0,activation='gelu',batch_first=True,norm_first=True)
        self.blocks=nn.ModuleList([copy.deepcopy(layer) for _ in range(depth)]);self.norm=nn.LayerNorm(dim);self.head=nn.Linear(dim,n_classes)
    def forward(self,x,return_features=False):
        x=self.patch(x).flatten(2).transpose(1,2);x=torch.cat([self.cls.expand(x.size(0),-1,-1),x],1)+self.pos[:,:x.size(1)+1];f=[]
        for b in self.blocks:x=b(x);f.append(x[:,0])
        z=self.norm(x[:,0]);y=self.head(z);return (y,f) if return_features else y
    def intervention_targets(self):return ['patch']+[f'blocks.{i}' for i in range(len(self.blocks))]+['head']
    def target_module(self,n):
        if n=='patch':return self.patch
        if n=='head':return self.head
        if n.startswith('blocks.'):return self.blocks[int(n.split('.')[1])]
        raise KeyError(n)

def build_model(family,n_classes,scale=1,seed=0,in_ch=1):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(stable_seed('model_init',family,scale,seed,in_ch,n_classes))
        if family=='calib':m=ConvCalib(n_classes,in_ch,w=16*scale)
        elif family=='resnet':m=ResNet18GN(n_classes,in_ch,base=16*scale)
        elif family=='vit':
            presets={1:(96,4,4),2:(192,6,6),3:(256,8,8),4:(384,8,8)};dim,depth,heads=presets[int(scale)];m=ViT(n_classes,in_ch,dim,depth,heads)
        else:raise ValueError(family)
    return m.to(DEVICE)

def parameter_count(m):return sum(p.numel() for p in m.parameters())

for fam,sc in [('calib',1),('resnet',1),('resnet',2),('resnet',4),('vit',1),('vit',2),('vit',4)]:
    m=build_model(fam,4,sc,0);print(f'{fam} scale={sc}: {parameter_count(m):,} params');del m
if torch.cuda.is_available():torch.cuda.empty_cache()


In [ ]:
# 8. Deterministic trainer, explicitly frozen AdamW, paired stochastic stream, checkpoint bank
@dataclass(frozen=True)
class TrainSpec:
    steps:int
    batch_size:int
    lr:float
    weight_decay:float
    beta1:float
    beta2:float
    eps:float
    amsgrad:bool
    maximize:bool
    foreach:bool
    capturable:bool
    differentiable:bool
    fused:bool

def s6_train_spec():
    return TrainSpec(
        steps=CORE.train_steps,batch_size=CORE.batch_size,lr=CORE.lr,weight_decay=CORE.weight_decay,
        beta1=CORE.adam_beta1,beta2=CORE.adam_beta2,eps=CORE.adam_eps,amsgrad=CORE.adam_amsgrad,
        maximize=CORE.adam_maximize,foreach=CORE.adam_foreach,capturable=CORE.adam_capturable,
        differentiable=CORE.adam_differentiable,fused=CORE.adam_fused,
    )

def resolved_checkpoints(steps):
    return tuple(sorted(set([int(round(CORE.donor_frac*steps)),int(round(CORE.recipient_frac*steps)),int(steps)])))

def paired_stream_id(task):
    return str(task['spec']['suite_hash'])

def make_schedule(n,batch_size,steps,seed):
    assert n>=batch_size
    needed=steps+max(CORE.horizons)+1
    batches=[];g=torch.Generator().manual_seed(seed);usable=(n//batch_size)*batch_size
    while len(batches)<needed:
        p=torch.randperm(n,generator=g)[:usable].reshape(-1,batch_size)
        batches.extend([row.clone() for row in p])
    cpu=torch.stack(batches[:needed])
    return cpu.to(DEVICE),hash_tree(cpu)

def make_optimizer(m,ts):
    # All AdamW knobs used by this protocol are explicit. No scientific parameter depends on PyTorch defaults.
    return torch.optim.AdamW(
        m.parameters(),lr=ts.lr,betas=(ts.beta1,ts.beta2),eps=ts.eps,weight_decay=ts.weight_decay,
        amsgrad=ts.amsgrad,maximize=ts.maximize,foreach=ts.foreach,capturable=ts.capturable,
        differentiable=ts.differentiable,fused=ts.fused,
    )

def optimizer_defaults_record(o):
    d=dict(o.defaults)
    out={}
    for k,v in d.items():
        if isinstance(v,(str,int,float,bool,type(None))):out[k]=v
        elif isinstance(v,tuple):out[k]=list(v)
        else:out[k]=str(v)
    return out

def model_hash(m):return hash_tree(tree_to_cpu(m.state_dict()))
def optimizer_hash(o):return hash_tree(tree_to_cpu(o.state_dict()))

def step_rng_seed(stream_id,family,scale,model_seed,absolute_step):
    return stable_seed('step_rng',stream_id,model_seed,absolute_step)

def set_step_rng(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def prepare_device_task(task):
    out={'spec':task['spec'],'task_hash':task['task_hash'],'sealed_hash':task['sealed_hash']};out['roles']={}
    for role,d in task['roles'].items():
        out['roles'][role]={k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in d.items()}
    return out

def train_step(m,o,dtask,idx,abs_step,family,scale,model_seed,frozen_target=None):
    set_step_rng(step_rng_seed(paired_stream_id(dtask),family,scale,model_seed,abs_step))
    m.train()
    if frozen_target is not None:m.target_module(frozen_target).eval()
    X=dtask['roles']['train']['X'];y=dtask['roles']['train']['y']
    o.zero_grad(set_to_none=True)
    loss=F.cross_entropy(m(X[idx]),y[idx]);loss.backward()
    if frozen_target is not None:
        for p in m.target_module(frozen_target).parameters():p.grad=None
    o.step()
    return float(loss.detach().cpu())

def snapshot(m,o,step,meta):
    return {
        'step':int(step),'model':tree_to_cpu(m.state_dict()),'optimizer':tree_to_cpu(o.state_dict()),
        'model_hash':model_hash(m),'optimizer_hash':optimizer_hash(o),**copy.deepcopy(meta)
    }

def load_snapshot(task,family,scale,model_seed,ts,snap):
    assert snap['task_hash']==task['task_hash']
    assert snap['core_config_hash']==CONFIG_HASH
    assert snap.get('protocol_hash')==PROTOCOL_HASH
    assert snap.get('env_compat_hash')==ENV_COMPAT_HASH
    in_ch=int(task['spec'].get('in_ch',1))
    m=build_model(family,task['spec']['n_classes'],scale,model_seed,in_ch)
    m.load_state_dict(copy.deepcopy(snap['model']))
    o=make_optimizer(m,ts)
    o.load_state_dict(copy.deepcopy(snap['optimizer']))
    for st in o.state.values():
        for k,v in st.items():
            if torch.is_tensor(v):st[k]=v.to(DEVICE)
    assert model_hash(m)==snap['model_hash'] and optimizer_hash(o)==snap['optimizer_hash']
    return m,o

def build_checkpoint_bank(task,family,scale,model_seed,ts):
    dt=prepare_device_task(task);stream=paired_stream_id(task)
    sch,shash=make_schedule(len(task['roles']['train']['X']),ts.batch_size,ts.steps,stable_seed('schedule',stream,model_seed,asdict(ts)))
    in_ch=int(task['spec'].get('in_ch',1))
    m=build_model(family,task['spec']['n_classes'],scale,model_seed,in_ch)
    o=make_optimizer(m,ts);cps=resolved_checkpoints(ts.steps);bank={}
    model_n=parameter_count(m)
    model_dtype=sorted({str(p.dtype) for p in m.parameters()})
    opt_defaults=optimizer_defaults_record(o)
    meta={
        'family':family,'scale':scale,'model_seed':model_seed,'task_hash':task['task_hash'],
        'paired_stream_id':stream,'schedule_hash':shash,'core_config_hash':CONFIG_HASH,
        'protocol_hash':PROTOCOL_HASH,'env_compat_hash':ENV_COMPAT_HASH,
        'train_spec_hash':hashlib.sha256(json.dumps(asdict(ts),sort_keys=True).encode()).hexdigest(),
        'train_spec':asdict(ts),'optimizer_defaults':opt_defaults,'model_parameter_count':int(model_n),
        'model_dtypes':model_dtype,'in_ch':in_ch,
    }
    for st in range(ts.steps+1):
        if st in cps:bank[st]=snapshot(m,o,st,meta)
        if st<ts.steps:train_step(m,o,dt,sch[st],st,family,scale,model_seed)
    final_eval=evaluate_public(m,dt)
    del m,o;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return bank,sch,dt,final_eval


In [ ]:
# 9. Observational and microscopic challenger descriptors
@torch.no_grad()
def forward_bundle(m,X,y=None,return_features=True):
    was=m.training;m.eval();out=m(X,return_features=return_features) if return_features else m(X)
    if return_features:logits,feats=out
    else:logits=out;feats=[]
    probs=logits.softmax(-1)
    ret={'logits':logits.detach(),'features':[f.detach() for f in feats],
         'entropy':float((-(probs*torch.log(probs.clamp_min(1e-12))).sum(-1).mean()).cpu())}
    if y is not None:
        true=logits.gather(1,y[:,None]).squeeze(1);tmp=logits.clone();tmp.scatter_(1,y[:,None],float('-inf'))
        ret.update({'loss':float(F.cross_entropy(logits,y).cpu()),
                    'acc':float((logits.argmax(1)==y).float().mean().cpu()),
                    'margin':float((true-tmp.max(1).values).mean().cpu())})
    m.train(was);return ret

@torch.no_grad()
def evaluate_public(m,dt):
    d=dt['roles']['evaluation'];return forward_bundle(m,d['X'],d['y_clean'],False)

def effective_rank(X):
    X=X.float()-X.float().mean(0,keepdim=True);s=torch.linalg.svdvals(X);q=s.pow(2);q=q/(q.sum()+1e-12)
    return float(torch.exp(-(q*torch.log(q+1e-12)).sum()).cpu())

def cka(a,b):
    a=a.float()-a.float().mean(0,keepdim=True);b=b.float()-b.float().mean(0,keepdim=True)
    num=((a.T@b)**2).sum();den=(((a.T@a)**2).sum()*((b.T@b)**2).sum()).sqrt()+1e-12
    return float((num/den).cpu())

def tuple_norm(gs):return math.sqrt(sum(float((g.detach().float()**2).sum().cpu()) for g in gs if g is not None))
def tuple_average(g1,g2):return tuple((a+b)*0.5 for a,b in zip(g1,g2))

def optimizer_moment_norms(o):
    a1=a2=0.0
    for st in o.state.values():
        if 'exp_avg' in st:a1+=float((st['exp_avg'].detach().float()**2).sum().cpu())
        if 'exp_avg_sq' in st:a2+=float((st['exp_avg_sq'].detach().float()**2).sum().cpu())
    return math.sqrt(a1),math.sqrt(a2)

def optimizer_preconditioned_norm(o,eps=None):
    eps=float(CORE.adam_eps if eps is None else eps);s=0.0
    for st in o.state.values():
        if 'exp_avg' in st and 'exp_avg_sq' in st:
            u=st['exp_avg'].detach().float()/(st['exp_avg_sq'].detach().float().sqrt()+eps)
            s+=float((u*u).sum().cpu())
    return math.sqrt(s)

def optimizer_step_stats(o):
    vals=[]
    for st in o.state.values():
        if 'step' in st:
            x=st['step']
            vals.append(float(x.detach().cpu().item()) if torch.is_tensor(x) else float(x))
    if not vals:return (0.0,0.0)
    return (float(min(vals)),float(max(vals)))

def gradient_tuple(m,X,y):
    was=m.training;m.eval();m.zero_grad(set_to_none=True);loss=F.cross_entropy(m(X),y)
    g=torch.autograd.grad(loss,tuple(m.parameters()),retain_graph=False,create_graph=False);m.train(was);return g

def grad_cosine(g1,g2):
    dot=n1=n2=0.0
    for a,b in zip(g1,g2):
        dot+=float((a.detach().float()*b.detach().float()).sum().cpu())
        n1+=float((a.detach().float()**2).sum().cpu());n2+=float((b.detach().float()**2).sum().cpu())
    return dot/(math.sqrt(n1*n2)+1e-30)

# Same task-independent input-only anchor bank for every fresh replicate.
_BASE_OBS_SCALAR_COLS=[
    'control_loss','control_acc','control_margin','control_entropy','control_weight_norm',
    'control_grad_norm','control_grad_alignment','control_objective_grad_norm',
    'control_clean_objective_grad_cosine','control_moment1_norm','control_moment2_norm',
    'control_preconditioned_update_norm','control_optimizer_step_min','control_optimizer_step_max',
    'control_feature_erank','control_direction_clean_acc','control_direction_objective_acc'
]
OBS_LOCAL_LOGIT_COLS=(
    [f'control_local_logit_mean_{i}' for i in range(4)]+
    [f'control_local_logit_std_{i}' for i in range(4)]+
    [f'control_local_logit_cov_{i}_{j}' for i in range(4) for j in range(i,4)]+
    [f'control_local_prob_mean_{i}' for i in range(4)]
)
OBS_ANCHOR_SKETCH_COLS=[f'control_anchor_logit_sketch_{i:02d}' for i in range(CORE.observational_logit_sketch_dim)]
OBS_M1_COORD_COLS=[f'control_m1_coord_{i:02d}' for i in range(CORE.microscopic_m1_coordinate_n)]
OBS_PRECOND_COORD_COLS=[f'control_precond_coord_{i:02d}' for i in range(CORE.microscopic_precond_coordinate_n)]
MICRO_TARGET_COLS=[]
for _i,_q in enumerate(CORE.target_positions):
    MICRO_TARGET_COLS += [
        f'control_target_param_norm_{_i}',f'control_target_m1_norm_{_i}',
        f'control_target_m2_norm_{_i}',f'control_target_preconditioned_norm_{_i}'
    ]
OBS_COLS=_BASE_OBS_SCALAR_COLS+OBS_LOCAL_LOGIT_COLS+OBS_ANCHOR_SKETCH_COLS+OBS_M1_COORD_COLS+OBS_PRECOND_COORD_COLS+MICRO_TARGET_COLS
_OBS_PROJ_CACHE={};_OBS_ANCHOR_CPU=None;_OBS_ANCHOR_DEVICE_CACHE={}

def _fixed_logit_sketch(logits,dim=None):
    dim=int(CORE.observational_logit_sketch_dim if dim is None else dim)
    v=logits.detach().float().reshape(-1).cpu();key=(int(v.numel()),dim)
    if key not in _OBS_PROJ_CACHE:
        g=torch.Generator(device='cpu').manual_seed(stable_seed('S6_observational_logit_sketch',key[0],key[1]))
        R=(torch.randint(0,2,(key[0],dim),generator=g,dtype=torch.int8).float()*2-1)/math.sqrt(key[0])
        _OBS_PROJ_CACHE[key]=R
    return (v@_OBS_PROJ_CACHE[key]).numpy().astype(float)

def _shared_observational_anchor_x():
    global _OBS_ANCHOR_CPU
    if _OBS_ANCHOR_CPU is None:
        n=int(CORE.observational_anchor_n);banks=int(CORE.observational_anchor_banks);K=32;H=SuiteSpec().image_size
        counts=[n//banks+(1 if b<n%banks else 0) for b in range(banks)];chunks=[]
        for b,nbank in enumerate(counts):
            if nbank<=0:continue
            basis=spatial_basis(K,H,stable_seed('S6_shared_obs_anchor_basis',b))
            g=torch.Generator().manual_seed(stable_seed('S6_shared_obs_anchor_z',b))
            z=torch.randn(nbank,K,generator=g);chunks.append(render_fixedK(z,basis,K))
        _OBS_ANCHOR_CPU=torch.cat(chunks,0).contiguous();assert len(_OBS_ANCHOR_CPU)==n
    return _OBS_ANCHOR_CPU

def _shared_observational_anchor_device():
    key=str(DEVICE)
    if key not in _OBS_ANCHOR_DEVICE_CACHE:
        _OBS_ANCHOR_DEVICE_CACHE[key]=_shared_observational_anchor_x().to(DEVICE)
    return _OBS_ANCHOR_DEVICE_CACHE[key]

def _local_logit_moments(logits):
    x=logits.detach().float();mu=x.mean(0);sd=x.std(0,unbiased=False);xc=x-mu
    cov=(xc.T@xc)/max(1,len(x));pm=x.softmax(-1).mean(0);vals=[]
    vals.extend(float(v.cpu()) for v in mu);vals.extend(float(v.cpu()) for v in sd)
    vals.extend(float(cov[i,j].cpu()) for i in range(4) for j in range(i,4))
    vals.extend(float(v.cpu()) for v in pm)
    return dict(zip(OBS_LOCAL_LOGIT_COLS,vals))

def _optimizer_coordinate_sketch(m,o,n,kind):
    n=int(n);items=[];total=0
    for name,p in m.named_parameters():
        st=o.state.get(p,{})
        if 'exp_avg' not in st:continue
        size=int(st['exp_avg'].numel());items.append((name,p,total,total+size));total+=size
    if total<n:raise RuntimeError(f'Optimizer coordinate sketch requests {n} coordinates from only {total}.')
    rng=np.random.default_rng(stable_seed('S6_optimizer_coordinate_sketch',kind,total,n))
    picks=rng.choice(total,size=n,replace=False)
    ends=np.array([b for _,_,_,b in items],dtype=np.int64)
    vals=[]
    for gi in picks:
        j=int(np.searchsorted(ends,int(gi),side='right'));name,p,a,b=items[j];local=int(gi-a)
        st=o.state[p];m1=st['exp_avg'].detach().float().reshape(-1)
        if kind=='m1':v=m1[local]
        elif kind=='precond':
            v=m1[local]/(st['exp_avg_sq'].detach().float().reshape(-1)[local].sqrt()+CORE.adam_eps)
        else:raise ValueError(kind)
        vals.append(float(v.cpu()))
    return vals

def observational_state(m,o,dt):
    ev=dt['roles']['evaluation'];resp=dt['roles']['response'];dire=dt['roles']['direction']
    b=forward_bundle(m,ev['X'],ev['y_clean'],False);fr=forward_bundle(m,resp['X'],None,True)
    anchor=forward_bundle(m,_shared_observational_anchor_device(),None,False)['logits'];anchor_sketch=_fixed_logit_sketch(anchor)
    n=len(dire['X']);mid=max(1,n//2)
    clean1=gradient_tuple(m,dire['X'][:mid],dire['y_clean'][:mid]);clean2=gradient_tuple(m,dire['X'][mid:],dire['y_clean'][mid:])
    gclean=tuple_average(clean1,clean2)
    obj1=gradient_tuple(m,dire['X'][:mid],dire['y_objective'][:mid]);obj2=gradient_tuple(m,dire['X'][mid:],dire['y_objective'][mid:])
    gobj=tuple_average(obj1,obj2)
    was=m.training;m.eval()
    with torch.no_grad():
        dlog=m(dire['X']);dclean=float((dlog.argmax(1)==dire['y_clean']).float().mean().cpu())
        dobj=float((dlog.argmax(1)==dire['y_objective']).float().mean().cpu())
    m.train(was)
    wnorm=math.sqrt(sum(float((p.detach().float()**2).sum().cpu()) for p in m.parameters()))
    m1,m2=optimizer_moment_norms(o);er=[effective_rank(f) for f in fr['features']]
    stmin,stmax=optimizer_step_stats(o)
    out={
        'control_loss':b['loss'],'control_acc':b['acc'],'control_margin':b['margin'],'control_entropy':b['entropy'],
        'control_weight_norm':wnorm,'control_grad_norm':tuple_norm(gclean),'control_grad_alignment':grad_cosine(clean1,clean2),
        'control_objective_grad_norm':tuple_norm(gobj),'control_clean_objective_grad_cosine':grad_cosine(gclean,gobj),
        'control_moment1_norm':m1,'control_moment2_norm':m2,
        'control_preconditioned_update_norm':optimizer_preconditioned_norm(o),
        'control_optimizer_step_min':stmin,'control_optimizer_step_max':stmax,
        'control_feature_erank':float(np.mean(er)) if er else np.nan,
        'control_direction_clean_acc':dclean,'control_direction_objective_acc':dobj,
    }
    out.update(_local_logit_moments(b['logits']))
    out.update({k:float(v) for k,v in zip(OBS_ANCHOR_SKETCH_COLS,anchor_sketch)})
    out.update({k:float(v) for k,v in zip(OBS_M1_COORD_COLS,_optimizer_coordinate_sketch(m,o,len(OBS_M1_COORD_COLS),'m1'))})
    out.update({k:float(v) for k,v in zip(OBS_PRECOND_COORD_COLS,_optimizer_coordinate_sketch(m,o,len(OBS_PRECOND_COORD_COLS),'precond'))})
    return out


In [ ]:
# 10. Intervention geometry and symmetric functional calibration
SIGNED_KINDS={'noise','gradient','orthogonal'}
STATE_KINDS={'freeze','optimizer_reset'}

def target_parameters(m,target):return list(m.target_module(target).parameters())
def flat_target(m,target):return torch.cat([p.detach().reshape(-1) for p in target_parameters(m,target)])
@torch.no_grad()
def add_delta(m,target,d):
    pos=0
    for p in target_parameters(m,target):n=p.numel();p.add_(d[pos:pos+n].view_as(p));pos+=n
    assert pos==d.numel()

def target_gradient(m,target,X,y):
    # Measurement gradient is defined in eval mode (same deterministic function used by calibration/evaluation).
    # This is crucial for BatchNorm models: a direction probe must never mutate running-stat buffers.
    ps=target_parameters(m,target);was=m.training;state=tree_to_cpu(m.state_dict())
    try:
        m.eval();m.zero_grad(set_to_none=True)
        g=torch.autograd.grad(F.cross_entropy(m(X),y),ps,retain_graph=False,create_graph=False)
        out=torch.cat([x.detach().reshape(-1) for x in g])
    finally:
        m.load_state_dict(copy.deepcopy(state));m.train(was)
    return out

def unit_direction(m,target,kind,Xdir,ydir,seed):
    w=flat_target(m,target)
    if kind=='gradient':
        g=target_gradient(m,target,Xdir,ydir);return -g/(g.norm()+1e-12)
    gen=torch.Generator(device=DEVICE.type).manual_seed(seed);r=torch.randn(w.shape,device=DEVICE,dtype=w.dtype,generator=gen)
    if kind=='orthogonal':
        g=target_gradient(m,target,Xdir,ydir);u=g/(g.norm()+1e-12);r=r-torch.dot(r,u)*u
    if float(r.norm().cpu())<1e-12:raise RuntimeError('Degenerate perturbation direction')
    return r/(r.norm()+1e-12)

@torch.no_grad()
def signed_functional_effect(m,target,d,Xcal,alpha,state):
    was=m.training;m.eval();base=m(Xcal).detach();vals=[]
    for sign in (-1,1):
        m.load_state_dict(copy.deepcopy(state));m.eval();add_delta(m,target,d*(sign*alpha));out=m(Xcal);vals.append(float(torch.sqrt(torch.mean((out-base)**2)).cpu()))
    m.load_state_dict(copy.deepcopy(state));m.train(was);return vals[0],vals[1]

def central_secant_locality(m,target,d,Xcal,alpha,state):
    """Check that the signed Q direction remains in a local linear-response regime.

    Central secants at 0.5x/0.75x are compared with the frozen 1x central secant.
    This is a measurement-validity gate, not evidence for any Stage-06 headline claim.
    """
    was=m.training;m.eval();vecs={}
    try:
        for fac in CORE.q_locality_dose_factors:
            outs=[]
            for sign in (-1,1):
                m.load_state_dict(copy.deepcopy(state));m.eval()
                add_delta(m,target,d*(float(sign)*float(fac)*float(alpha)))
                outs.append(m(Xcal).detach().reshape(-1).float())
            sec=(outs[1]-outs[0])/(2.0*float(fac)*max(float(alpha),1e-30))
            vecs[float(fac)]=sec.detach().cpu()
        ref=vecs[1.0];rn=float(torch.linalg.vector_norm(ref))
        cos=[];drift=[]
        for fac in CORE.q_locality_dose_factors:
            v=vecs[float(fac)];vn=float(torch.linalg.vector_norm(v))
            c=float(torch.dot(v,ref)/(torch.linalg.vector_norm(v)*torch.linalg.vector_norm(ref)+1e-30))
            cos.append(c);drift.append(abs(vn-rn)/max(rn,1e-30))
        out={'min_secant_cosine':float(min(cos)),
             'max_relative_secant_norm_drift':float(max(drift))}
        for fac,c,dr in zip(CORE.q_locality_dose_factors,cos,drift):
            tag=f'{float(fac):.2f}'.replace('.','p')
            out[f'secant_cosine_{tag}']=float(c)
            out[f'secant_relative_norm_drift_{tag}']=float(dr)
        return out
    finally:
        m.load_state_dict(copy.deepcopy(state));m.train(was)

def calibrate_symmetric(m,target,d,Xcal,target_rmse):
    """Dual-budget local calibration.

    First attempts to hit the same immediate function-space RMSE for +/- directions.
    If that would require too large a microscopic displacement, alpha is capped at the
    predeclared relative parameter budget. If sign asymmetry remains too large, alpha is
    halved until the perturbation is locally symmetric enough. The achieved function-space
    amplitude is always recorded and later used to normalize pulse responses.
    """
    state=tree_to_cpu(m.state_dict());wn=float(flat_target(m,target).norm().cpu())
    a=max(1e-8,1e-4*max(wn,1e-6));e0m,e0p=signed_functional_effect(m,target,d,Xcal,a,state);slope=(e0m+e0p)/(2*a)
    if not np.isfinite(slope) or slope<=1e-12:raise RuntimeError('Cannot calibrate functional perturbation: near-zero local response.')
    alpha=target_rmse/slope
    for _ in range(7):
        em,ep=signed_functional_effect(m,target,d,Xcal,alpha,state);mean=(em+ep)/2
        if mean<=1e-12 or not np.isfinite(mean):raise RuntimeError('Functional calibration collapsed.')
        if abs(mean-target_rmse)/target_rmse<=CORE.calibration_rel_tol:break
        alpha*=float(np.clip(target_rmse/mean,0.25,4.0))
    uncapped_alpha=float(alpha);cap=CORE.max_parameter_fraction*max(wn,1e-12);capped=alpha>cap
    if capped:alpha=cap
    em,ep=signed_functional_effect(m,target,d,Xcal,alpha,state);mean=(em+ep)/2;asym=abs(ep-em)/(mean+1e-12)
    # If the local map is still too sign-asymmetric, shrink until it enters the local regime.
    shrink_steps=0
    while asym>CORE.calibration_asymmetry_max and mean>CORE.min_functional_rmse and shrink_steps<8:
        alpha*=0.5;shrink_steps+=1;em,ep=signed_functional_effect(m,target,d,Xcal,alpha,state);mean=(em+ep)/2;asym=abs(ep-em)/(mean+1e-12);capped=True
    frac=alpha/max(wn,1e-12)
    if mean<CORE.min_functional_rmse:raise RuntimeError(f'Intervention response below minimum measurable functional amplitude ({mean:.3e}).')
    if asym>CORE.calibration_asymmetry_max:raise RuntimeError(f'Could not enter symmetric local regime (asymmetry={asym:.3f}).')
    locality=central_secant_locality(m,target,d,Xcal,alpha,state)
    return {'alpha':float(alpha),'uncapped_alpha':uncapped_alpha,'minus_rmse':float(em),'plus_rmse':float(ep),'mean_rmse':float(mean),'sign_asymmetry':float(asym),'parameter_fraction':float(frac),'local_slope':float(slope),'calibration_capped':bool(capped),'calibration_shrink_steps':int(shrink_steps),**locality}
def reset_optimizer_target(o,m,target):
    changed=0
    for p in target_parameters(m,target):
        st=o.state.get(p,{})
        for key in ('exp_avg','exp_avg_sq','max_exp_avg_sq'):
            if key in st:st[key].zero_();changed+=1
    return changed


In [ ]:
# 11. Matched counterfactual continuation engine

def continuation(m,o,dt,sch,start,family,scale,model_seed,frozen_target=None):
    resp=dt['roles']['response'];ev=dt['roles']['evaluation']
    def measure():
        a=forward_bundle(m,ev['X'],ev['y_clean'],False);b=forward_bundle(m,resp['X'],None,True)
        return {'loss':a['loss'],'acc':a['acc'],'margin':a['margin'],'entropy':a['entropy'],'resp_logits':b['logits'],'features':b['features']}
    out={0:measure()}
    for h in range(1,max(CORE.horizons)+1):
        fr=frozen_target if (frozen_target is not None and h<=CORE.freeze_steps) else None
        train_step(m,o,dt,sch[start+h-1],start+h-1,family,scale,model_seed,fr)
        if h in CORE.horizons:out[h]=measure()
    return out

def paired_response_rows(control,branch,meta):
    rows=[]
    for h in CORE.horizons:
        c,z=control[h],branch[h];ck=[cka(a,b) for a,b in zip(c['features'],z['features'])]
        dl=z['loss']-c['loss']; lr=float(torch.sqrt(torch.mean((z['resp_logits']-c['resp_logits'])**2)).cpu()); rd=float(np.mean([1-x for x in ck])) if ck else np.nan
        amp=float(meta.get('mean_rmse',0.0)) if meta.get('kind') in SIGNED_KINDS else 1.0
        amp=max(amp,1e-12)
        rows.append({**meta,'horizon':int(h),'delta_loss':dl,'delta_acc':z['acc']-c['acc'],'delta_margin':z['margin']-c['margin'],'delta_entropy':z['entropy']-c['entropy'],'logit_rmse':lr,'mean_rep_displacement':rd,'delta_loss_per_unit':dl/amp,'logit_rmse_per_unit':lr/amp,'mean_rep_displacement_per_unit':rd/amp,'layer_cka_json':json.dumps(ck)})
    return rows

def target_panel(m,n=4):
    names=m.intervention_targets();actual=np.linspace(0,1,len(names));desired=np.linspace(0,1,min(n,len(names)));out=[];used=set()
    for q in desired:
        idx=int(np.argmin([abs(a-q) if i not in used else 999 for i,a in enumerate(actual)]));used.add(idx);out.append((names[idx],float(q),float(actual[idx])))
    return out

def target_panel_positions(m,positions):
    """Resolve a frozen list of normalized target positions to unique architecture-specific targets.

    The requested normalized positions are retained in the query identity. This lets ResNet and ViT
    use architecture-specific layer names while sharing exactly the same macroscopic intervention support.
    """
    names=m.intervention_targets();actual=np.linspace(0,1,len(names));out=[];used=set()
    for q0 in positions:
        q=float(q0);candidates=[(abs(float(a)-q),i) for i,a in enumerate(actual) if i not in used]
        if not candidates:raise RuntimeError(f'Architecture exposes too few distinct intervention targets for frozen support {tuple(positions)}.')
        _,idx=min(candidates);used.add(idx);out.append((names[idx],q,float(actual[idx])))
    if len(out)!=len(tuple(positions)):raise RuntimeError('Frozen target support collapsed to duplicate architecture targets.')
    return out

def experiment_signs(kind):return (-1,1) if kind in SIGNED_KINDS else (0,)

def experiment_reps(kind,reps):return reps if kind in {'noise','orthogonal'} else 1

def expected_uids(task_hash,checkpoint,target_pairs,kinds,reps):
    out=[]
    for target,tpos,tactual in target_pairs:
        for kind in kinds:
            for rep in range(experiment_reps(kind,reps)):
                for sign in experiment_signs(kind):
                    for h in CORE.horizons:
                        out.append(hashlib.sha256(f'{task_hash}|{checkpoint}|{target}|{tpos}|{kind}|{rep}|{sign}|{h}'.encode()).hexdigest()[:24])
    return set(out)

def run_checkpoint_panel(task,dt,family,scale,model_seed,ts,snap,sch,target_pairs,kinds,reps,replicate_id,cohort):
    cm,co=load_snapshot(task,family,scale,model_seed,ts,snap);obs=observational_state(cm,co,dt);control=continuation(cm,co,dt,sch,snap['step'],family,scale,model_seed)
    pieces=[]
    for target,tpos,tactual in target_pairs:
        for kind in kinds:
            for rep in range(experiment_reps(kind,reps)):
                direction_id=stable_seed('measurement_direction',family,scale,model_seed,snap['step'],round(float(tpos),6),kind,rep)
                calibration={'alpha':0.0,'uncapped_alpha':0.0,'minus_rmse':0.0,'plus_rmse':0.0,'mean_rmse':0.0,'sign_asymmetry':0.0,'parameter_fraction':0.0,'local_slope':0.0,'calibration_capped':False,'calibration_shrink_steps':0}
                d=None
                if kind in SIGNED_KINDS:
                    tm,_=load_snapshot(task,family,scale,model_seed,ts,snap);dire=dt['roles']['direction'];d=unit_direction(tm,target,kind,dire['X'],dire['y_clean'],direction_id);calibration=calibrate_symmetric(tm,target,d,dt['roles']['calibration']['X'],CORE.functional_target_rmse);del tm
                for sign in experiment_signs(kind):
                    m,o=load_snapshot(task,family,scale,model_seed,ts,snap)
                    if kind in SIGNED_KINDS:add_delta(m,target,d*(sign*calibration['alpha']))
                    elif kind=='optimizer_reset':reset_optimizer_target(o,m,target)
                    frozen=target if kind=='freeze' else None
                    z=continuation(m,o,dt,sch,snap['step'],family,scale,model_seed,frozen)
                    base_meta={'family':family,'scale':int(scale),'model_seed':int(model_seed),'replicate_id':int(replicate_id),'cohort':cohort,'checkpoint':int(snap['step']),'checkpoint_frac':float(snap['step']/ts.steps),'target':target,'target_pos':float(tpos),'target_actual_pos':float(tactual),'kind':kind,'sign':int(sign),'replicate':int(rep),'direction_id':int(direction_id),'task_hash':task['task_hash'],'suite_hash':task['spec']['suite_hash'],'task_K':int(task['spec']['K']),'task_r':int(task['spec']['r']),'label_noise':float(task['spec']['label_noise']),'task_mode':task['spec']['mode'],'checkpoint_hash':snap['model_hash'],'schedule_hash':snap['schedule_hash'],'core_config_hash':CONFIG_HASH,'protocol_hash':PROTOCOL_HASH,'env_compat_hash':ENV_COMPAT_HASH,'measurement_gradient_label_source':'clean_reference_probe',**calibration,**obs}
                    rows=paired_response_rows(control,z,base_meta)
                    for row in rows:row['experiment_uid']=hashlib.sha256(f"{task['task_hash']}|{snap['step']}|{target}|{tpos}|{kind}|{rep}|{sign}|{row['horizon']}".encode()).hexdigest()[:24]
                    pieces.append(pd.DataFrame(rows));del m,o
    out=pd.concat(pieces,ignore_index=True);expected=expected_uids(task['task_hash'],snap['step'],target_pairs,kinds,reps)
    assert set(out.experiment_uid)==expected and len(out)==len(expected) and not out.experiment_uid.duplicated().any()
    assert np.isfinite(out.select_dtypes(include=[np.number]).to_numpy()).all()
    return out


## Stage-06 function-preserving hidden trainer-state instruments

RC13B keeps the physically clean intervention mechanism from RC12D: only AdamW `exp_avg` is modified at the recipient checkpoint while the model parameters and instantaneous function remain exactly unchanged.

Discovery sees:
- untreated baseline;
- history doses {-1,-0.5,+0.5,+1};
- random-A {-1,+1}.

Confirmation sees:
- untreated baseline;
- history {-1,+1};
- random-A {-1,+1};
- random-B {-1,+1}, where random-B is not constructed until after the discovery lock.

The primary Stage-06 representation no longer mixes heterogeneous query families. It is built only from matched independent noise-direction banks Q_A and Q_B. Gradient, orthogonal, and optimizer-reset probes are still saved as diagnostics for continuity with earlier NEXUS stages.

Every nonzero hidden optimizer-state instrument must additionally pass a prospective one-step central-secant locality gate (cosine >=0.98; relative secant-norm drift <=0.25 at 0.5x/0.75x/1.0). This prevents H6-W/H6-A from relying on a nonlinear dose regime.


In [ ]:
# 2. Function-preserving first-moment instruments, calibration, orthogonality, and field-level audits

def _named_param_map(m):return dict(m.named_parameters())

def _moment_tensor(o,p,key):
    st=o.state.get(p,{})
    if key not in st:return torch.zeros_like(p)
    return st[key]

def _axis_norm(axis):
    return math.sqrt(sum(float((v.detach().float()**2).sum().cpu()) for v in axis.values()))

def _axis_dot(a,b):
    keys=set(a)&set(b)
    return sum(float((a[k].detach().float()*b[k].detach().float()).sum().cpu()) for k in keys)

def _axis_cos(a,b):
    return _axis_dot(a,b)/(_axis_norm(a)*_axis_norm(b)+1e-30)

def _axis_scale(axis,s):
    return {k:v*float(s) for k,v in axis.items()}

def _axis_sub_projection(axis,basis):
    den=_axis_dot(basis,basis)
    if den<=1e-30:return axis
    c=_axis_dot(axis,basis)/den
    return {k:axis[k]-float(c)*basis[k] for k in axis}

def _axis_normalize_to(axis,target_norm):
    n=_axis_norm(axis)
    if n<=1e-30:raise ScientificGateStop('Degenerate first-moment instrument axis.')
    return _axis_scale(axis,float(target_norm)/n)

@torch.no_grad()
def build_m1_axes(rec_m,rec_o,don_m,don_o,replicate_id,need_random_b=True):
    rn=_named_param_map(rec_m);dn=_named_param_map(don_m);assert rn.keys()==dn.keys()
    history={}
    for name,pr in rn.items():
        pd=dn[name];sr=rec_o.state.get(pr,{});sd=don_o.state.get(pd,{})
        if 'exp_avg' in sr:
            md=sd.get('exp_avg',torch.zeros_like(sr['exp_avg'])).to(sr['exp_avg'].device)
            history[name]=(sr['exp_avg'].detach().float()-md.detach().float()).clone()
    hnorm=_axis_norm(history)
    m1norm=math.sqrt(sum(float((rec_o.state[p]['exp_avg'].detach().float()**2).sum().cpu())
                         for p in rn.values() if 'exp_avg' in rec_o.state.get(p,{})))
    if hnorm<=1e-30 or m1norm<=1e-30:
        raise ScientificGateStop(f'Historical exp_avg axis is degenerate: axis_norm={hnorm:.3e}, m1_norm={m1norm:.3e}')

    # Discovery-visible random A: Gaussian; generated on CPU for device-independent determinism.
    raw_a={}
    for name,pr in rn.items():
        st=rec_o.state.get(pr,{})
        if 'exp_avg' not in st:continue
        g=torch.Generator(device='cpu').manual_seed(stable_seed('S6_random_a_tensor',replicate_id,name))
        raw=torch.randn(tuple(st['exp_avg'].shape),generator=g,dtype=torch.float32)
        raw_a[name]=raw.to(st['exp_avg'].device)
    random_a=_axis_sub_projection(raw_a,history)
    random_a=_axis_normalize_to(random_a,hnorm)

    axes={'history':history,'random_a':random_a}
    if need_random_b:
        # Confirmation-only random B uses a different distribution and is orthogonal to both frozen discovery axes.
        raw_b={}
        for name,pr in rn.items():
            st=rec_o.state.get(pr,{})
            if 'exp_avg' not in st:continue
            g=torch.Generator(device='cpu').manual_seed(stable_seed('S6_random_b_tensor',replicate_id,name))
            raw=(torch.randint(0,2,tuple(st['exp_avg'].shape),generator=g,dtype=torch.int8).float()*2-1)
            raw_b[name]=raw.to(st['exp_avg'].device)
        random_b=_axis_sub_projection(raw_b,history)
        random_b=_axis_sub_projection(random_b,random_a)
        random_b=_axis_normalize_to(random_b,hnorm)
        axes['random_b']=random_b

    meta={}
    for k,a in axes.items():
        meta[k]={
            'axis_norm':float(_axis_norm(a)),
            'axis_relative_to_recipient_m1':float(_axis_norm(a)/max(m1norm,1e-30)),
            'axis_sha256':hash_tree(tree_to_cpu(a)),
            'seed_recipe':('historical_recipient_minus_donor' if k=='history' else f'deterministic_{k}_{replicate_id}')
        }
    meta['history']['cos_history']=1.0
    meta['random_a']['cos_history']=float(_axis_cos(random_a,history))
    if 'random_b' in axes:
        meta['random_b']['cos_history']=float(_axis_cos(axes['random_b'],history))
        meta['random_b']['cos_random_a']=float(_axis_cos(axes['random_b'],random_a))
    if abs(meta['random_a']['cos_history'])>CORE.axis_orthogonality_abs_cos_max:
        raise RuntimeError(f'random_a failed orthogonality: {meta["random_a"]["cos_history"]}')
    if 'random_b' in axes and (abs(meta['random_b']['cos_history'])>CORE.axis_orthogonality_abs_cos_max or
                               abs(meta['random_b']['cos_random_a'])>CORE.axis_orthogonality_abs_cos_max):
        raise RuntimeError(f'random_b failed orthogonality: {meta["random_b"]}')
    return axes,meta


def build_anisotropy_axes(rec_m,rec_o,history_axis,replicate_id,n_random):
    """Deterministic Gaussian axes in the orthogonal complement of history.

    Random panel axes are also sequentially orthogonalized against earlier panel axes so the
    comparison is not accidentally dominated by near-duplicate random directions.
    """
    rn=_named_param_map(rec_m);hnorm=_axis_norm(history_axis)
    m1norm=math.sqrt(sum(float((rec_o.state[p]['exp_avg'].detach().float()**2).sum().cpu())
                         for p in rn.values() if 'exp_avg' in rec_o.state.get(p,{})))
    if hnorm<=1e-30 or m1norm<=1e-30:raise ScientificGateStop('Degenerate history/m1 norm in anisotropy panel.')
    axes={};metas={};basis=[history_axis]
    for j in range(int(n_random)):
        raw={}
        for name,pr in rn.items():
            st=rec_o.state.get(pr,{})
            if 'exp_avg' not in st:continue
            g=torch.Generator(device='cpu').manual_seed(stable_seed('S6_anisotropy_tensor',replicate_id,j,name))
            raw[name]=torch.randn(tuple(st['exp_avg'].shape),generator=g,dtype=torch.float32).to(st['exp_avg'].device)
        v=raw
        for b in basis:v=_axis_sub_projection(v,b)
        if _axis_norm(v)<=1e-20:raise ScientificGateStop(f'Anisotropy random axis {j} degenerate after orthogonalization.')
        v=_axis_normalize_to(v,hnorm)
        if abs(_axis_cos(v,history_axis))>CORE.axis_orthogonality_abs_cos_max:
            raise RuntimeError(f'Anisotropy random axis {j} failed history orthogonality.')
        max_prev=max([abs(_axis_cos(v,b)) for b in basis[1:]]+[0.0])
        if max_prev>CORE.axis_orthogonality_abs_cos_max:
            raise RuntimeError(f'Anisotropy random axis {j} failed panel orthogonality: {max_prev}')
        key=f'random_{j:02d}';axes[key]=v;basis.append(v)
        metas[key]={
            'axis_norm':float(_axis_norm(v)),
            'axis_relative_to_recipient_m1':float(_axis_norm(v)/max(m1norm,1e-30)),
            'cos_history':float(_axis_cos(v,history_axis)),
            'max_abs_cos_previous_random':float(max_prev),
            'axis_sha256':hash_tree(tree_to_cpu(v)),
            'seed_recipe':f'deterministic_anisotropy_gaussian_{replicate_id}_{j}',
        }
    return axes,metas

@torch.no_grad()
def apply_m1_axis_surgery(m,o,axis,signed_scale):
    nm=_named_param_map(m);changed=0
    for name,d in axis.items():
        p=nm[name];st=o.state.get(p,{})
        if 'exp_avg' not in st:continue
        st['exp_avg'].add_(float(signed_scale)*d.to(st['exp_avg'].device,dtype=st['exp_avg'].dtype))
        changed+=1
    if changed==0:raise RuntimeError('First-moment instrument changed zero optimizer tensors.')
    return changed

def _field_tree(m,o,key):
    out={};nm=_named_param_map(m)
    for name,p in nm.items():
        st=o.state.get(p,{})
        if key in st:out[name]=tree_to_cpu(st[key])
    return out

def optimizer_field_fingerprints(m,o):
    sd=o.state_dict()
    exp_avg=_field_tree(m,o,'exp_avg')
    exp_avg_sq=_field_tree(m,o,'exp_avg_sq')
    step=_field_tree(m,o,'step')
    other={}
    nm=_named_param_map(m)
    for name,p in nm.items():
        st=o.state.get(p,{})
        extra={k:tree_to_cpu(v) for k,v in st.items() if k not in ('exp_avg','exp_avg_sq','step')}
        if extra:other[name]=extra
    return {
        'exp_avg_hash':hash_tree(exp_avg),
        'exp_avg_sq_hash':hash_tree(exp_avg_sq),
        'step_hash':hash_tree(step),
        'other_state_hash':hash_tree(other),
        'param_groups_hash':hash_tree(tree_to_cpu(sd['param_groups'])),
        'full_optimizer_hash':optimizer_hash(o),
    }

def _one_step_logits_axis(task,family,scale,model_seed,ts,recipient_snap,sch,dt,axis=None,signed_scale=0.0):
    m,o=load_snapshot(task,family,scale,model_seed,ts,recipient_snap)
    if axis is not None and float(signed_scale)!=0:
        apply_m1_axis_surgery(m,o,axis,float(signed_scale))
    train_step(m,o,dt,sch[recipient_snap['step']],recipient_snap['step'],family,scale,model_seed)
    out=forward_bundle(m,dt['roles']['calibration']['X'],None,False)['logits'].detach().cpu()
    del m,o
    return out


def hidden_axis_central_secant_locality(task,family,scale,model_seed,ts,recipient_snap,sch,dt,axis,alpha):
    """Prospective one-step locality of an optimizer-state intervention.

    The intervention is applied to exp_avg, then exactly one frozen training update is taken.
    Central logit secants at 0.5x/0.75x are compared with 1x. This prevents a potency or
    causal-state claim from relying on a strongly nonlinear hidden-state dose regime.
    """
    vecs={}
    for fac in CORE.hidden_locality_dose_factors:
        zm=_one_step_logits_axis(task,family,scale,model_seed,ts,recipient_snap,sch,dt,axis,-float(fac)*float(alpha)).reshape(-1).float()
        zp=_one_step_logits_axis(task,family,scale,model_seed,ts,recipient_snap,sch,dt,axis,+float(fac)*float(alpha)).reshape(-1).float()
        vecs[float(fac)]=(zp-zm)/(2.0*float(fac)*max(float(alpha),1e-30))
    ref=vecs[1.0];rn=float(torch.linalg.vector_norm(ref))
    cos=[];drift=[]
    for fac in CORE.hidden_locality_dose_factors:
        v=vecs[float(fac)];vn=float(torch.linalg.vector_norm(v))
        c=float(torch.dot(v,ref)/(torch.linalg.vector_norm(v)*torch.linalg.vector_norm(ref)+1e-30))
        cos.append(c);drift.append(abs(vn-rn)/max(rn,1e-30))
    out={'hidden_min_secant_cosine':float(min(cos)),
         'hidden_max_relative_secant_norm_drift':float(max(drift))}
    for fac,c,dr in zip(CORE.hidden_locality_dose_factors,cos,drift):
        tag=f'{float(fac):.2f}'.replace('.','p')
        out[f'hidden_secant_cosine_{tag}']=float(c)
        out[f'hidden_secant_relative_norm_drift_{tag}']=float(dr)
    return out


def calibrate_hidden_axis(task,family,scale,model_seed,ts,recipient_snap,sch,dt,axis,instrument_name):
    base=_one_step_logits_axis(task,family,scale,model_seed,ts,recipient_snap,sch,dt,None,0.0)
    rm,ro=load_snapshot(task,family,scale,model_seed,ts,recipient_snap)
    axis_norm=_axis_norm(axis);m1norm=optimizer_moment_norms(ro)[0]
    cap=1.0 if axis_norm<=1e-30 else min(1.0,CORE.surgery_max_m1_fraction*max(m1norm,1e-30)/axis_norm)
    del rm,ro
    def effects(a):
        vals=[]
        for sign in (-1,1):
            z=_one_step_logits_axis(task,family,scale,model_seed,ts,recipient_snap,sch,dt,axis,sign*a)
            vals.append(float(torch.sqrt(torch.mean((z-base)**2))))
        return vals
    a0=max(1e-5,min(cap,0.05));em,ep=effects(a0);slope=(em+ep)/(2*a0)
    if not np.isfinite(slope) or slope<=1e-12:
        raise ScientificGateStop(f'{instrument_name} surgery prospectively non-measurable: local slope={slope}')
    a=min(cap,CORE.surgery_target_one_step_rmse/slope)
    for _ in range(8):
        em,ep=effects(a);mean=(em+ep)/2
        if mean<=1e-12:break
        relerr=abs(mean-CORE.surgery_target_one_step_rmse)/CORE.surgery_target_one_step_rmse
        if relerr<=CORE.surgery_rel_tol:break
        a=min(cap,a*float(np.clip(CORE.surgery_target_one_step_rmse/mean,.4,2.5)))
    em,ep=effects(a);mean=(em+ep)/2;asym=abs(ep-em)/(mean+1e-12)
    relerr=abs(mean-CORE.surgery_target_one_step_rmse)/CORE.surgery_target_one_step_rmse
    if mean<CORE.surgery_min_one_step_rmse:
        raise ScientificGateStop(f'{instrument_name} surgery below minimum prospective amplitude: {mean:.3e}')
    if relerr>CORE.surgery_rel_tol:
        raise ScientificGateStop(f'{instrument_name} surgery failed frozen functional target: relative error={relerr:.4f}')
    if asym>CORE.surgery_sign_asymmetry_max:
        raise ScientificGateStop(f'{instrument_name} surgery sign asymmetry too large: {asym:.3f}')
    locality=hidden_axis_central_secant_locality(task,family,scale,model_seed,ts,recipient_snap,sch,dt,axis,a)
    if locality['hidden_min_secant_cosine']<CORE.hidden_locality_min_secant_cosine:
        raise ScientificGateStop(f'{instrument_name} hidden-state secant direction is not local enough: cosine={locality["hidden_min_secant_cosine"]:.6f}')
    if locality['hidden_max_relative_secant_norm_drift']>CORE.hidden_locality_max_relative_norm_drift:
        raise ScientificGateStop(f'{instrument_name} hidden-state secant magnitude is not local enough: drift={locality["hidden_max_relative_secant_norm_drift"]:.6f}')
    return {
        'instrument_family':instrument_name,'alpha':float(a),'cap':float(cap),
        'minus_one_step_rmse':float(em),'plus_one_step_rmse':float(ep),
        'mean_one_step_rmse':float(mean),'target_relative_error':float(relerr),'sign_asymmetry':float(asym),
        'axis_norm':float(axis_norm),'recipient_m1_norm':float(m1norm),
        'relative_m1_displacement_at_unit_dose':float(a*axis_norm/max(m1norm,1e-30)),
        'calibration_capped':bool(a>=cap*(1-1e-8)),**locality,
    }

def make_surgery_snapshot(task,dt,family,scale,model_seed,ts,recipient_snap,instrument_family,dose,alpha,axis=None,axis_meta=None):
    m,o=load_snapshot(task,family,scale,model_seed,ts,recipient_snap)
    before_model=model_hash(m);before_fields=optimizer_field_fingerprints(m,o)
    before_logits=forward_bundle(m,dt['roles']['evaluation']['X'],None,False)['logits'].detach().cpu()
    if axis is not None and float(dose)!=0:
        apply_m1_axis_surgery(m,o,axis,float(dose)*float(alpha))
    after_logits=forward_bundle(m,dt['roles']['evaluation']['X'],None,False)['logits'].detach().cpu()
    inv=float(torch.max(torch.abs(after_logits-before_logits)))
    after_model=model_hash(m);after_fields=optimizer_field_fingerprints(m,o)

    function_ok=bool(after_model==before_model==recipient_snap['model_hash'] and inv==0.0)
    non_expavg_ok=bool(
        after_fields['exp_avg_sq_hash']==before_fields['exp_avg_sq_hash'] and
        after_fields['step_hash']==before_fields['step_hash'] and
        after_fields['other_state_hash']==before_fields['other_state_hash'] and
        after_fields['param_groups_hash']==before_fields['param_groups_hash']
    )
    expavg_changed=bool(after_fields['exp_avg_hash']!=before_fields['exp_avg_hash'])
    expected_change=bool(axis is not None and float(dose)!=0)
    expavg_semantics_ok=bool(expavg_changed==expected_change)
    if not function_ok:raise RuntimeError('Hidden-state surgery changed model parameters or instantaneous logits.')
    if not non_expavg_ok:raise RuntimeError('Hidden-state surgery changed an optimizer field outside exp_avg.')
    if not expavg_semantics_ok:raise RuntimeError('Hidden-state surgery exp_avg change semantics failed.')

    snap=snapshot(m,o,recipient_snap['step'],{k:v for k,v in recipient_snap.items() if k not in ('model','optimizer','model_hash','optimizer_hash','step')})
    snap.update({
        'parent_model_hash':recipient_snap['model_hash'],'parent_optimizer_hash':before_fields['full_optimizer_hash'],
        'instrument_family':instrument_family,'dose':float(dose),'alpha':float(alpha),
        'instantaneous_max_logit_diff':inv,'function_invariance_passed':function_ok,
        'optimizer_isolation_passed':bool(non_expavg_ok and expavg_semantics_ok),
        'exp_avg_changed':expavg_changed,
        'before_exp_avg_hash':before_fields['exp_avg_hash'],'after_exp_avg_hash':after_fields['exp_avg_hash'],
        'exp_avg_sq_unchanged':after_fields['exp_avg_sq_hash']==before_fields['exp_avg_sq_hash'],
        'step_unchanged':after_fields['step_hash']==before_fields['step_hash'],
        'other_optimizer_state_unchanged':after_fields['other_state_hash']==before_fields['other_state_hash'],
        'param_groups_unchanged':after_fields['param_groups_hash']==before_fields['param_groups_hash'],
        'axis_meta':copy.deepcopy(axis_meta or {}),
    })
    del m,o
    return snap


In [ ]:
# 3. Stage-06 Q measurement at one manipulated microstate
S6_STATE_COLS=[
    'family','scale','replicate_id','cohort','task_hash','suite_hash','task_K','task_r','label_noise','task_mode',
    'checkpoint','checkpoint_frac','branch_id','instrument_family','surgery_dose','surgery_alpha'
]
S6_QUERY_BASE=['kind','target_pos','sign','replicate','horizon']

def s6_long_responses(df,metrics=CORE.primary_response_metrics):
    parts=[]
    for metric in metrics:
        x=df[S6_STATE_COLS+S6_QUERY_BASE+[metric]].copy()
        x['response_metric']=metric;x=x.rename(columns={metric:'response_value'});parts.append(x)
    return pd.concat(parts,ignore_index=True)

def s6_response_matrix(df):
    key=S6_STATE_COLS+['kind','target_pos','sign','replicate','horizon']
    dup=df.duplicated(key,keep=False)
    if bool(dup.any()):raise RuntimeError(f'Duplicate Stage-06 physical query rows detected; refusing silent averaging: {int(dup.sum())} rows.')
    w=s6_long_responses(df)
    long_key=S6_STATE_COLS+['kind','target_pos','sign','replicate','horizon','response_metric']
    if bool(w.duplicated(long_key,keep=False).any()):raise RuntimeError('Duplicate Stage-06 long-form response cells detected.')
    M=w.pivot(index=S6_STATE_COLS,columns=['kind','target_pos','sign','replicate','horizon','response_metric'],values='response_value').sort_index(axis=1)
    if M.isna().any().any():raise RuntimeError(f'Incomplete Stage-06 response matrix: {int(M.isna().sum().sum())} missing values.')
    return M


def s6_state_observables(df):
    cols=S6_STATE_COLS+OBS_COLS
    nun=df[cols].groupby(S6_STATE_COLS,dropna=False)[OBS_COLS].nunique(dropna=False)
    if len(nun) and int(nun.to_numpy().max())>1:raise RuntimeError('Stage-06 challenger descriptors vary within a single branch state.')
    out=df.sort_values('horizon').drop_duplicates(S6_STATE_COLS)[cols].set_index(S6_STATE_COLS)
    if out.isna().any().any():raise RuntimeError('Stage-06 challenger state contains missing values.')
    return out


def s6_query_partition(columns):
    """Primary Stage-06 matched physical views.

    Q_A and Q_B are statistically matched independent noise-direction banks at the same
    positive short horizons. Q_T is a disjoint noise direction at the frozen future horizon.
    Gradient/orthogonal/reset measurements remain legacy diagnostics only.
    """
    qa=[];qb=[];qt=[]
    for c in list(columns):
        kind=str(c[0]);qrep=int(c[3]);h=int(c[4])
        if kind=='noise' and qrep==CORE.qA_noise_replicate and h in set(CORE.context_horizons):qa.append(c)
        if kind=='noise' and qrep==CORE.qB_noise_replicate and h in set(CORE.context_horizons):qb.append(c)
        if kind=='noise' and qrep==CORE.qT_noise_replicate and h==CORE.primary_horizon:qt.append(c)
    key=lambda c:(float(c[1]),int(c[2]),int(c[4]),str(c[5]))
    qa=sorted(qa,key=key);qb=sorted(qb,key=key);qt=sorted(qt,key=key)
    if not qa or not qb or not qt:raise RuntimeError(f'Frozen matched Q partition empty: A={len(qa)} B={len(qb)} T={len(qt)}')
    if len(qa)!=len(qb):raise RuntimeError(f'Matched Q_A/Q_B size mismatch: {len(qa)} != {len(qb)}')
    if set(qa)&set(qb) or set(qa)&set(qt) or set(qb)&set(qt):raise RuntimeError('Stage-06 physical query-bank overlap detected.')
    # Structural equivalence after removing direction-replicate identity.
    ka=[(float(c[1]),int(c[2]),int(c[4]),str(c[5])) for c in qa]
    kb=[(float(c[1]),int(c[2]),int(c[4]),str(c[5])) for c in qb]
    if ka!=kb:raise RuntimeError('Q_A and Q_B are not structurally matched.')
    return qa,qb,qt

def target_optimizer_state_descriptors(m,o,target_pairs):
    out={}
    for i,(target,_,_) in enumerate(target_pairs):
        pn=m1n=m2n=un=0.0
        for p in target_parameters(m,target):
            pn+=float((p.detach().float()**2).sum().cpu())
            st=o.state.get(p,{})
            if 'exp_avg' in st:m1n+=float((st['exp_avg'].detach().float()**2).sum().cpu())
            if 'exp_avg_sq' in st:
                vv=st['exp_avg_sq'].detach().float();m2n+=float((vv*vv).sum().cpu())
                if 'exp_avg' in st:
                    u=st['exp_avg'].detach().float()/(vv.sqrt()+CORE.adam_eps);un+=float((u*u).sum().cpu())
        out[f'control_target_param_norm_{i}']=math.sqrt(pn)
        out[f'control_target_m1_norm_{i}']=math.sqrt(m1n)
        out[f'control_target_m2_norm_{i}']=math.sqrt(m2n)
        out[f'control_target_preconditioned_norm_{i}']=math.sqrt(un)
    if set(out)!=set(MICRO_TARGET_COLS):
        raise RuntimeError('Target microscopic descriptor schema mismatch.')
    return out

def _q_calibration_pass(kind,cal):
    if kind not in SIGNED_KINDS:return True
    return bool(
        float(cal['mean_rmse'])>=CORE.min_functional_rmse and
        float(cal['sign_asymmetry'])<=CORE.calibration_asymmetry_max and
        float(cal['parameter_fraction'])<=CORE.max_parameter_fraction*(1+1e-10) and
        float(cal.get('min_secant_cosine',1.0))>=CORE.q_locality_min_secant_cosine and
        float(cal.get('max_relative_secant_norm_drift',0.0))<=CORE.q_locality_max_relative_norm_drift
    )



def _rc13c_noise_direction_id(family,scale,model_seed,recipient_step,tpos,qrep,candidate_index):
    base=('S6_measurement_direction',family,scale,model_seed,recipient_step,round(float(tpos),6),'noise',int(qrep))
    if int(candidate_index)==0:
        return stable_seed(*base)
    return stable_seed(*base,RC13C_Q_REJECTION_NAMESPACE,int(candidate_index))

def build_q_bank(task,dt,family,scale,model_seed,ts,recipient_snap,q_plan_entry=None):
    """Freeze one physically matched Q bank; RC13C completion uses the prelocked accepted candidate."""
    candidate_index=0 if q_plan_entry is None else int(q_plan_entry['accepted_candidate'])
    m,o=load_snapshot(task,family,scale,model_seed,ts,recipient_snap)
    pairs=target_panel_positions(m,CORE.target_positions)
    dire=dt['roles']['direction'];Xcal=dt['roles']['calibration']['X'];entries={}
    for target,tpos,tactual in pairs:
        for kind,qrep in [('noise',0),('noise',1),('noise',2),('gradient',0),('orthogonal',0),('optimizer_reset',0)]:
            if kind=='noise':
                direction_id=_rc13c_noise_direction_id(family,scale,model_seed,recipient_snap['step'],tpos,qrep,candidate_index)
            else:
                direction_id=stable_seed('S6_measurement_direction',family,scale,model_seed,recipient_snap['step'],round(float(tpos),6),kind,qrep)
            cal={'alpha':0.0,'uncapped_alpha':0.0,'minus_rmse':0.0,'plus_rmse':0.0,'mean_rmse':0.0,
                 'sign_asymmetry':0.0,'parameter_fraction':0.0,'local_slope':0.0,
                 'calibration_capped':False,'calibration_shrink_steps':0,
                 'min_secant_cosine':1.0,'max_relative_secant_norm_drift':0.0,
                 'secant_cosine_0p50':1.0,'secant_relative_norm_drift_0p50':0.0,
                 'secant_cosine_0p75':1.0,'secant_relative_norm_drift_0p75':0.0,
                 'secant_cosine_1p00':1.0,'secant_relative_norm_drift_1p00':0.0}
            d_cpu=None
            if kind in SIGNED_KINDS:
                d=unit_direction(m,target,kind,dire['X'],dire['y_clean'],direction_id)
                cal=calibrate_symmetric(m,target,d,Xcal,CORE.functional_target_rmse)
                d_cpu=d.detach().cpu().clone()
            if not _q_calibration_pass(kind,cal):
                raise ScientificGateStop(f'Q calibration gate failed while freezing Q: {kind} target={target}.')
            entries[(target,kind,int(qrep))]={'direction_id':int(direction_id),'calibration':copy.deepcopy(cal),'direction_cpu':d_cpu}

    noise_cosines={};global_max=0.0;noise_ids={}
    for target,_,_ in pairs:
        vec=[entries[(target,'noise',r)]['direction_cpu'].reshape(-1).float() for r in (0,1,2)]
        pairvals={}
        for i,j,label in ((0,1,'A_T'),(0,2,'A_B'),(1,2,'T_B')):
            c=float(torch.dot(vec[i],vec[j])/(torch.linalg.vector_norm(vec[i])*torch.linalg.vector_norm(vec[j])+1e-30))
            pairvals[label]=c;global_max=max(global_max,abs(c))
        noise_cosines[str(target)]=pairvals
        noise_ids[str(target)]={str(r):int(entries[(target,'noise',r)]['direction_id']) for r in (0,1,2)}
    if global_max>CORE.q_noise_max_abs_cosine:
        raise ScientificGateStop(f'Matched Q noise directions insufficiently independent after RC13C qualification: max |cos|={global_max:.6f} > {CORE.q_noise_max_abs_cosine:.6f}.')

    if q_plan_entry is not None:
        if int(q_plan_entry['accepted_candidate'])!=candidate_index:
            raise RuntimeError('RC13C Q-plan candidate index drift.')
        if q_plan_entry.get('noise_direction_ids')!=noise_ids:
            raise RuntimeError('RC13C Q-plan direction-ID replay mismatch.')
        if abs(float(q_plan_entry['noise_max_abs_cosine'])-float(global_max))>2e-6:
            raise RuntimeError('RC13C Q-plan cosine replay mismatch.')
        planned=q_plan_entry.get('noise_pairwise_cosines',{})
        for target,pvals in noise_cosines.items():
            for k,v in pvals.items():
                if abs(float(planned[target][k])-float(v))>2e-6:
                    raise RuntimeError(f'RC13C Q-plan pairwise-cosine replay mismatch at {target}/{k}.')

    payload={'recipient_model_hash':recipient_snap['model_hash'],
             'pairs':[(str(a),float(b),float(c)) for a,b,c in pairs],'entries':entries,
             'noise_pairwise_cosines':noise_cosines,'noise_max_abs_cosine':float(global_max),
             'q_candidate_index':int(candidate_index),'noise_direction_ids':noise_ids}
    payload['q_bank_sha256']=hash_tree({
        'recipient_model_hash':payload['recipient_model_hash'],'pairs':payload['pairs'],
        'noise_pairwise_cosines':payload['noise_pairwise_cosines'],'noise_max_abs_cosine':payload['noise_max_abs_cosine'],
        'q_candidate_index':payload['q_candidate_index'],'noise_direction_ids':noise_ids,
        'entries':{str(k):{'direction_id':v['direction_id'],'calibration':v['calibration'],'direction':v['direction_cpu']} for k,v in entries.items()}
    })
    del m,o
    return payload

def measure_branch(task,dt,family,scale,model_seed,ts,branch_snap,sch,replicate_id,cohort,
                   branch_id,instrument_family,dose,alpha,instrument_calibration=None,axis_meta=None,q_bank=None):
    cm,co=load_snapshot(task,family,scale,model_seed,ts,branch_snap)
    if q_bank is None:raise RuntimeError('RC13B branch measurement requires a frozen replicate-level Q bank.')
    if model_hash(cm)!=q_bank['recipient_model_hash'] or branch_snap['model_hash']!=q_bank['recipient_model_hash']:
        raise RuntimeError('Hidden-state branch model differs from the frozen replicate-level Q model.')
    pairs=[(a,float(b),float(c)) for a,b,c in q_bank['pairs']]
    obs=observational_state(cm,co,dt);obs.update(target_optimizer_state_descriptors(cm,co,pairs))
    control=continuation(cm,co,dt,sch,branch_snap['step'],family,scale,model_seed)
    del cm,co

    query_specs=[('noise',0),('noise',1),('noise',2),('gradient',0),('orthogonal',0),('optimizer_reset',0)]
    pieces=[]
    for target,tpos,tactual in pairs:
        for kind,qrep in query_specs:
            qe=q_bank['entries'][(target,kind,int(qrep))]
            direction_id=int(qe['direction_id']);cal=copy.deepcopy(qe['calibration'])
            d=None if qe['direction_cpu'] is None else qe['direction_cpu'].to(DEVICE)
            qpass=_q_calibration_pass(kind,cal)
            if not qpass:raise ScientificGateStop(f'Frozen Q calibration invalid for {kind} target={target} branch={branch_id}.')
            for sign in experiment_signs(kind):
                m,o=load_snapshot(task,family,scale,model_seed,ts,branch_snap)
                if kind in SIGNED_KINDS:add_delta(m,target,d*(sign*cal['alpha']))
                elif kind=='optimizer_reset':reset_optimizer_target(o,m,target)
                z=continuation(m,o,dt,sch,branch_snap['step'],family,scale,model_seed)
                ical=instrument_calibration or {};am=axis_meta or {}
                meta={
                    'family':family,'scale':int(scale),'replicate_id':int(replicate_id),'cohort':cohort,
                    'task_hash':task['task_hash'],'suite_hash':task['spec']['suite_hash'],
                    'task_K':int(task['spec']['K']),'task_r':int(task['spec']['r']),
                    'label_noise':float(task['spec']['label_noise']),'task_mode':task['spec']['mode'],
                    'checkpoint':int(branch_snap['step']),'checkpoint_frac':float(branch_snap['step']/ts.steps),
                    'branch_id':branch_id,'instrument_family':instrument_family,
                    'surgery_dose':float(dose),'surgery_alpha':float(alpha),
                    'instantaneous_max_logit_diff':float(branch_snap['instantaneous_max_logit_diff']),
                    'function_invariance_passed':bool(branch_snap['function_invariance_passed']),
                    'optimizer_isolation_passed':bool(branch_snap['optimizer_isolation_passed']),
                    'exp_avg_changed':bool(branch_snap['exp_avg_changed']),
                    'exp_avg_sq_unchanged':bool(branch_snap['exp_avg_sq_unchanged']),
                    'optimizer_step_unchanged':bool(branch_snap['step_unchanged']),
                    'other_optimizer_state_unchanged':bool(branch_snap['other_optimizer_state_unchanged']),
                    'param_groups_unchanged':bool(branch_snap['param_groups_unchanged']),
                    'instrument_one_step_rmse':float(ical.get('mean_one_step_rmse',0.0)),
                    'instrument_sign_asymmetry':float(ical.get('sign_asymmetry',0.0)),
                    'instrument_hidden_min_secant_cosine':float(ical.get('hidden_min_secant_cosine',1.0)),
                    'instrument_hidden_max_relative_secant_norm_drift':float(ical.get('hidden_max_relative_secant_norm_drift',0.0)),
                    'instrument_relative_m1_displacement':float(ical.get('relative_m1_displacement_at_unit_dose',0.0))*abs(float(dose)),
                    'instrument_axis_norm':float(am.get('axis_norm',0.0)),
                    'instrument_axis_relative_to_recipient_m1':float(am.get('axis_relative_to_recipient_m1',0.0)),
                    'instrument_axis_cos_history':float(am.get('cos_history',0.0)),
                    'instrument_axis_cos_random_a':float(am.get('cos_random_a',0.0)),
                    'q_calibration_passed':bool(qpass),'q_bank_sha256':str(q_bank['q_bank_sha256']),
                    'q_candidate_index':int(q_bank.get('q_candidate_index',0)),
                    'q_noise_max_abs_cosine':float(q_bank['noise_max_abs_cosine']),
                    'target':target,'target_pos':float(tpos),'target_actual_pos':float(tactual),
                    'kind':kind,'sign':int(sign),'replicate':int(qrep),'direction_id':int(direction_id),
                    'checkpoint_hash':branch_snap['model_hash'],'optimizer_hash':branch_snap['optimizer_hash'],
                    'protocol_hash':PROTOCOL_HASH,'env_compat_hash':ENV_COMPAT_HASH,
                    **cal,**obs
                }
                pieces.append(pd.DataFrame(paired_response_rows(control,z,meta)));del m,o
    out=pd.concat(pieces,ignore_index=True)
    if not np.isfinite(out.select_dtypes(include=[np.number]).to_numpy()).all():raise RuntimeError('Non-finite value in Stage-06 branch table.')
    return out


def _make_s6_task(rep):
    suite=make_suite(SuiteSpec(seed=stable_seed('S6_suite',rep)))
    task,_=make_task_variant(suite,32,4,0.0,'fixedK',rep)
    return task

def run_s6_replicate(rep,cohort):
    ts=s6_train_spec();scale=2 if PROFILE=='research' else 1
    task=_make_s6_task(rep);bank,sch,dt,final_eval=build_checkpoint_bank(task,'resnet',scale,rep,ts)
    donor_step=int(round(CORE.donor_frac*ts.steps));recipient_step=int(round(CORE.recipient_frac*ts.steps))
    donor=bank[donor_step];recipient=bank[recipient_step]

    rm,ro=load_snapshot(task,'resnet',scale,rep,ts,recipient);recipient_eval=evaluate_public(rm,dt)
    dm,do=load_snapshot(task,'resnet',scale,rep,ts,donor)
    axes,axes_meta=build_m1_axes(rm,ro,dm,do,rep,need_random_b=(cohort=='confirmation'))
    anis_axes=anis_meta=None
    if cohort=='confirmation':
        anis_axes,anis_meta=build_anisotropy_axes(rm,ro,axes['history'],rep,CORE.anisotropy_random_directions)
    del rm,ro,dm,do
    q_plan_entry=None
    if cohort=='confirmation' and int(rep) in RC13C_COMPLETION_REPLICATES:
        if not RC13C_Q_PLAN_LOCKED or RC13C_Q_PLAN is None or str(int(rep)) not in RC13C_Q_PLAN['replicates']:
            raise RuntimeError(f'RC13C completion rep {rep} cannot run before the frozen Q-plan lock.')
        q_plan_entry=RC13C_Q_PLAN['replicates'][str(int(rep))]
    q_bank=build_q_bank(task,dt,'resnet',scale,rep,ts,recipient,q_plan_entry=q_plan_entry)

    cals={}
    for inst in ('history','random_a')+(('random_b',) if cohort=='confirmation' else tuple()):
        cals[inst]=calibrate_hidden_axis(task,'resnet',scale,rep,ts,recipient,sch,dt,axes[inst],inst)

    # H6-A is confirmation-only and logically independent of H6-B/H6-W. A failed random calibration
    # is recorded as an invalid anisotropy panel entry rather than aborting the other two claims.
    anisotropy_panel=None
    if cohort=='confirmation':
        anisotropy_panel={'history':copy.deepcopy(cals['history']),'random':[],'n_requested':int(CORE.anisotropy_random_directions)}
        for key in sorted(anis_axes):
            try:
                cc=calibrate_hidden_axis(task,'resnet',scale,rep,ts,recipient,sch,dt,anis_axes[key],f'anisotropy_{key}')
                rec={'axis_id':key,'valid':True,'calibration':cc,'axis_meta':anis_meta[key]}
            except ScientificGateStop as e:
                rec={'axis_id':key,'valid':False,'error':str(e),'axis_meta':anis_meta[key]}
            anisotropy_panel['random'].append(rec)

    history_doses=CORE.discovery_history_doses if cohort=='discovery' else CORE.confirmation_history_doses
    specs=[('baseline',0.0,0.0,None,None)]
    specs += [('history',float(d),cals['history']['alpha'],axes['history'],axes_meta['history']) for d in history_doses]
    specs += [('random_a',float(d),cals['random_a']['alpha'],axes['random_a'],axes_meta['random_a']) for d in CORE.random_a_doses]
    if cohort=='confirmation':
        specs += [('random_b',float(d),cals['random_b']['alpha'],axes['random_b'],axes_meta['random_b']) for d in CORE.random_b_doses]

    parts=[];branch_meta=[]
    for inst,dose,alpha,axis,ameta in specs:
        branch_id='baseline_0' if inst=='baseline' else f'{inst}_{dose:+.2f}'
        snap=make_surgery_snapshot(task,dt,'resnet',scale,rep,ts,recipient,inst,dose,alpha,axis,ameta)
        part=measure_branch(task,dt,'resnet',scale,rep,ts,snap,sch,rep,cohort,branch_id,inst,dose,alpha,cals.get(inst),ameta,q_bank=q_bank)
        parts.append(part)
        branch_meta.append({'replicate_id':int(rep),'cohort':cohort,'branch_id':branch_id,'instrument_family':inst,
            'dose':float(dose),'alpha':float(alpha),'instantaneous_max_logit_diff':float(snap['instantaneous_max_logit_diff']),
            'function_invariance_passed':bool(snap['function_invariance_passed']),'optimizer_isolation_passed':bool(snap['optimizer_isolation_passed']),
            'exp_avg_changed':bool(snap['exp_avg_changed']),'exp_avg_sq_unchanged':bool(snap['exp_avg_sq_unchanged']),
            'step_unchanged':bool(snap['step_unchanged']),'other_optimizer_state_unchanged':bool(snap['other_optimizer_state_unchanged']),
            'param_groups_unchanged':bool(snap['param_groups_unchanged']),'model_hash':snap['model_hash'],'optimizer_hash':snap['optimizer_hash'],
            'axis_meta':ameta or {},'q_bank_sha256':q_bank['q_bank_sha256'],'q_candidate_index':int(q_bank.get('q_candidate_index',0))})
        print('done S6',cohort,'rep',rep,branch_id,'rows',len(part))

    out=pd.concat(parts,ignore_index=True)
    q_per_target=(2*3)+2+2+1;expected_rows=len(specs)*len(CORE.target_positions)*q_per_target*len(CORE.horizons)
    key=S6_STATE_COLS+['kind','target_pos','sign','replicate','horizon']
    if len(out)!=expected_rows:raise RuntimeError(f'RC13B replicate row count mismatch: {len(out)} != {expected_rows}')
    if bool(out.duplicated(key,keep=False).any()):raise RuntimeError('RC13B replicate contains duplicate physical query rows.')
    if out.q_bank_sha256.nunique()!=1 or str(out.q_bank_sha256.iloc[0])!=str(q_bank['q_bank_sha256']):
        raise RuntimeError('RC13B replicate Q-bank identity drift detected.')
    out['final_public_eval_acc']=float(final_eval['acc']);out['recipient_public_eval_acc']=float(recipient_eval['acc'])
    for inst,cal in cals.items():
        out[f'{inst}_one_step_rmse']=float(cal['mean_one_step_rmse'])
        out[f'{inst}_calibration_asymmetry']=float(cal['sign_asymmetry'])
    meta={'replicate_id':int(rep),'cohort':cohort,'final_acc':float(final_eval['acc']),'recipient_acc':float(recipient_eval['acc']),
        'instrument_calibrations':cals,'instrument_axes_meta':axes_meta,'branches':branch_meta,
        'anisotropy_panel':anisotropy_panel,
        'donor_step':donor_step,'recipient_step':recipient_step,'task_hash':task['task_hash'],'train_spec':asdict(ts),
        'model_parameter_count':int(recipient['model_parameter_count']),'optimizer_defaults':recipient['optimizer_defaults'],
        'q_bank_sha256':q_bank['q_bank_sha256'],'q_candidate_index':int(q_bank.get('q_candidate_index',0)),'q_noise_pairwise_cosines':q_bank['noise_pairwise_cosines'],
        'q_noise_max_abs_cosine':float(q_bank['noise_max_abs_cosine'])}
    del axes,anis_axes,q_bank,bank,sch,dt,task,donor,recipient,parts
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return out,meta



## Discovery-only matched-view decomposition

Stage 06 explicitly separates two objects.

**Between-task phenotype (H6-B):** one untreated baseline state per task is used. The discovery-only rCCA model asks whether matched Q_A/Q_B measurements recover a common low-dimensional susceptibility coordinate that predicts the physically disjoint Q_T future response.

**Within-task state (H6-W):** before any scaler, rCCA fit, decoder, or dimension selection, every manipulated branch is differenced against the same task's untouched baseline. Baseline rows are then removed. Thus task-level response offsets cannot themselves create the within-task state.

Regularized CCA is chosen because RC12D showed that the predictively useful direction need not be the maximum-variance PCA direction in every physical query view. Dimension error is estimated only on discovery with outer task-grouped folds; within each outer-training split, rCCA regularization is selected by an inner task-grouped CV. Final full-discovery regularization is then chosen by grouped CV after the dimension error curve is defined. Confirmation transforms only.

A single common decoder is fitted to stacked Q_A/Q_B canonical scores. This makes the query-invariance requirement operational: both independently measured views must support the same frozen future-response mapping.


In [ ]:
# 4. Discovery-only matched-view estimators and confirmatory H6 statistics

def _mse(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    if y.shape!=p.shape:raise RuntimeError(f'MSE shape mismatch: {y.shape} != {p.shape}')
    if not np.isfinite(y).all() or not np.isfinite(p).all():raise RuntimeError('Non-finite value reached confirmatory MSE.')
    v=float(np.mean((y-p)**2))
    if not np.isfinite(v):raise RuntimeError('Non-finite confirmatory MSE result.')
    return v

def _relative_improvement(base,new):
    base=float(base);new=float(new)
    if not np.isfinite(base) or not np.isfinite(new):raise RuntimeError('Non-finite value reached relative-improvement statistic.')
    if base<0 or new<0:raise RuntimeError('MSE-derived improvement received a negative error.')
    v=float((base-new)/max(base,1e-30))
    if not np.isfinite(v):raise RuntimeError('Non-finite relative-improvement result.')
    return v

def _bootstrap_replicates(values_by_rep,n_boot=None,seed=0,stat='mean'):
    n_boot=int((globals().get('_QA_BOOTSTRAP_N',CORE.bootstrap_n)) if n_boot is None else n_boot)
    reps=sorted(values_by_rep);vals=np.array([values_by_rep[r] for r in reps],float)
    if len(vals)==0:return {'n':0,'mean':float('nan'),'median':float('nan'),'ci95':[float('nan'),float('nan')]}
    if not np.isfinite(vals).all():
        bad=[str(reps[i]) for i in np.where(~np.isfinite(vals))[0][:8]]
        raise RuntimeError(f'Non-finite task statistic reached bootstrap; refusing silent task deletion. bad_replicates={bad}')
    fn=np.mean if stat=='mean' else np.median
    rng=np.random.default_rng(seed);boots=np.empty(n_boot,float)
    for i in range(n_boot):boots[i]=float(fn(rng.choice(vals,size=len(vals),replace=True)))
    return {'n':len(vals),'mean':float(np.mean(vals)),'median':float(np.median(vals)),
            'statistic':stat,'estimate':float(fn(vals)),
            'ci95':[float(np.quantile(boots,.025)),float(np.quantile(boots,.975))]}


def _bootstrap_corr(x,y,n_boot=None,seed=0):
    n_boot=int((globals().get('_QA_BOOTSTRAP_N',CORE.bootstrap_n)) if n_boot is None else n_boot)
    x=np.asarray(x,float);y=np.asarray(y,float)
    if x.shape!=y.shape:raise RuntimeError(f'Correlation shape mismatch: {x.shape} != {y.shape}')
    if not np.isfinite(x).all() or not np.isfinite(y).all():raise RuntimeError('Non-finite task value reached correlation bootstrap; refusing silent task deletion.')
    if len(x)<3 or np.std(x)<=1e-12 or np.std(y)<=1e-12:
        return {'n':len(x),'corr':float('nan'),'ci95':[float('nan'),float('nan')]}
    obs=float(np.corrcoef(x,y)[0,1]);rng=np.random.default_rng(seed);boot=[]
    for _ in range(n_boot):
        ii=rng.integers(0,len(x),size=len(x));xx=x[ii];yy=y[ii]
        if np.std(xx)>1e-12 and np.std(yy)>1e-12:boot.append(float(np.corrcoef(xx,yy)[0,1]))
    if not boot:return {'n':len(x),'corr':obs,'ci95':[float('nan'),float('nan')]}
    return {'n':len(x),'corr':obs,'ci95':[float(np.quantile(boot,.025)),float(np.quantile(boot,.975))]}


def _exact_or_mc_signflip(vals,alternative='greater',seed=0):
    x=np.asarray(vals,float);n=len(x)
    if n==0:return float('nan')
    if not np.isfinite(x).all():raise RuntimeError('Non-finite task statistic reached sign-flip test; refusing silent task deletion.')
    obs=float(np.mean(x))
    if n<=20:
        total=1<<n;count=0;batch=65536;bitpos=np.arange(n,dtype=np.uint64)
        for start in range(0,total,batch):
            ids=np.arange(start,min(total,start+batch),dtype=np.uint64)[:,None]
            signs=(((ids>>bitpos)&1).astype(np.float64)*2.0-1.0);stats=(signs@x)/n
            if alternative=='greater':count+=int(np.sum(stats>=obs))
            elif alternative=='less':count+=int(np.sum(stats<=obs))
            else:count+=int(np.sum(np.abs(stats)>=abs(obs)))
        return float(count/total)
    rng=np.random.default_rng(seed);total=int(CORE.signflip_mc_draws);count=0;batch=10000;done=0
    while done<total:
        b=min(batch,total-done)
        stats=np.mean(x[None,:]*rng.choice([-1,1],size=(b,n)),axis=1)
        if alternative=='greater':count+=int(np.sum(stats>=obs))
        elif alternative=='less':count+=int(np.sum(stats<=obs))
        else:count+=int(np.sum(np.abs(stats)>=abs(obs)))
        done+=b
    # Plus-one correction prevents an impossible p=0 from a finite Monte-Carlo null.
    return float((count+1)/(total+1))

def _select_ridge_alpha(X,Y,groups):
    X=np.asarray(X,float);Y=np.asarray(Y,float);groups=np.asarray(groups)
    uniq=np.unique(groups);splits=min(5,len(uniq));best=None
    for a in CORE.ridge_alphas:
        losses=[]
        if splits>=2:
            for tr,va in GroupKFold(splits).split(X,groups=groups):
                mdl=Ridge(alpha=a).fit(X[tr],Y[tr]);losses.append(_mse(Y[va],mdl.predict(X[va])))
        else:losses=[_mse(Y,Ridge(alpha=a).fit(X,Y).predict(X))]
        v=float(np.mean(losses));best=(v,a) if best is None or v<best[0] else best
    return float(best[1])

def _replicate_mse(meta,Y,pred,mask=None):
    out={};m=np.ones(len(meta),dtype=bool) if mask is None else np.asarray(mask,bool);rr=meta.replicate_id.to_numpy(int)
    for r in sorted(np.unique(rr)):
        idx=np.where((rr==r)&m)[0]
        if len(idx):out[int(r)]=_mse(Y[idx],pred[idx])
    return out

def _relative_improvement_boot(base_by_rep,new_by_rep,seed):
    reps=sorted(set(base_by_rep)&set(new_by_rep))
    vals={r:_relative_improvement(base_by_rep[r],new_by_rep[r]) for r in reps}
    return _bootstrap_replicates(vals,seed=seed),_exact_or_mc_signflip(list(vals.values()),'greater',seed+1),vals

def _invsqrt_psd(C,eps=1e-12):
    C=np.asarray(C,dtype=np.float64)
    if C.ndim!=2 or C.shape[0]!=C.shape[1] or not np.isfinite(C).all():
        raise ScientificGateStop('rCCA covariance matrix is non-finite or malformed.')
    # Numerical symmetrization is semantics-preserving for a covariance matrix.
    C=0.5*(C+C.T)
    try:
        w,V=np.linalg.eigh(C)
    except np.linalg.LinAlgError as e:
        raise ScientificGateStop(f'rCCA covariance eigendecomposition failed: {e}') from e
    if not np.isfinite(w).all() or not np.isfinite(V).all():
        raise ScientificGateStop('rCCA covariance eigendecomposition produced non-finite values.')
    scale=max(1.0,float(np.max(np.abs(w))) if w.size else 1.0)
    floor=max(float(eps)*scale,np.finfo(np.float64).eps*scale*32.0)
    w=np.maximum(w,floor)
    W=(V*(1.0/np.sqrt(w))[None,:])@V.T
    if not np.isfinite(W).all():
        raise ScientificGateStop('rCCA inverse-square-root whitening produced non-finite values.')
    return W

def _stable_svd(M):
    M=np.asarray(M,dtype=np.float64)
    if M.ndim!=2 or not np.isfinite(M).all():
        raise ScientificGateStop('rCCA whitened cross-covariance is non-finite or malformed.')
    mag=float(np.max(np.abs(M))) if M.size else 0.0
    if not np.isfinite(mag):
        raise ScientificGateStop('rCCA whitened cross-covariance scale is non-finite.')
    if mag==0.0:
        # A zero cross-view matrix has no identifiable canonical direction.
        raise ScientificGateStop('rCCA whitened cross-covariance is identically zero.')
    Mn=M/mag  # positive scalar normalization leaves singular vectors unchanged
    try:
        U,s,Vt=np.linalg.svd(Mn,full_matrices=False)
    except np.linalg.LinAlgError:
        try:
            from scipy.linalg import svd as _scipy_svd
            U,s,Vt=_scipy_svd(Mn,full_matrices=False,lapack_driver='gesvd',check_finite=True)
        except Exception as e:
            raise ScientificGateStop(f'rCCA SVD failed under both NumPy and LAPACK gesvd: {e}') from e
    if not (np.isfinite(U).all() and np.isfinite(s).all() and np.isfinite(Vt).all()):
        raise ScientificGateStop('rCCA SVD produced non-finite values.')
    return U,s*mag,Vt

def _fit_rcca(XA,XB,reg,d):
    XA=np.asarray(XA,dtype=np.float64);XB=np.asarray(XB,dtype=np.float64)
    if XA.ndim!=2 or XB.ndim!=2 or XA.shape[0]!=XB.shape[0] or XA.shape[0]<2:
        raise ScientificGateStop('rCCA received malformed or insufficient paired samples.')
    if not np.isfinite(XA).all() or not np.isfinite(XB).all():
        raise ScientificGateStop('rCCA received non-finite paired samples.')
    if not np.isfinite(reg) or float(reg)<=0:
        raise ScientificGateStop('rCCA regularization must be finite and positive.')
    n=max(1,len(XA)-1)
    Caa=(XA.T@XA)/n+float(reg)*np.eye(XA.shape[1],dtype=np.float64)
    Cbb=(XB.T@XB)/n+float(reg)*np.eye(XB.shape[1],dtype=np.float64)
    Cab=(XA.T@XB)/n
    Wa=_invsqrt_psd(Caa);Wb=_invsqrt_psd(Cbb)
    U,s,Vt=_stable_svd(Wa@Cab@Wb)
    d=int(min(int(d),U.shape[1],Vt.shape[0]))
    if d<1:raise ScientificGateStop('rCCA produced zero usable canonical dimensions.')
    A=Wa@U[:,:d];B=Wb@Vt.T[:,:d]
    if not np.isfinite(A).all() or not np.isfinite(B).all():
        raise ScientificGateStop('rCCA canonical loadings are non-finite.')
    za=XA@A;zb=XB@B
    # SVD makes pair orientation arbitrary; force positive training correlations componentwise.
    for j in range(d):
        c=np.corrcoef(za[:,j],zb[:,j])[0,1] if np.std(za[:,j])>1e-12 and np.std(zb[:,j])>1e-12 else 1.0
        if not np.isfinite(c):raise ScientificGateStop('rCCA canonical correlation is non-finite.')
        if c<0:B[:,j]*=-1;zb[:,j]*=-1
    return {'A':A,'B':B,'singular_values':s[:d]}

def _common_decoder(ZA,ZB,Y,groups):
    Z=np.vstack([ZA,ZB]);YY=np.vstack([Y,Y]);gg=np.concatenate([groups,groups])
    a=_select_ridge_alpha(Z,YY,gg);mdl=Ridge(alpha=a).fit(Z,YY)
    return mdl,float(a)

def _fit_one_view_decoder(X,Y,groups):
    a=_select_ridge_alpha(X,Y,groups);return Ridge(alpha=a).fit(X,Y),float(a)


def _rcca_fold_loss(Atr,Btr,Ytr,Ava,Bva,Yva,d,reg):
    sa=StandardScaler().fit(Atr);sb=StandardScaler().fit(Btr);sy=StandardScaler().fit(Ytr)
    At=sa.transform(Atr);Bt=sb.transform(Btr);Av=sa.transform(Ava);Bv=sb.transform(Bva)
    Yt=sy.transform(Ytr);Yv=sy.transform(Yva)
    rr=_fit_rcca(At,Bt,reg,d)
    za=At@rr['A'];zb=Bt@rr['B'];zav=Av@rr['A'];zbv=Bv@rr['B']
    mdl=Ridge(alpha=CORE.dimension_cv_ridge_alpha).fit(np.vstack([za,zb]),np.vstack([Yt,Yt]))
    return 0.5*(_mse(Yv,mdl.predict(zav))+_mse(Yv,mdl.predict(zbv)))


def _full_group_cv_reg(Araw,Braw,Traw,groups,d):
    groups=np.asarray(groups,int);uniq=np.unique(groups);splits=min(int(CORE.rcca_final_reg_folds),len(uniq))
    scores=[]
    for reg in CORE.rcca_regs:
        losses=[]
        folds=GroupKFold(splits).split(Araw,groups=groups) if splits>=2 else [(np.arange(len(Araw)),np.arange(len(Araw)))]
        for tr,va in folds:
            losses.append(_rcca_fold_loss(Araw[tr],Braw[tr],Traw[tr],Araw[va],Braw[va],Traw[va],d,float(reg)))
        scores.append((float(np.mean(losses)),float(reg)))
    return min(scores,key=lambda x:x[0])



def _rcca_cv_curve(Araw,Braw,Traw,groups):
    """Nested task-grouped error curve.

    Outer task folds estimate each dimension's future-Q_T error. Within every outer-training
    split, rCCA regularization is selected by an inner task-grouped CV. A separate full-discovery
    grouped CV then chooses the regularization used to fit the frozen model for that dimension.
    """
    Araw=np.asarray(Araw,float);Braw=np.asarray(Braw,float);Traw=np.asarray(Traw,float);groups=np.asarray(groups,int)
    uniq=np.unique(groups);outer_splits=min(int(CORE.rcca_outer_folds),len(uniq));rows=[]
    if outer_splits<2:raise ScientificGateStop('Nested rCCA selection requires at least two task groups.')
    outer=list(GroupKFold(outer_splits).split(Araw,groups=groups))
    for d in CORE.candidate_dims:
        if int(d)>min(Araw.shape[1],Braw.shape[1],max(1,len(Araw)-1)):continue
        outer_losses=[];outer_regs=[]
        for tr,va in outer:
            gtr=groups[tr];inner_unique=np.unique(gtr);inner_splits=min(int(CORE.rcca_inner_folds),len(inner_unique))
            if inner_splits<2:raise ScientificGateStop('Nested rCCA inner selection has too few task groups.')
            reg_scores=[]
            for reg in CORE.rcca_regs:
                iloss=[]
                for itr,iva in GroupKFold(inner_splits).split(Araw[tr],groups=gtr):
                    tr2=tr[itr];va2=tr[iva]
                    iloss.append(_rcca_fold_loss(Araw[tr2],Braw[tr2],Traw[tr2],Araw[va2],Braw[va2],Traw[va2],d,float(reg)))
                reg_scores.append((float(np.mean(iloss)),float(reg)))
            selected_reg=min(reg_scores,key=lambda x:x[0])[1]
            outer_regs.append(float(selected_reg))
            outer_losses.append(_rcca_fold_loss(Araw[tr],Braw[tr],Traw[tr],Araw[va],Braw[va],Traw[va],d,float(selected_reg)))
        full_cv_mse,final_reg=_full_group_cv_reg(Araw,Braw,Traw,groups,d)
        rows.append({'dimension':int(d),'selected_reg':float(final_reg),
                     'cv_standardized_target_mse':float(np.mean(outer_losses)),
                     'nested_outer_fold_mse_json':json.dumps([float(x) for x in outer_losses],separators=(',',':')),
                     'nested_outer_selected_regs_json':json.dumps([float(x) for x in outer_regs],separators=(',',':')),
                     'full_discovery_reg_cv_mse':float(full_cv_mse),
                     'selection_method':'nested_task_grouped_cv'})
    curve=pd.DataFrame(rows).sort_values('dimension').reset_index(drop=True)
    if curve.empty:raise ScientificGateStop('No valid rCCA dimension candidate.')
    best=float(curve.cv_standardized_target_mse.min())
    curve['within_tolerance']=curve.cv_standardized_target_mse<=best*(1+CORE.dimension_error_tolerance)
    return curve

def _jackknife_d1_agreement(A,B,T,groups):
    reps=sorted(np.unique(groups));ok=0;total=0
    # Smoke keeps the same logic but its tiny cohort makes the result machinery-only.
    for r in reps:
        keep=np.asarray(groups)!=r
        if len(np.unique(np.asarray(groups)[keep]))<3:continue
        curve=_rcca_cv_curve(A[keep],B[keep],T[keep],np.asarray(groups)[keep])
        row=curve[curve.dimension==1]
        if len(row):ok+=int(bool(row.within_tolerance.iloc[0]));total+=1
    return float(ok/max(total,1))

def _fit_shared_bundle(Araw,Braw,Traw,Hraw,meta,label):
    Araw=np.asarray(Araw,float);Braw=np.asarray(Braw,float);Traw=np.asarray(Traw,float);Hraw=np.asarray(Hraw,float)
    groups=meta.replicate_id.to_numpy(int)
    curve=_rcca_cv_curve(Araw,Braw,Traw,groups)
    if 1 not in set(curve.dimension):raise ScientificGateStop(f'{label}: d=1 unavailable.')
    d1row=curve[curve.dimension==1].iloc[0]
    higher=curve[curve.dimension>1]
    if higher.empty:raise ScientificGateStop(f'{label}: no higher-dimensional challenger.')
    challenger=int(higher.sort_values('cv_standardized_target_mse').iloc[0].dimension)
    jack=_jackknife_d1_agreement(Araw,Braw,Traw,groups)

    sa=StandardScaler().fit(Araw);sb=StandardScaler().fit(Braw);sy=StandardScaler().fit(Traw)
    sh=StandardScaler().fit(Hraw)
    A=sa.transform(Araw);B=sb.transform(Braw);Y=sy.transform(Traw);H=sh.transform(Hraw)

    dim_models={}
    for d in sorted({1,challenger}):
        row=curve[curve.dimension==d].iloc[0];rr=_fit_rcca(A,B,float(row.selected_reg),d)
        za=A@rr['A'];zb=B@rr['B'];dec,alpha=_common_decoder(za,zb,Y,groups)
        dim_models[int(d)]={'rcca':rr,'decoder':dec,'ridge_alpha':alpha,'reg':float(row.selected_reg)}

    # Strong full-view challengers are allowed separate decoders.
    fullA,aA=_fit_one_view_decoder(A,Y,groups);fullB,aB=_fit_one_view_decoder(B,Y,groups)
    rich,ar=_fit_one_view_decoder(H,Y,groups)

    rr1=dim_models[1]['rcca'];za=A@rr1['A'];zb=B@rr1['B']
    ZH=np.vstack([np.concatenate([za,H],1),np.concatenate([zb,H],1)])
    YY=np.vstack([Y,Y]);gg=np.concatenate([groups,groups]);az=_select_ridge_alpha(ZH,YY,gg);zrich=Ridge(alpha=az).fit(ZH,YY)

    bundle={'label':label,'A_scaler':sa,'B_scaler':sb,'Y_scaler':sy,'H_scaler':sh,
            'dimension_curve':curve,'challenger_dim':challenger,'jackknife_d1_agreement':float(jack),
            'dimension_models':dim_models,'fullA_decoder':fullA,'fullB_decoder':fullB,'rich_decoder':rich,'zrich_decoder':zrich,
            'fullA_alpha':aA,'fullB_alpha':aB,'rich_alpha':ar,'zrich_alpha':float(az)}
    return bundle


def _transform_shared_bundle(Araw,Braw,Traw,Hraw,meta,bundle):
    A=bundle['A_scaler'].transform(Araw);B=bundle['B_scaler'].transform(Braw)
    Y=bundle['Y_scaler'].transform(Traw);H=bundle['H_scaler'].transform(Hraw)
    # Physical zero effect is generally NOT numerical zero after StandardScaler centering.
    Y_zero=bundle['Y_scaler'].transform(np.zeros_like(np.asarray(Traw,float)))
    out={'Y':Y,'Y_zero':Y_zero,'A':A,'B':B,'H':H}
    for d,dm in bundle['dimension_models'].items():
        za=A@dm['rcca']['A'];zb=B@dm['rcca']['B'];dec=dm['decoder']
        out[f'zA_{d}']=za;out[f'zB_{d}']=zb;out[f'predA_{d}']=dec.predict(za);out[f'predB_{d}']=dec.predict(zb)
    zA=out['zA_1'];zB=out['zB_1']
    out['pred_fullA']=bundle['fullA_decoder'].predict(A);out['pred_fullB']=bundle['fullB_decoder'].predict(B)
    out['pred_rich']=bundle['rich_decoder'].predict(H)
    out['pred_zrichA']=bundle['zrich_decoder'].predict(np.concatenate([zA,H],1))
    out['pred_zrichB']=bundle['zrich_decoder'].predict(np.concatenate([zB,H],1))
    for k,v in out.items():
        if isinstance(v,np.ndarray) and not np.isfinite(v).all():
            raise RuntimeError(f'Non-finite confirmation transform output: {k}')
    return out

def _matched_query_columns(M):
    return s6_query_partition(M.columns)

def _branch_extra_features(meta):
    fam=meta.instrument_family.astype(str)
    levels=('history','random_a','random_b')
    one=np.stack([(fam==k).to_numpy(float) for k in levels],axis=1)
    d=meta.surgery_dose.to_numpy(float)[:,None];a=meta.surgery_alpha.to_numpy(float)[:,None]
    return np.concatenate([d,np.abs(d),d*d,a,d*a,one],axis=1)

def prepare_stage06_views(df,mode):
    M=s6_response_matrix(df);O=s6_state_observables(df).loc[M.index]
    qa,qb,qt=_matched_query_columns(M)
    meta=M.index.to_frame(index=False).reset_index(drop=True)
    A=M[qa].to_numpy(float);B=M[qb].to_numpy(float);T=M[qt].to_numpy(float);H=O.to_numpy(float)
    if not (np.isfinite(A).all() and np.isfinite(B).all() and np.isfinite(T).all() and np.isfinite(H).all()):
        raise RuntimeError('Non-finite Stage-06 analysis view.')
    if mode=='between':
        keep=meta.instrument_family.astype(str).eq('baseline').to_numpy()
        if int(keep.sum())!=meta.replicate_id.nunique():raise RuntimeError('Between-task view requires exactly one baseline branch per task.')
        return A[keep],B[keep],T[keep],H[keep],meta.loc[keep].reset_index(drop=True)
    if mode!='within':raise ValueError(mode)

    # Exact untreated-baseline subtraction occurs before any fit/scaler.
    Ac=[];Bc=[];Tc=[];Hc=[];rows=[]
    for r in sorted(meta.replicate_id.unique()):
        ids=np.where(meta.replicate_id.to_numpy(int)==int(r))[0]
        bi=[i for i in ids if str(meta.instrument_family.iloc[i])=='baseline']
        if len(bi)!=1:raise RuntimeError(f'Task {r}: expected one baseline branch, found {len(bi)}.')
        b=bi[0]
        for i in ids:
            if i==b:continue
            Ac.append(A[i]-A[b]);Bc.append(B[i]-B[b]);Tc.append(T[i]-T[b])
            # Center all observational/microscopic numeric state, then append exact treatment metadata.
            Hc.append(np.concatenate([H[i]-H[b],_branch_extra_features(meta.iloc[[i]])[0]]))
            rows.append(meta.iloc[i].to_dict())
    mm=pd.DataFrame(rows).reset_index(drop=True)
    return np.asarray(Ac),np.asarray(Bc),np.asarray(Tc),np.asarray(Hc),mm

def fit_stage06_discovery(discovery_df):
    Ab,Bb,Tb,Hb,mb=prepare_stage06_views(discovery_df,'between')
    Aw,Bw,Tw,Hw,mw=prepare_stage06_views(discovery_df,'within')
    return {'between':_fit_shared_bundle(Ab,Bb,Tb,Hb,mb,'between'),
            'within':_fit_shared_bundle(Aw,Bw,Tw,Hw,mw,'within')}


def _dimension_noninferiority(meta,tr,bundle,seed):
    d1A=_replicate_mse(meta,tr['Y'],tr['predA_1']);d1B=_replicate_mse(meta,tr['Y'],tr['predB_1'])
    cd=int(bundle['challenger_dim'])
    chA=_replicate_mse(meta,tr['Y'],tr[f'predA_{cd}']);chB=_replicate_mse(meta,tr['Y'],tr[f'predB_{cd}'])
    vals={}
    for r in sorted(set(d1A)&set(d1B)&set(chA)&set(chB)):
        d1=0.5*(d1A[r]+d1B[r]);ch=0.5*(chA[r]+chB[r]);vals[r]=(d1-ch)/max(ch,1e-30)
    expected=int(meta.replicate_id.nunique());coverage=bool(len(vals)==expected)
    boot=_bootstrap_replicates(vals,seed=seed);boot['expected_n']=expected;boot['coverage_passed']=coverage
    passed=bool(coverage and bundle['jackknife_d1_agreement']>=CORE.min_dimension_jackknife_agreement and
                bool(bundle['dimension_curve'].loc[bundle['dimension_curve'].dimension==1,'within_tolerance'].iloc[0]) and
                boot['ci95'][1]<=CORE.confirmation_d1_noninferiority_margin)
    return passed,boot


def _between_identity(meta,tr):
    zA=tr['zA_1'][:,0];zB=tr['zB_1'][:,0]
    out=_bootstrap_corr(zA,zB,seed=stable_seed('S6_B_identity'))
    out['expected_n']=int(meta.replicate_id.nunique())
    out['all_tasks_valid']=bool(out['n']==out['expected_n'])
    return out


def _within_identity(meta,tr):
    zA=tr['zA_1'][:,0];zB=tr['zB_1'][:,0];vals={}
    rr=meta.replicate_id.to_numpy(int)
    expected=int(meta.replicate_id.nunique())
    for r in sorted(np.unique(rr)):
        ii=np.where(rr==r)[0]
        if len(ii)>=3 and np.std(zA[ii])>1e-12 and np.std(zB[ii])>1e-12:
            vals[int(r)]=float(np.corrcoef(zA[ii],zB[ii])[0,1])
    boot=_bootstrap_replicates(vals,seed=stable_seed('S6_W_identity'))
    boot['positive_fraction']=float(np.mean(np.array(list(vals.values()))>0)) if vals else float('nan')
    boot['rows']=vals;boot['expected_n']=expected;boot['all_tasks_valid']=bool(len(vals)==expected)
    return boot


def _two_view_prediction_gate(meta,Y,tr,base_key,min_improvement,seed_prefix,mask=None):
    m=np.ones(len(meta),dtype=bool) if mask is None else np.asarray(mask,bool)
    expected=set(meta.loc[m,'replicate_id'].astype(int).tolist())
    base=_replicate_mse(meta,Y,tr[base_key],m)
    out={};passed=True
    for view in ('A','B'):
        z=_replicate_mse(meta,Y,tr[f'pred{view}_1'],m)
        coverage=bool(set(base)==expected and set(z)==expected)
        boot,p,vals=_relative_improvement_boot(base,z,stable_seed(seed_prefix,view))
        vp=bool(coverage and boot['ci95'][0]>0 and boot['mean']>=min_improvement and p<0.05)
        out[view]={'improvement':boot,'signflip_p':p,'coverage_passed':coverage,'expected_n':len(expected),'passed':vp}
        passed=passed and vp
    return bool(passed),out



def _zero_prediction_gate(meta,Y,tr,min_improvement,seed_prefix,mask=None):
    if 'Y_zero' not in tr:raise RuntimeError('Physical-zero target transform missing.')
    m=np.ones(len(meta),dtype=bool) if mask is None else np.asarray(mask,bool)
    expected=set(meta.loc[m,'replicate_id'].astype(int).tolist())
    base=_replicate_mse(meta,Y,tr['Y_zero'],m)
    out={};passed=True
    for view in ('A','B'):
        z=_replicate_mse(meta,Y,tr[f'pred{view}_1'],m)
        coverage=bool(set(base)==expected and set(z)==expected)
        boot,p,_=_relative_improvement_boot(base,z,stable_seed(seed_prefix,view))
        vp=bool(coverage and boot['ci95'][0]>0 and boot['mean']>=min_improvement and p<0.05)
        out[view]={'improvement':boot,'signflip_p':p,'coverage_passed':coverage,'expected_n':len(expected),'passed':vp};passed=passed and vp
    return bool(passed),out


def _sufficiency_gate(meta,Y,tr,bundle,seed_prefix,mask=None,contrast=False):
    m=np.ones(len(meta),dtype=bool) if mask is None else np.asarray(mask,bool)
    expected=set(meta.loc[m,'replicate_id'].astype(int).tolist())
    out={};passed=True
    for view in ('A','B'):
        mz=_replicate_mse(meta,Y,tr[f'pred{view}_1'],m)
        mf=_replicate_mse(meta,Y,tr[f'pred_full{view}'],m)
        mzr=_replicate_mse(meta,Y,tr[f'pred_zrich{view}'],m)
        coverage=bool(set(mz)==expected and set(mf)==expected and set(mzr)==expected)
        ex={r:(mz[r]-mf[r])/max(mf[r],1e-30) for r in set(mz)&set(mf)}
        rg={r:_relative_improvement(mz[r],mzr[r]) for r in set(mz)&set(mzr)}
        eb=_bootstrap_replicates(ex,seed=stable_seed(seed_prefix,view,'full'))
        rb=_bootstrap_replicates(rg,seed=stable_seed(seed_prefix,view,'residual'))
        margin=CORE.contrast_full_context_noninferiority_margin if contrast else CORE.within_full_context_noninferiority_margin
        residual_max=CORE.contrast_max_residual_gain_given_z if contrast else CORE.within_max_residual_gain_given_z
        vp=bool(coverage and eb['ci95'][1]<=margin and rb['ci95'][1]<=residual_max)
        out[view]={'z_excess_vs_full':eb,'rich_gain_given_z':rb,'coverage_passed':coverage,'expected_n':len(expected),'passed':vp};passed=passed and vp
    return bool(passed),out


def _between_sufficiency_gate(meta,Y,tr,bundle):
    expected=set(meta.replicate_id.astype(int).tolist())
    out={};passed=True
    for view in ('A','B'):
        mz=_replicate_mse(meta,Y,tr[f'pred{view}_1']);mf=_replicate_mse(meta,Y,tr[f'pred_full{view}'])
        mzr=_replicate_mse(meta,Y,tr[f'pred_zrich{view}'])
        coverage=bool(set(mz)==expected and set(mf)==expected and set(mzr)==expected)
        ex={r:(mz[r]-mf[r])/max(mf[r],1e-30) for r in set(mz)&set(mf)}
        rg={r:_relative_improvement(mz[r],mzr[r]) for r in set(mz)&set(mzr)}
        eb=_bootstrap_replicates(ex,seed=stable_seed('S6_B_suff',view,'full'))
        rb=_bootstrap_replicates(rg,seed=stable_seed('S6_B_suff',view,'residual'))
        vp=bool(coverage and eb['ci95'][1]<=CORE.between_full_context_noninferiority_margin and rb['ci95'][1]<=CORE.between_max_residual_gain_given_z)
        out[view]={'z_excess_vs_full':eb,'rich_gain_given_z':rb,'coverage_passed':coverage,'expected_n':len(expected),'passed':vp};passed=passed and vp
    return bool(passed),out

def _contrast_table(meta,Y,tr):
    rows=[];rr=meta.replicate_id.to_numpy(int)
    for fam in ('history','random_a','random_b'):
        for r in sorted(np.unique(rr)):
            idx=np.where((rr==r)&(meta.instrument_family.astype(str).to_numpy()==fam))[0]
            ip=[i for i in idx if float(meta.surgery_dose.iloc[i])==1.0];im=[i for i in idx if float(meta.surgery_dose.iloc[i])==-1.0]
            if len(ip)!=1 or len(im)!=1:continue
            p=ip[0];m=im[0];yt=Y[p]-Y[m]
            row={'replicate_id':int(r),'instrument_family':fam,'zero_mse':_mse(yt,np.zeros_like(yt))}
            for view in ('A','B'):
                row[f'z_{view}_mse']=_mse(yt,tr[f'pred{view}_1'][p]-tr[f'pred{view}_1'][m])
                row[f'full_{view}_mse']=_mse(yt,tr[f'pred_full{view}'][p]-tr[f'pred_full{view}'][m])
                row[f'zrich_{view}_mse']=_mse(yt,tr[f'pred_zrich{view}'][p]-tr[f'pred_zrich{view}'][m])
            row['rich_mse']=_mse(yt,tr['pred_rich'][p]-tr['pred_rich'][m]);rows.append(row)
    return pd.DataFrame(rows)


def _family_contrast_gate(Cdf,fam):
    g=Cdf[Cdf.instrument_family==fam].copy();expected=int(globals().get('_QA_EXPECTED_CONFIRMATION_N',len(S6_CONFIRMATION)))
    coverage=bool(len(g)==expected and g.replicate_id.nunique()==expected)
    out={'n':int(len(g)),'expected_n':expected,'coverage_passed':coverage};predpass=coverage;suffpass=coverage
    for view in ('A','B'):
        gz={int(r.replicate_id):_relative_improvement(float(r.zero_mse),float(getattr(r,f'z_{view}_mse'))) for _,r in g.iterrows()}
        gr={int(r.replicate_id):_relative_improvement(float(r.rich_mse),float(getattr(r,f'z_{view}_mse'))) for _,r in g.iterrows()}
        bz=_bootstrap_replicates(gz,seed=stable_seed('S6_C',fam,view,'zero'));br=_bootstrap_replicates(gr,seed=stable_seed('S6_C',fam,view,'rich'))
        pz=_exact_or_mc_signflip(list(gz.values()),'greater',stable_seed('S6_C',fam,view,'zp'));pr=_exact_or_mc_signflip(list(gr.values()),'greater',stable_seed('S6_C',fam,view,'rp'))
        vp=bool(coverage and bz['ci95'][0]>0 and bz['mean']>=CORE.min_contrast_improvement_vs_zero and pz<0.05 and
                br['ci95'][0]>0 and br['mean']>=CORE.min_contrast_improvement_vs_rich and pr<0.05)
        ex={int(r.replicate_id):(float(getattr(r,f'z_{view}_mse'))-float(getattr(r,f'full_{view}_mse')))/max(float(getattr(r,f'full_{view}_mse')),1e-30) for _,r in g.iterrows()}
        rg={int(r.replicate_id):_relative_improvement(float(getattr(r,f'z_{view}_mse')),float(getattr(r,f'zrich_{view}_mse'))) for _,r in g.iterrows()}
        be=_bootstrap_replicates(ex,seed=stable_seed('S6_C',fam,view,'ex'));bg=_bootstrap_replicates(rg,seed=stable_seed('S6_C',fam,view,'gain'))
        sp=bool(coverage and be['ci95'][1]<=CORE.contrast_full_context_noninferiority_margin and bg['ci95'][1]<=CORE.contrast_max_residual_gain_given_z)
        out[view]={'gain_vs_zero':bz,'gain_vs_zero_p':pz,'gain_vs_rich':br,'gain_vs_rich_p':pr,'prediction_passed':vp,
                   'z_excess_vs_full':be,'rich_gain_given_z':bg,'sufficiency_passed':sp}
        predpass=predpass and vp;suffpass=suffpass and sp
    out['prediction_passed']=bool(predpass);out['sufficiency_passed']=bool(suffpass)
    return out


def analyze_anisotropy(anisotropy_df):
    rows=[];valid=True
    for _,r in anisotropy_df.iterrows():
        hist=float(r.history_relative_m1_displacement)
        random_vals=json.loads(r.random_relative_m1_displacements_json)
        random_valid=json.loads(r.random_valid_json);random_capped=json.loads(r.random_capped_json)
        random_cos=json.loads(r.random_hidden_min_secant_cosines_json)
        random_drift=json.loads(r.random_hidden_max_relative_secant_norm_drifts_json)
        random_err=json.loads(r.random_target_relative_errors_json)
        random_asym=json.loads(r.random_sign_asymmetries_json)
        history_valid=bool(
            np.isfinite(hist) and hist>0 and not bool(r.history_calibration_capped) and
            float(r.history_target_relative_error)<=CORE.surgery_rel_tol and
            float(r.history_sign_asymmetry)<=CORE.surgery_sign_asymmetry_max and
            float(r.history_hidden_min_secant_cosine)>=CORE.hidden_locality_min_secant_cosine and
            float(r.history_hidden_max_relative_secant_norm_drift)<=CORE.hidden_locality_max_relative_norm_drift
        )
        random_local_valid=bool(
            len(random_cos)==CORE.anisotropy_random_directions and
            all(x is not None and float(x)>=CORE.hidden_locality_min_secant_cosine for x in random_cos) and
            all(x is not None and float(x)<=CORE.hidden_locality_max_relative_norm_drift for x in random_drift) and
            all(x is not None and float(x)<=CORE.surgery_rel_tol for x in random_err) and
            all(x is not None and float(x)<=CORE.surgery_sign_asymmetry_max for x in random_asym)
        )
        if (len(random_vals)!=CORE.anisotropy_random_directions or not all(bool(x) for x in random_valid) or
            any(bool(x) for x in random_capped) or not history_valid or not random_local_valid):
            valid=False;continue
        rv=np.asarray(random_vals,float)
        if not np.isfinite(rv).all() or np.any(rv<=0):valid=False;continue
        rows.append({'replicate_id':int(r.replicate_id),
                     'median_ratio':float(np.median(rv)/hist),
                     'best_random_ratio':float(np.min(rv)/hist),
                     'history_relative_m1_displacement':hist,
                     'median_random_relative_m1_displacement':float(np.median(rv)),
                     'min_random_relative_m1_displacement':float(np.min(rv))})
    R=pd.DataFrame(rows)
    if len(R)==0:
        return False,{'valid_all_tasks':False,'n':0},R
    med={int(r.replicate_id):float(r.median_ratio) for _,r in R.iterrows()}
    best={int(r.replicate_id):float(r.best_random_ratio) for _,r in R.iterrows()}
    bm=_bootstrap_replicates(med,seed=stable_seed('S6_A_median'),stat='median')
    bb=_bootstrap_replicates(best,seed=stable_seed('S6_A_best'),stat='median')
    frac=float(np.mean(R.median_ratio.to_numpy(float)>=CORE.anisotropy_task_ratio_floor))
    expected=int(globals().get('_QA_EXPECTED_CONFIRMATION_N',len(S6_CONFIRMATION)))
    passed=bool(valid and len(R)==expected and bm['median']>=CORE.anisotropy_median_ratio_min and
                bm['ci95'][0]>=CORE.anisotropy_ratio_ci_lower_min and frac>=CORE.anisotropy_fraction_above_floor and
                bb['median']>=CORE.anisotropy_best_random_median_min and bb['ci95'][0]>=CORE.anisotropy_best_random_ci_lower_min)
    return passed,{'valid_all_tasks':bool(valid),'n':int(len(R)),'expected_n':expected,'median_ratio':bm,'fraction_above_task_floor':frac,
                   'best_random_ratio':bb,'thresholds':{
                       'median_ratio_min':CORE.anisotropy_median_ratio_min,'median_ratio_ci_lower_min':CORE.anisotropy_ratio_ci_lower_min,
                       'task_ratio_floor':CORE.anisotropy_task_ratio_floor,'fraction_above_floor':CORE.anisotropy_fraction_above_floor,
                       'best_random_median_min':CORE.anisotropy_best_random_median_min,'best_random_ci_lower_min':CORE.anisotropy_best_random_ci_lower_min,
                       'hidden_locality_min_secant_cosine':CORE.hidden_locality_min_secant_cosine,
                       'hidden_locality_max_relative_norm_drift':CORE.hidden_locality_max_relative_norm_drift}},R

def confirm_stage06(confirmation_df,bundles,anisotropy_df):
    # Foundation: function invariance / optimizer isolation / Q locality.
    branches=confirmation_df.sort_values('horizon').drop_duplicates(S6_STATE_COLS)
    Fpass=bool((branches.instantaneous_max_logit_diff.to_numpy(float)==0.0).all() and branches.function_invariance_passed.astype(bool).all())
    nonzero=branches.surgery_dose.to_numpy(float)!=0
    Ipass=bool(branches.optimizer_isolation_passed.astype(bool).all() and branches.exp_avg_sq_unchanged.astype(bool).all() and
               branches.optimizer_step_unchanged.astype(bool).all() and branches.other_optimizer_state_unchanged.astype(bool).all() and
               branches.param_groups_unchanged.astype(bool).all() and branches.exp_avg_changed.to_numpy(bool)[nonzero].all() and
               (~branches.exp_avg_changed.to_numpy(bool)[~nonzero]).all())
    signed=confirmation_df.kind.astype(str).isin(SIGNED_KINDS)
    qfrac=float(confirmation_df.q_calibration_passed.astype(bool).mean())
    qmin=float(confirmation_df.loc[signed,'min_secant_cosine'].astype(float).min())
    qdrift=float(confirmation_df.loc[signed,'max_relative_secant_norm_drift'].astype(float).max())
    qnoise=float(confirmation_df.q_noise_max_abs_cosine.astype(float).max())
    Qpass=bool(qfrac>=CORE.q_required_pass_fraction and qmin>=CORE.q_locality_min_secant_cosine and
               qdrift<=CORE.q_locality_max_relative_norm_drift and qnoise<=CORE.q_noise_max_abs_cosine)

    nz=branches.surgery_dose.to_numpy(float)!=0
    hmin=float(branches.loc[nz,'instrument_hidden_min_secant_cosine'].astype(float).min()) if np.any(nz) else 1.0
    hdrift=float(branches.loc[nz,'instrument_hidden_max_relative_secant_norm_drift'].astype(float).max()) if np.any(nz) else 0.0
    Hlocalpass=bool(hmin>=CORE.hidden_locality_min_secant_cosine and hdrift<=CORE.hidden_locality_max_relative_norm_drift)

    # H6-B.
    Ab,Bb,Tb,Hb,mb=prepare_stage06_views(confirmation_df,'between')
    tb=_transform_shared_bundle(Ab,Bb,Tb,Hb,mb,bundles['between'])
    NB,NBstat=_dimension_noninferiority(mb,tb,bundles['between'],stable_seed('S6_B_N'))
    GBstat=_between_identity(mb,tb);GB=bool(GBstat.get('all_tasks_valid',False) and GBstat['corr']>=CORE.between_min_query_corr and GBstat['ci95'][0]>=CORE.between_min_query_corr_ci_lower)
    PB,PBstat=_two_view_prediction_gate(mb,tb['Y'],tb,'pred_rich',CORE.min_between_predictive_improvement,'S6_B_P')
    SB,SBstat=_between_sufficiency_gate(mb,tb['Y'],tb,bundles['between'])
    H6B=bool(Qpass and NB and GB and PB and SB)

    # H6-W.
    Aw,Bw,Tw,Hw,mw=prepare_stage06_views(confirmation_df,'within')
    tw=_transform_shared_bundle(Aw,Bw,Tw,Hw,mw,bundles['within'])
    NW,NWstat=_dimension_noninferiority(mw,tw,bundles['within'],stable_seed('S6_W_N'))
    GWstat=_within_identity(mw,tw)
    GW=bool(GWstat.get('all_tasks_valid',False) and GWstat['mean']>=CORE.within_min_mean_query_corr and GWstat['ci95'][0]>=CORE.within_min_query_corr_ci_lower and
            GWstat['positive_fraction']>=CORE.within_min_positive_corr_fraction)
    Pzero,PzeroStat=_zero_prediction_gate(mw,tw['Y'],tw,CORE.min_within_predictive_improvement,'S6_W_P_zero')
    Prich,PrichStat=_two_view_prediction_gate(mw,tw['Y'],tw,'pred_rich',CORE.min_within_predictive_improvement,'S6_W_P_rich')
    PW=bool(Pzero and Prich)

    rbmask=mw.instrument_family.astype(str).eq('random_b').to_numpy()
    Ezero,EzeroStat=_zero_prediction_gate(mw,tw['Y'],tw,CORE.min_unseen_instrument_improvement,'S6_W_E_zero',rbmask)
    Erich,ErichStat=_two_view_prediction_gate(mw,tw['Y'],tw,'pred_rich',CORE.min_unseen_instrument_improvement,'S6_W_E_rich',rbmask)
    EW=bool(Ezero and Erich)

    Cdf=_contrast_table(mw,tw['Y'],tw)
    Cfam={f:_family_contrast_gate(Cdf,f) for f in ('history','random_a','random_b')}
    CW=bool(all(v['prediction_passed'] for v in Cfam.values()))
    SWabs,SWabsStat=_sufficiency_gate(mw,tw['Y'],tw,bundles['within'],'S6_W_S')
    SWcontrast=bool(all(v['sufficiency_passed'] for v in Cfam.values()))
    SW=bool(SWabs and SWcontrast)
    H6W=bool(Fpass and Ipass and Qpass and Hlocalpass and NW and GW and PW and EW and CW and SW)

    H6A,Astat,Adf=analyze_anisotropy(anisotropy_df)

    summary={
      'protocol_hash':PROTOCOL_HASH,
      'claims':{'H6_B_between_task_predictive_phenotype':H6B,
                'H6_W_within_task_causal_state':H6W,
                'H6_A_optimizer_anisotropy':H6A},
      'no_master_vote':True,
      'foundation':{'F_function_invariance':Fpass,'I_optimizer_field_isolation':Ipass,'Q_measurement_locality_and_independence':Qpass,
                    'hidden_intervention_locality':Hlocalpass,
                    'n_confirmation_branches':int(len(branches)),'max_instantaneous_logit_diff':float(branches.instantaneous_max_logit_diff.max()),
                    'q_pass_fraction':qfrac,'minimum_signed_secant_cosine':qmin,'maximum_signed_relative_secant_norm_drift':qdrift,
                    'maximum_QA_QT_QB_abs_direction_cosine':qnoise,
                    'minimum_hidden_secant_cosine':hmin,'maximum_hidden_relative_secant_norm_drift':hdrift},
      'H6_B':{'passed':H6B,'N_dimension':{'passed':NB,'confirmation_d1_excess_vs_challenger':NBstat,
                 'discovery_jackknife_d1_agreement':bundles['between']['jackknife_d1_agreement'],
                 'challenger_dim':int(bundles['between']['challenger_dim']),'dimension_curve':bundles['between']['dimension_curve'].to_dict(orient='records')},
              'G_matched_query_identity':{'passed':GB,**GBstat},'P_future_prediction':{'passed':PB,**PBstat},
              'S_scalar_sufficiency':{'passed':SB,**SBstat}},
      'H6_W':{'passed':H6W,'N_dimension':{'passed':NW,'confirmation_d1_excess_vs_challenger':NWstat,
                 'discovery_jackknife_d1_agreement':bundles['within']['jackknife_d1_agreement'],
                 'challenger_dim':int(bundles['within']['challenger_dim']),'dimension_curve':bundles['within']['dimension_curve'].to_dict(orient='records')},
              'G_within_task_matched_query_identity':{'passed':GW,**GWstat},
              'P_centered_future_prediction':{'passed':PW,'vs_zero':PzeroStat,'vs_rich':PrichStat},
              'E_confirmation_only_random_b':{'passed':EW,'vs_zero':EzeroStat,'vs_rich':ErichStat},
              'C_familywise_causal_contrasts':{'passed':CW,'families':Cfam},
              'S_scalar_sufficiency':{'passed':SW,'absolute':SWabsStat,'familywise_contrast_passed':SWcontrast}},
      'H6_A':{'passed':H6A,**Astat},
      'inference_unit':'replicate/task only',
      'claim_boundary':'Each claim is independent and restricted to the frozen controlled synthetic clean-learning ResNet18-GN/AdamW population and physical Stage-06 query/intervention family.'
    }

    # Saved coordinates are descriptive evidence; they never feed back into confirmation fitting.
    Bstate=mb.copy();Bstate['zA']=tb['zA_1'][:,0];Bstate['zB']=tb['zB_1'][:,0]
    Wstate=mw.copy();Wstate['zA']=tw['zA_1'][:,0];Wstate['zB']=tw['zB_1'][:,0]
    return summary,Bstate,Wstate,Cdf,Adf


## Durable execution, discovery lock, and confirmation firewall

The research campaign is deliberately sequential:

1. `06A_discovery_competency`
2. `06B_discovery_surgery_atlas`
3. `06C_discovery_decomposition_lock`
4. open cryptographically bound confirmation firewall
5. `06D_confirmation_competency`
6. `06E_confirmation_surgery_atlas`
7. `06F_confirmation_analysis`

Each expensive replicate is independently persisted with table hash, metadata hash, scientific-code fingerprint, protocol hash, and environment-compatibility hash. Stage-completion markers additionally bind the byte size and SHA-256 of every required stage output.

`random_b` and the H6-A anisotropy panel are confirmation-only. The discovery lock contains the raw discovery atlas hash, the frozen between/within rCCA models, regularization choices, dimension curves, one fixed higher-dimensional challenger per claim, and all transformation/decoder state needed by confirmation.


In [ ]:
# 5. Replicate-level durable Stage-06 campaign, discovery lock, confirmation firewall, and final export
STAGES=(
    '06A_discovery_competency','06B_discovery_surgery_atlas','06C_discovery_decomposition_lock',
    '06D_confirmation_competency','06E_confirmation_surgery_atlas','06F_confirmation_analysis'
)

def _current_scientific_code_hash():
    h=globals().get('SCIENTIFIC_CODE_MANIFEST',{}).get('campaign_code_hash')
    if not h:raise RuntimeError('Scientific-code fingerprint is not initialized.')
    return str(h)

def _identity_payload():
    # Fail closed if a mutable notebook global has drifted from the frozen protocol/environment.
    # Code-body drift is checked immediately before/after zero-hour QA and before paid launch.
    _assert_runtime_integrity(check_code=False)
    return {'protocol_hash':PROTOCOL_HASH,'implementation_revision':IMPLEMENTATION_REVISION,
            'implementation_revision_hash':IMPLEMENTATION_REVISION_HASH,'env_compat_hash':ENV_COMPAT_HASH,
            'scientific_code_hash':_current_scientific_code_hash()}

def _stage_marker(s):return BASE/'manifests'/f'{s}.json'
def _resolve_artifact_path(p):
    p=Path(p)
    if p.is_file():return p
    try:return table_path(p)
    except Exception:return None

def _artifact_record(p):
    q=_resolve_artifact_path(p)
    if q is None or not q.is_file():raise RuntimeError(f'Stage output missing at commit: {p}')
    return {'path':str(q.relative_to(BASE)),'bytes':int(q.stat().st_size),'sha256':file_sha256(q)}

def _done(s):
    p=_stage_marker(s)
    if not p.exists():return False
    try:
        d=json.loads(p.read_text());ident=_identity_payload()
        if not bool(d.get('complete',False)) or any(d.get(k)!=v for k,v in ident.items()):return False
        for rec in d.get('outputs',[]):
            q=BASE/rec['path']
            if not q.is_file() or int(rec.get('bytes',-1))!=q.stat().st_size or rec.get('sha256')!=file_sha256(q):return False
        return True
    except Exception:return False

def _mark(s,extra=None,output_paths=None):
    outs=[_artifact_record(p) for p in (output_paths or [])]
    atomic_json(_stage_marker(s),{'stage':s,'complete':True,**_identity_payload(),'time':time.time(),'outputs':outs,**(extra or {})})
    _durability_barrier()
    if not _done(s):raise RuntimeError(f'Stage marker durable output verification failed: {s}')

def _rep_stem(cohort,rep):return BASE/'blocks'/'stage06'/cohort/f'rep_{int(rep):04d}'
def _meta_path(cohort,rep):return BASE/'blocks'/'stage06'/cohort/f'rep_{int(rep):04d}_meta.json'
def _commit_path(cohort,rep):return BASE/'blocks'/'stage06'/cohort/f'rep_{int(rep):04d}_commit.json'

def _validate_rep_commit(cohort,rep):
    stem=_rep_stem(cohort,rep);mp=_meta_path(cohort,rep);cp=_commit_path(cohort,rep)
    if not mp.exists() or not cp.exists():return None
    try:
        tp=table_path(stem);c=json.loads(cp.read_text());m=json.loads(mp.read_text());ident=_identity_payload()
        if any(c.get(k)!=v for k,v in ident.items()) or any(m.get(k)!=v for k,v in ident.items()):return None
        if c.get('sha256_table')!=file_sha256(tp) or c.get('sha256_meta')!=file_sha256(mp):return None
        if int(c.get('bytes_table',-1))!=tp.stat().st_size:return None
        if int(m.get('replicate_id',-1))!=int(rep) or str(m.get('cohort'))!=str(cohort):return None
        df=load_table(stem)
        if 'replicate_id' not in df.columns or 'cohort' not in df.columns:return None
        if df.empty or df.replicate_id.nunique()!=1 or int(df.replicate_id.iloc[0])!=int(rep):return None
        if df.cohort.nunique()!=1 or str(df.cohort.iloc[0])!=str(cohort):return None
        return df,m
    except Exception:return None

def _load_or_run_rep(rep,cohort):
    cached=_validate_rep_commit(cohort,rep)
    if cached is not None:
        print('resume S6',cohort,rep);return cached
    t0=time.time();df,meta=run_s6_replicate(rep,cohort);ident=_identity_payload()
    for k,v in ident.items():df[k]=v;meta[k]=v
    meta['seconds']=time.time()-t0
    stem=_rep_stem(cohort,rep);mp=_meta_path(cohort,rep);cp=_commit_path(cohort,rep)
    path=save_table(df,stem);atomic_json(mp,meta)
    atomic_json(cp,{'sha256_table':file_sha256(path),'sha256_meta':file_sha256(mp),'bytes_table':path.stat().st_size,**ident,'time':time.time()})
    _durability_barrier();check=_validate_rep_commit(cohort,rep)
    if check is None:raise RuntimeError(f'Replicate durable readback failed: {cohort} {rep}')
    print('DURABLE S6',cohort,rep,'sec',round(meta['seconds'],1));return df,meta

def run_competency_preflight(cohort):
    ids=S6_DISCOVERY if cohort=='discovery' else S6_CONFIRMATION
    rows=[];ts=s6_train_spec();sc=2 if PROFILE=='research' else 1
    for rep in ids:
        task=bank=sch=dt=final_eval=rec=m=o=reval=None
        try:
            task=_make_s6_task(rep);bank,sch,dt,final_eval=build_checkpoint_bank(task,'resnet',sc,rep,ts)
            rec=bank[int(round(CORE.recipient_frac*ts.steps))]
            m,o=load_snapshot(task,'resnet',sc,rep,ts,rec);reval=evaluate_public(m,dt)
            rows.append({'cohort':cohort,'replicate_id':int(rep),'final_acc':float(final_eval['acc']),'recipient_acc':float(reval['acc']),
                'task_hash':str(task['task_hash']),'protocol_hash':PROTOCOL_HASH,'train_spec_hash':str(rec['train_spec_hash']),
                'model_parameter_count':int(rec['model_parameter_count'])})
            print('S6 competency',cohort,rep,'final',round(rows[-1]['final_acc'],4),'recipient',round(rows[-1]['recipient_acc'],4))
        finally:
            m=o=bank=sch=dt=final_eval=rec=reval=task=None;gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
    df=pd.DataFrame(rows)
    if len(df)!=len(ids) or df.replicate_id.nunique()!=len(ids):raise RuntimeError(f'{cohort} competency row/replicate count mismatch.')
    if df.task_hash.nunique()!=len(ids):raise RuntimeError(f'{cohort} competency contains duplicate task hashes; independence violated.')
    if cohort=='confirmation':
        dp=BASE/'artifacts'/'stage06_discovery_competency'
        if not _resolve_artifact_path(dp):raise RuntimeError('Confirmation competency requires frozen discovery competency evidence.')
        ddf=load_table(dp)
        overlap=set(ddf.task_hash.astype(str)) & set(df.task_hash.astype(str))
        if overlap:raise RuntimeError(f'Discovery/confirmation task-hash overlap detected in competency cohorts: {len(overlap)}')
    p=save_table(df,BASE/'artifacts'/f'stage06_{cohort}_competency')
    stat={'median_final_acc':float(df.final_acc.median()),'median_recipient_acc':float(df.recipient_acc.median()),
          'final_nontrivial_fraction':float((df.final_acc>=CORE.clean_task_nontrivial_floor).mean()),
          'recipient_nontrivial_fraction':float((df.recipient_acc>=CORE.recipient_task_nontrivial_floor).mean())}
    stat['passed']=bool(stat['median_final_acc']>=CORE.clean_median_acc_floor and stat['median_recipient_acc']>=CORE.recipient_acc_floor and
                        stat['final_nontrivial_fraction']>=CORE.min_nontrivial_task_fraction and
                        stat['recipient_nontrivial_fraction']>=CORE.min_nontrivial_task_fraction)
    summary={'protocol_hash':PROTOCOL_HASH,'cohort':cohort,
        'thresholds':{'median_final':CORE.clean_median_acc_floor,'median_recipient':CORE.recipient_acc_floor,
                      'final_nontrivial_floor':CORE.clean_task_nontrivial_floor,'recipient_nontrivial_floor':CORE.recipient_task_nontrivial_floor,
                      'minimum_nontrivial_fraction':CORE.min_nontrivial_task_fraction},
        'table_sha256':file_sha256(p),**stat}
    atomic_json(BASE/'artifacts'/f'stage06_{cohort}_competency_summary.json',summary)
    if not stat['passed']:raise ScientificGateStop(f'RC13B {cohort} clean-training competency failed before Stage-06 intervention outcomes.')
    return summary

def _anisotropy_table_from_metas(metas):
    rows=[]
    for m in metas:
        panel=m.get('anisotropy_panel')
        if panel is None:continue
        hist=panel.get('history',{})
        random=panel.get('random',[])
        vals=[];valid=[];capped=[];errors=[];axis_ids=[];loc_cos=[];loc_drift=[];target_err=[];asym=[];mean_rmse=[]
        for q in random:
            axis_ids.append(str(q.get('axis_id')))
            valid.append(bool(q.get('valid',False)))
            if q.get('valid',False):
                cal=q['calibration'];vals.append(float(cal['relative_m1_displacement_at_unit_dose']))
                capped.append(bool(cal.get('calibration_capped',False)));errors.append('')
                loc_cos.append(float(cal.get('hidden_min_secant_cosine',float('nan'))));loc_drift.append(float(cal.get('hidden_max_relative_secant_norm_drift',float('nan'))))
                target_err.append(float(cal.get('target_relative_error',float('nan'))));asym.append(float(cal.get('sign_asymmetry',float('nan'))));mean_rmse.append(float(cal.get('mean_one_step_rmse',float('nan'))))
            else:
                vals.append(None);capped.append(False);errors.append(str(q.get('error','')));loc_cos.append(None);loc_drift.append(None);target_err.append(None);asym.append(None);mean_rmse.append(None)
        rows.append({
            'replicate_id':int(m['replicate_id']),
            'history_relative_m1_displacement':float(hist.get('relative_m1_displacement_at_unit_dose',float('nan'))),
            'history_target_relative_error':float(hist.get('target_relative_error',float('nan'))),
            'history_sign_asymmetry':float(hist.get('sign_asymmetry',float('nan'))),
            'history_mean_one_step_rmse':float(hist.get('mean_one_step_rmse',float('nan'))),
            'history_calibration_capped':bool(hist.get('calibration_capped',False)),
            'history_hidden_min_secant_cosine':float(hist.get('hidden_min_secant_cosine',float('nan'))),
            'history_hidden_max_relative_secant_norm_drift':float(hist.get('hidden_max_relative_secant_norm_drift',float('nan'))),
            'random_axis_ids_json':json.dumps(axis_ids,separators=(',',':')),
            'random_relative_m1_displacements_json':json.dumps(vals,separators=(',',':')),
            'random_valid_json':json.dumps(valid,separators=(',',':')),
            'random_capped_json':json.dumps(capped,separators=(',',':')),
            'random_hidden_min_secant_cosines_json':json.dumps(loc_cos,separators=(',',':')),
            'random_hidden_max_relative_secant_norm_drifts_json':json.dumps(loc_drift,separators=(',',':')),
            'random_target_relative_errors_json':json.dumps(target_err,separators=(',',':')),
            'random_sign_asymmetries_json':json.dumps(asym,separators=(',',':')),
            'random_mean_one_step_rmse_json':json.dumps(mean_rmse,separators=(',',':')),
            'random_errors_json':json.dumps(errors,separators=(',',':')),
            'n_requested':int(panel.get('n_requested',len(random))),
            'protocol_hash':PROTOCOL_HASH,
        })
    return pd.DataFrame(rows)

def run_atlas(cohort):
    ids=S6_DISCOVERY if cohort=='discovery' else S6_CONFIRMATION
    parts=[];metas=[]
    hdoses=CORE.discovery_history_doses if cohort=='discovery' else CORE.confirmation_history_doses
    expected_branches=1+len(hdoses)+len(CORE.random_a_doses)+(len(CORE.random_b_doses) if cohort=='confirmation' else 0)
    q_per_target=(2*3)+2+2+1;rows_per_branch=len(CORE.target_positions)*q_per_target*len(CORE.horizons)
    for rep in ids:
        df,meta=_load_or_run_rep(rep,cohort);nbranches=df[S6_STATE_COLS].drop_duplicates().shape[0]
        if nbranches!=expected_branches:raise RuntimeError(f'{cohort} rep {rep}: branch count {nbranches} != frozen {expected_branches}')
        if len(df)!=expected_branches*rows_per_branch:raise RuntimeError(f'{cohort} rep {rep}: exact row count mismatch.')
        parts.append(df);metas.append(meta)
    out=pd.concat(parts,ignore_index=True);expected_rows=len(ids)*expected_branches*rows_per_branch
    if len(out)!=expected_rows:raise RuntimeError(f'{cohort} atlas row count mismatch: {len(out)} != {expected_rows}')
    ident=out[['replicate_id','task_hash','suite_hash','q_bank_sha256']].drop_duplicates().copy()
    if len(ident)!=len(ids) or ident.replicate_id.nunique()!=len(ids):raise RuntimeError(f'{cohort} atlas state identity is not one-to-one with frozen replicate IDs.')
    if ident.task_hash.nunique()!=len(ids) or ident.suite_hash.nunique()!=len(ids):raise RuntimeError(f'{cohort} atlas contains duplicate task/suite hashes; task independence violated.')
    if ident.q_bank_sha256.nunique()!=len(ids):raise RuntimeError(f'{cohort} atlas contains duplicate replicate-level Q-bank hashes.')
    if cohort=='confirmation':
        dd=load_table(BASE/'artifacts'/'stage06_discovery_atlas')
        if set(dd.task_hash.astype(str)) & set(out.task_hash.astype(str)):raise RuntimeError('Discovery/confirmation task-hash overlap detected in atlases.')
        if set(dd.suite_hash.astype(str)) & set(out.suite_hash.astype(str)):raise RuntimeError('Discovery/confirmation suite-hash overlap detected in atlases.')
        if set(dd.q_bank_sha256.astype(str)) & set(out.q_bank_sha256.astype(str)):raise RuntimeError('Discovery/confirmation Q-bank hash overlap detected in atlases.')
    p=save_table(out,BASE/'artifacts'/f'stage06_{cohort}_atlas')
    atomic_json(BASE/'artifacts'/f'stage06_{cohort}_replicate_meta.json',{'protocol_hash':PROTOCOL_HASH,'replicates':metas,
        'atlas_sha256':file_sha256(p),'expected_branches_per_replicate':expected_branches,'expected_rows':expected_rows})
    if cohort=='confirmation':
        adf=_anisotropy_table_from_metas(metas)
        if len(adf)!=len(ids):raise RuntimeError(f'Confirmation anisotropy table rows {len(adf)} != tasks {len(ids)}.')
        save_table(adf,BASE/'artifacts'/'stage06_confirmation_anisotropy_panel')
    return out

def freeze_discovery_lock(discovery):
    if 'random_b' in set(discovery.instrument_family.astype(str)):
        raise RuntimeError('random_b appeared in discovery before lock.')
    bundles=fit_stage06_discovery(discovery)
    lockp=BASE/'artifacts'/'stage06_discovery_model.pkl'
    with open(lockp,'wb') as f:pickle.dump(bundles,f,protocol=4)
    curves=[]
    for label,b in bundles.items():
        q=b['dimension_curve'].copy();q.insert(0,'analysis',label);curves.append(q)
    curve=pd.concat(curves,ignore_index=True);cp=save_table(curve,BASE/'artifacts'/'stage06_discovery_dimension_curves')
    atlas_path=table_path(BASE/'artifacts'/'stage06_discovery_atlas')
    summary={**_identity_payload(),
        'model_sha256':file_sha256(lockp),'discovery_atlas_sha256':file_sha256(atlas_path),'dimension_curves_sha256':file_sha256(cp),
        'between':{'dimension_fixed':1,'challenger_dim':int(bundles['between']['challenger_dim']),
                   'jackknife_d1_agreement':float(bundles['between']['jackknife_d1_agreement']),
                   'd1_within_tolerance':bool(bundles['between']['dimension_curve'].loc[bundles['between']['dimension_curve'].dimension==1,'within_tolerance'].iloc[0])},
        'within':{'dimension_fixed':1,'challenger_dim':int(bundles['within']['challenger_dim']),
                  'jackknife_d1_agreement':float(bundles['within']['jackknife_d1_agreement']),
                  'd1_within_tolerance':bool(bundles['within']['dimension_curve'].loc[bundles['within']['dimension_curve'].dimension==1,'within_tolerance'].iloc[0])},
        'matched_query_definition':{'Q_A':'noise_rep0_short','Q_B':'noise_rep2_short','Q_T':'noise_rep1_future'},
        'discovery_instruments':['baseline','history','random_a'],'confirmation_only_instrument':'random_b',
        'confirmation_opened':False,'locked_unix':time.time()}
    atomic_json(BASE/'artifacts'/'stage06_discovery_lock.json',summary);_durability_barrier()
    return summary

def load_discovery_lock():
    jp=BASE/'artifacts'/'stage06_discovery_lock.json';pp=BASE/'artifacts'/'stage06_discovery_model.pkl'
    if not jp.exists() or not pp.exists():raise RuntimeError('Stage-06 discovery lock missing; confirmation is forbidden.')
    j=json.loads(jp.read_text());ident=_identity_payload()
    if any(j.get(k)!=v for k,v in ident.items()) or j.get('model_sha256')!=file_sha256(pp):raise RuntimeError('Stage-06 discovery lock identity/hash mismatch.')
    atlas=table_path(BASE/'artifacts'/'stage06_discovery_atlas');curve=table_path(BASE/'artifacts'/'stage06_discovery_dimension_curves')
    if j.get('discovery_atlas_sha256')!=file_sha256(atlas) or j.get('dimension_curves_sha256')!=file_sha256(curve):
        raise RuntimeError('Stage-06 discovery lock upstream artifact hash mismatch.')
    with open(pp,'rb') as f:b=pickle.load(f)
    return b,j

def open_confirmation_firewall(lock):
    p=BASE/'manifests'/'confirmation_opened.json'
    payload={**_identity_payload(),'discovery_model_sha256':lock['model_sha256'],
        'discovery_atlas_sha256':lock['discovery_atlas_sha256'],
        'dimension_curves_sha256':lock['dimension_curves_sha256'],
        'random_b_was_absent_from_discovery':True,'opened_unix':time.time(),
        'policy':'From this point onward no scientific parameter, threshold, cohort, query bank, centering rule, estimator, branch schema, anisotropy rule, statistic, or claim language may change under this protocol hash.'}
    if not p.exists():atomic_json(p,payload);_durability_barrier()
    else:
        old=json.loads(p.read_text());ident=_identity_payload()
        checks=('discovery_model_sha256','discovery_atlas_sha256','dimension_curves_sha256')
        if any(old.get(k)!=v for k,v in ident.items()) or any(old.get(k)!=payload.get(k) for k in checks):
            raise RuntimeError('Stage-06 confirmation firewall identity/upstream-lock mismatch.')
        if not bool(old.get('random_b_was_absent_from_discovery',False)):
            raise RuntimeError('Stage-06 confirmation firewall lost the random_b discovery-exclusion assertion.')
    return json.loads(p.read_text())


def final_export(summary):
    atomic_json(BASE/'manifests'/'release_identity.json',{**_identity_payload(),'run_tag':RUN_TAG,'profile':PROFILE,'claims':summary.get('claims',{})})
    manifest_path=BASE/'manifests'/'artifact_manifest.json'
    # The manifest never lists itself. This makes repeated final_export() calls idempotent and
    # prevents a stale self-hash from entering a later result bundle.
    manifest=[]
    for p in sorted(BASE.rglob('*')):
        if p.is_file() and '.tmp' not in p.name and p!=manifest_path:
            manifest.append({'path':str(p.relative_to(BASE)),'bytes':int(p.stat().st_size),'sha256':file_sha256(p)})
    atomic_json(manifest_path,manifest)
    manifest_sha=file_sha256(manifest_path)
    files_to_zip=[p for p in sorted(BASE.rglob('*')) if p.is_file() and '.tmp' not in p.name]
    out=BASE.parent/f'{RUN_TAG}_STAGE06_RESULTS.zip'
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as zf:
        for p in files_to_zip:zf.write(p,p.relative_to(BASE.parent))
    # Verify not only CRC but exact member set and every manifest-bound payload inside the ZIP.
    expected_members={str(p.relative_to(BASE.parent)).replace(os.sep,'/') for p in files_to_zip}
    prefix=str(BASE.name).replace(os.sep,'/')+'/'
    with zipfile.ZipFile(out,'r') as zf:
        bad=zf.testzip()
        if bad is not None:raise RuntimeError(f'Final Stage-06 ZIP CRC verification failed at {bad}')
        got=set(zf.namelist())
        if got!=expected_members:
            raise RuntimeError(f'Final Stage-06 ZIP member-set mismatch: missing={sorted(expected_members-got)[:8]} extra={sorted(got-expected_members)[:8]}')
        for rec in manifest:
            arc=prefix+str(rec['path']).replace(os.sep,'/')
            info=zf.getinfo(arc)
            if int(info.file_size)!=int(rec['bytes']):raise RuntimeError(f'ZIP size mismatch for {arc}')
            h=hashlib.sha256()
            with zf.open(info,'r') as f:
                for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
            if h.hexdigest()!=str(rec['sha256']):raise RuntimeError(f'ZIP hash mismatch for {arc}')
        mi=zf.getinfo(prefix+'manifests/artifact_manifest.json')
        mh=hashlib.sha256()
        with zf.open(mi,'r') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''):mh.update(chunk)
        if mh.hexdigest()!=manifest_sha:raise RuntimeError('ZIP artifact-manifest hash mismatch.')
    _durability_barrier()
    receipt=BASE.parent/f'{RUN_TAG}_STAGE06_RESULTS.receipt.json'
    atomic_json(receipt,{**_identity_payload(),'zip_name':out.name,'zip_bytes':int(out.stat().st_size),
                         'zip_sha256':file_sha256(out),'artifact_manifest_sha256':manifest_sha,
                         'zip_crc_verified':True,'zip_member_set_verified':True,'zip_payload_hashes_verified':True,
                         'time':time.time()})
    _durability_barrier()
    print('FINAL ZIP',out,'sha256',file_sha256(out))
    print('FINAL RECEIPT',receipt)
    if _IN_COLAB:
        try:
            from google.colab import files
            files.download(str(out))
        except Exception as e:
            print('Browser download not completed; durable copy remains at',out,'reason:',e)
    return out

def run_stage06_campaign():
    if not _done('06A_discovery_competency'):
        a=run_competency_preflight('discovery');_mark('06A_discovery_competency',{'summary':a},[
            BASE/'artifacts'/'stage06_discovery_competency',BASE/'artifacts'/'stage06_discovery_competency_summary.json'])
    if not _done('06B_discovery_surgery_atlas'):
        d=run_atlas('discovery');_mark('06B_discovery_surgery_atlas',{'rows':len(d)},[
            BASE/'artifacts'/'stage06_discovery_atlas',BASE/'artifacts'/'stage06_discovery_replicate_meta.json'])
    else:d=load_table(BASE/'artifacts'/'stage06_discovery_atlas')
    if not _done('06C_discovery_decomposition_lock'):
        lock=freeze_discovery_lock(d)
        _mark('06C_discovery_decomposition_lock',{'model_sha256':lock['model_sha256'],'between':lock['between'],'within':lock['within']},[
            BASE/'artifacts'/'stage06_discovery_atlas',BASE/'artifacts'/'stage06_discovery_lock.json',
            BASE/'artifacts'/'stage06_discovery_model.pkl',BASE/'artifacts'/'stage06_discovery_dimension_curves'])
    bundles,lock=load_discovery_lock()
    open_confirmation_firewall(lock)

    if not _done('06D_confirmation_competency'):
        a=run_competency_preflight('confirmation')
        _mark('06D_confirmation_competency',{'summary':a,'discovery_model_sha256':lock['model_sha256']},[
            BASE/'manifests'/'confirmation_opened.json',BASE/'artifacts'/'stage06_confirmation_competency',
            BASE/'artifacts'/'stage06_confirmation_competency_summary.json'])
    if not _done('06E_confirmation_surgery_atlas'):
        c=run_atlas('confirmation')
        _mark('06E_confirmation_surgery_atlas',{'rows':len(c),'discovery_model_sha256':lock['model_sha256']},[
            BASE/'manifests'/'confirmation_opened.json',BASE/'artifacts'/'stage06_confirmation_atlas',
            BASE/'artifacts'/'stage06_confirmation_replicate_meta.json',BASE/'artifacts'/'stage06_confirmation_anisotropy_panel'])
    else:c=load_table(BASE/'artifacts'/'stage06_confirmation_atlas')
    if not _done('06F_confirmation_analysis'):
        adf=load_table(BASE/'artifacts'/'stage06_confirmation_anisotropy_panel')
        summary,bstate,wstate,contrast,atask=confirm_stage06(c,bundles,adf)
        atomic_json(BASE/'artifacts'/'stage06_confirmation_summary.json',summary)
        save_table(bstate,BASE/'artifacts'/'stage06_confirmation_between_coordinates')
        save_table(wstate,BASE/'artifacts'/'stage06_confirmation_within_coordinates')
        save_table(contrast,BASE/'artifacts'/'stage06_confirmation_causal_contrasts')
        save_table(atask,BASE/'artifacts'/'stage06_confirmation_anisotropy_task_stats')
        _mark('06F_confirmation_analysis',{'claims':summary['claims'],'discovery_model_sha256':lock['model_sha256']},[
            BASE/'manifests'/'confirmation_opened.json',BASE/'artifacts'/'stage06_discovery_model.pkl',
            BASE/'artifacts'/'stage06_discovery_dimension_curves',BASE/'artifacts'/'stage06_confirmation_atlas',
            BASE/'artifacts'/'stage06_confirmation_anisotropy_panel',BASE/'artifacts'/'stage06_confirmation_summary.json',
            BASE/'artifacts'/'stage06_confirmation_between_coordinates',BASE/'artifacts'/'stage06_confirmation_within_coordinates',
            BASE/'artifacts'/'stage06_confirmation_causal_contrasts',BASE/'artifacts'/'stage06_confirmation_anisotropy_task_stats'])
    else:summary=json.loads((BASE/'artifacts'/'stage06_confirmation_summary.json').read_text())
    final_export(summary);print(json.dumps(summary,indent=2));return summary

# ---------------- RC13C confirmation-completion amendment ----------------

def _locate_rc13b_source_zip():
    candidates=[]
    if Path('/content/drive/MyDrive').exists():
        candidates.append(Path('/content/drive/MyDrive/NEXUS_II_v1_9_1_STAGE06_RC13B/AUTO_CAMPAIGN')/RC13B_SOURCE_ZIP_NAME)
    candidates.append(Path('/content')/RC13B_SOURCE_ZIP_NAME)
    candidates.append(Path.cwd()/RC13B_SOURCE_ZIP_NAME)
    for p in candidates:
        if p.exists() and file_sha256(p)==RC13B_SOURCE_ZIP_SHA256:return p
    if Path('/content/drive/MyDrive').exists():
        for p in Path('/content/drive/MyDrive').rglob(RC13B_SOURCE_ZIP_NAME):
            if p.is_file() and file_sha256(p)==RC13B_SOURCE_ZIP_SHA256:return p
    raise FileNotFoundError(f'Canonical RC13B source ZIP not found. Required name={RC13B_SOURCE_ZIP_NAME} sha256={RC13B_SOURCE_ZIP_SHA256}')

def _zip_root_and_names(zf):
    names=[n for n in zf.namelist() if not n.endswith('/')]
    roots={n.split('/',1)[0] for n in names}
    if len(roots)!=1:raise RuntimeError(f'RC13B source ZIP must have one root; found {sorted(roots)}')
    return next(iter(roots)),names

def _zip_read_json(zf,arc):
    return json.loads(zf.read(arc).decode('utf-8'))

def validate_rc13b_source_zip(path):
    """Byte/hash validation only; confirmation Parquet payloads are never deserialized here."""
    if file_sha256(path)!=RC13B_SOURCE_ZIP_SHA256:raise RuntimeError('RC13B source ZIP SHA-256 mismatch.')
    with zipfile.ZipFile(path,'r') as zf:
        bad=zf.testzip()
        if bad is not None:raise RuntimeError(f'RC13B source ZIP CRC failure at {bad}')
        root,names=_zip_root_and_names(zf);prefix=root+'/'
        manifest=_zip_read_json(zf,prefix+'manifests/artifact_manifest.json')
        expected={prefix+str(r['path']).replace(os.sep,'/') for r in manifest}|{prefix+'manifests/artifact_manifest.json'}
        if set(names)!=expected:
            raise RuntimeError(f'RC13B source member-set mismatch: missing={sorted(expected-set(names))[:5]} extra={sorted(set(names)-expected)[:5]}')
        for rec in manifest:
            arc=prefix+str(rec['path']).replace(os.sep,'/')
            data=zf.read(arc)
            if len(data)!=int(rec['bytes']) or hashlib.sha256(data).hexdigest()!=str(rec['sha256']):
                raise RuntimeError(f'RC13B source manifest mismatch at {rec["path"]}')
        stop=_zip_read_json(zf,prefix+'artifacts/STAGE06_SCIENTIFIC_STOP.json')
        if stop.get('status')!='scientific_stop_before_complete_claim_evaluation' or bool(stop.get('claims_evaluated',True)):
            raise RuntimeError('RC13B parent is not the expected pre-claim scientific-stop bundle.')
        if stop.get('protocol_hash')!=RC13B_PROTOCOL_HASH or any(v is not None for v in stop.get('claims',{}).values()):
            raise RuntimeError('RC13B stop claim/protocol identity mismatch.')
        if '0.201394' not in str(stop.get('message','')):
            raise RuntimeError('RC13B stop reason does not match the canonical Q-geometry failure.')
        lock=_zip_read_json(zf,prefix+'artifacts/stage06_discovery_lock.json')
        checks={
            'model_sha256':RC13B_DISCOVERY_MODEL_SHA256,
            'discovery_atlas_sha256':RC13B_DISCOVERY_ATLAS_SHA256,
            'dimension_curves_sha256':RC13B_DISCOVERY_CURVES_SHA256,
            'protocol_hash':RC13B_PROTOCOL_HASH,
            'scientific_code_hash':RC13B_SCIENTIFIC_CODE_HASH,
        }
        for k,v in checks.items():
            if str(lock.get(k))!=str(v):raise RuntimeError(f'RC13B discovery lock mismatch: {k}')
        if hashlib.sha256(zf.read(prefix+'artifacts/stage06_discovery_lock.json')).hexdigest()!=RC13B_DISCOVERY_LOCK_SHA256:
            raise RuntimeError('RC13B discovery-lock file SHA mismatch.')
        if prefix+'artifacts/stage06_confirmation_summary.json' in names:
            raise RuntimeError('RC13B parent unexpectedly contains a final confirmation analysis.')
        reps=[]
        for rep in RC13B_SEALED_CONFIRMATION:
            stem=f'blocks/stage06/confirmation/rep_{rep}'
            for suf in ('.parquet','_commit.json','_meta.json'):
                if prefix+stem+suf not in names:raise RuntimeError(f'Missing sealed RC13B confirmation artifact: {stem+suf}')
            commit=_zip_read_json(zf,prefix+stem+'_commit.json')
            if commit.get('protocol_hash')!=RC13B_PROTOCOL_HASH or commit.get('scientific_code_hash')!=RC13B_SCIENTIFIC_CODE_HASH:
                raise RuntimeError(f'RC13B commit identity mismatch at {rep}')
            # PRE-LOCK FIREWALL: confirmation table and outcome-bearing metadata remain opaque bytes.
            table_bytes=zf.read(prefix+stem+'.parquet');meta_bytes=zf.read(prefix+stem+'_meta.json')
            if hashlib.sha256(table_bytes).hexdigest()!=commit.get('sha256_table') or len(table_bytes)!=int(commit.get('bytes_table',-1)):
                raise RuntimeError(f'RC13B sealed table commit mismatch at {rep}')
            if hashlib.sha256(meta_bytes).hexdigest()!=commit.get('sha256_meta'):
                raise RuntimeError(f'RC13B sealed meta commit mismatch at {rep}')
            reps.append(rep)
        found_reps=sorted({int(re.search(r'rep_(\d+)\.parquet$',n).group(1)) for n in names if '/blocks/stage06/confirmation/rep_' in n and n.endswith('.parquet')})
        if found_reps!=list(RC13B_SEALED_CONFIRMATION):
            raise RuntimeError(f'RC13B confirmation completeness mismatch: found {found_reps[:3]}...{found_reps[-3:]} expected 3100-3126 only.')
        parent_code=_zip_read_json(zf,prefix+'manifests/scientific_code_manifest.json')
        if parent_code.get('campaign_code_hash')!=RC13B_SCIENTIFIC_CODE_HASH or parent_code.get('protocol_hash')!=RC13B_PROTOCOL_HASH:
            raise RuntimeError('RC13B parent scientific-code manifest identity mismatch.')
        parent_core=_zip_read_json(zf,prefix+'manifests/core_config.json')
        parent_env_doc=_zip_read_json(zf,prefix+'manifests/environment_compatibility.json')
        parent_env=parent_env_doc.get('compat',parent_env_doc)
        parent_env_sha=hashlib.sha256(json.dumps(parent_env,sort_keys=True).encode()).hexdigest()
        if parent_env_doc.get('sha256',parent_env_sha)!=parent_env_sha:
            raise RuntimeError('RC13B parent environment-compatibility manifest hash mismatch.')
        if parent_env!=ENV_COMPAT or parent_env_sha!=ENV_COMPAT_HASH:
            raise RuntimeError('RC13C runtime environment compatibility differs from the canonical RC13B run; deterministic completion refused.')
    return {'path':str(path),'zip_sha256':RC13B_SOURCE_ZIP_SHA256,'root':root,
            'sealed_replicates':reps,'discovery_lock':lock,
            'parent_code_manifest':parent_code,'parent_core_config':parent_core,'parent_env_compat':parent_env,
            'claims_evaluated':False,'stop_message':stop['message'],'passed':True}

def _q_geometry_candidate(model,rep,candidate_index,recipient_step):
    pairs=target_panel_positions(model,CORE.target_positions)
    noise_cosines={};noise_ids={};global_max=0.0
    for target,tpos,tactual in pairs:
        vec=[];ids={}
        for qrep in (0,1,2):
            did=_rc13c_noise_direction_id('resnet',2,int(rep),int(recipient_step),float(tpos),int(qrep),int(candidate_index))
            d=unit_direction(model,target,'noise',None,None,did).detach()
            vec.append(d.reshape(-1).float());ids[str(qrep)]=int(did)
        pvals={}
        for i,j,label in ((0,1,'A_T'),(0,2,'A_B'),(1,2,'T_B')):
            c=float(torch.dot(vec[i],vec[j])/(torch.linalg.vector_norm(vec[i])*torch.linalg.vector_norm(vec[j])+1e-30))
            pvals[label]=c;global_max=max(global_max,abs(c))
        noise_cosines[str(target)]=pvals;noise_ids[str(target)]=ids
    return {'candidate':int(candidate_index),'noise_max_abs_cosine':float(global_max),
            'noise_pairwise_cosines':noise_cosines,'noise_direction_ids':noise_ids,
            'targets':[(str(a),float(b),float(c)) for a,b,c in pairs]}

def prequalify_and_lock_rc13c_q_plan(parent_audit):
    """Runs before any parent confirmation Parquet is deserialized."""
    global RC13C_Q_PLAN,RC13C_Q_PLAN_LOCKED
    lock_path=BASE/'manifests'/'rc13c_q_plan_lock.json'
    if lock_path.exists():
        plan=json.loads(lock_path.read_text())
        ident=_identity_payload()
        if plan.get('protocol_hash')!=PROTOCOL_HASH or plan.get('parent_zip_sha256')!=RC13B_SOURCE_ZIP_SHA256 or any(plan.get(k)!=v for k,v in ident.items()):
            raise RuntimeError('Existing RC13C Q-plan lock identity mismatch.')
        stored=plan.get('q_plan_sha256');tmp=dict(plan);tmp.pop('q_plan_sha256',None);tmp.pop('locked_unix',None)
        raw=json.dumps(tmp,sort_keys=True,separators=(',',':')).encode()
        if stored!=hashlib.sha256(raw).hexdigest():raise RuntimeError('Existing RC13C Q-plan lock hash mismatch.')
        RC13C_Q_PLAN=plan;RC13C_Q_PLAN_LOCKED=True;return plan

    recipient_step=int(round(CORE.recipient_frac*CORE.train_steps))
    m=build_model('resnet',4,2,0,in_ch=1)
    reps={}
    try:
        for rep in S6_CONFIRMATION:
            candidates=[]
            accepted=None
            for c in range(RC13C_Q_MAX_CANDIDATES):
                q=_q_geometry_candidate(m,rep,c,recipient_step);candidates.append({'candidate':c,'noise_max_abs_cosine':q['noise_max_abs_cosine']})
                if q['noise_max_abs_cosine']<=CORE.q_noise_max_abs_cosine:
                    accepted=q;break
            if accepted is None:
                raise ScientificGateStop(f'RC13C no valid matched-Q bank found for rep {rep} within {RC13C_Q_MAX_CANDIDATES} geometry-only candidates.')
            if rep in RC13B_SEALED_CONFIRMATION and int(accepted['candidate'])!=0:
                raise RuntimeError(f'Sealed RC13B rep {rep} does not replay as valid candidate 0 from architecture/seed geometry.')
            reps[str(rep)]={**accepted,'accepted_candidate':int(accepted['candidate']),'tried_candidates':candidates}
    finally:
        del m
        if torch.cuda.is_available():torch.cuda.empty_cache()

    # Canonical failed proposal must replay before we allow completion.
    q3127c0=_q_geometry_candidate(build_model('resnet',4,2,0,in_ch=1),3127,0,recipient_step)
    if q3127c0['noise_max_abs_cosine']<=CORE.q_noise_max_abs_cosine:
        raise RuntimeError('RC13C failed to replay the canonical RC13B candidate-0 support failure for 3127.')
    if abs(float(q3127c0['noise_max_abs_cosine'])-0.201394)>5e-4:
        raise RuntimeError(f'RC13C 3127 candidate-0 geometry differs materially from parent stop: {q3127c0["noise_max_abs_cosine"]:.6f}.')
    del q3127c0
    if torch.cuda.is_available():torch.cuda.empty_cache()

    payload={**_identity_payload(),'parent_zip_sha256':RC13B_SOURCE_ZIP_SHA256,
             'rule':PREREG['q_rejection_sampling_rule'],'threshold':float(CORE.q_noise_max_abs_cosine),
             'max_candidates':int(RC13C_Q_MAX_CANDIDATES),'recipient_step':recipient_step,
             'replicates':reps,'created_before_parent_confirmation_deserialization':True}
    # Scientific plan identity excludes wall-clock metadata, so an independently repeated
    # zero-hour qualification produces the same content hash for the same accepted Q plan.
    raw=json.dumps(payload,sort_keys=True,separators=(',',':')).encode()
    payload['q_plan_sha256']=hashlib.sha256(raw).hexdigest()
    payload['locked_unix']=time.time()
    atomic_json(lock_path,payload);_durability_barrier()
    RC13C_Q_PLAN=payload;RC13C_Q_PLAN_LOCKED=True
    print('RC13C Q PLAN LOCKED',payload['q_plan_sha256'])
    print('RC13C accepted candidates',{r:reps[str(r)]['candidate'] for r in S6_CONFIRMATION})
    return payload

def _extract_parent_after_q_lock(source_path):
    if not RC13C_Q_PLAN_LOCKED or RC13C_Q_PLAN is None:
        raise RuntimeError('Parent RC13B response evidence cannot be extracted before RC13C Q-plan lock.')
    outroot=BASE/'parent_rc13b'/'bundle'
    marker=BASE/'manifests'/'rc13b_parent_import.json'
    if marker.exists():
        j=json.loads(marker.read_text())
        ident=_identity_payload()
        if j.get('source_zip_sha256')!=RC13B_SOURCE_ZIP_SHA256 or j.get('q_plan_sha256')!=RC13C_Q_PLAN['q_plan_sha256'] or any(j.get(k)!=v for k,v in ident.items()):
            raise RuntimeError('Existing parent import marker mismatch.')
        extracted=Path(j['extracted_root']);_verify_extracted_parent_bundle(extracted);return extracted
    if outroot.exists():shutil.rmtree(outroot)
    outroot.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(source_path,'r') as zf:
        root,_=_zip_root_and_names(zf)
        zf.extractall(outroot)
    extracted=outroot/root
    _verify_extracted_parent_bundle(extracted)
    atomic_json(marker,{**_identity_payload(),'source_zip_sha256':RC13B_SOURCE_ZIP_SHA256,
        'q_plan_sha256':RC13C_Q_PLAN['q_plan_sha256'],'extracted_root':str(extracted),
        'policy':'Extraction occurred only after the RC13C Q-plan lock; no parent confirmation outcomes informed the amendment.',
        'time':time.time()})
    _durability_barrier()
    return extracted

def _verify_parent_q_replay_after_lock(parent_root):
    """First allowed deserialization of parent replicate metadata; Q plan is already immutable."""
    if not RC13C_Q_PLAN_LOCKED:raise RuntimeError('Parent replicate metadata cannot be opened before Q-plan lock.')
    for rep in RC13B_SEALED_CONFIRMATION:
        meta=json.loads((parent_root/'blocks/stage06/confirmation'/f'rep_{rep}_meta.json').read_text())
        if int(meta.get('replicate_id',-1))!=rep or meta.get('cohort')!='confirmation':
            raise RuntimeError(f'RC13B sealed meta identity mismatch at {rep}.')
        plan=RC13C_Q_PLAN['replicates'][str(rep)]
        if int(plan['accepted_candidate'])!=0:raise RuntimeError(f'Sealed rep {rep} Q plan no longer candidate 0.')
        if abs(float(meta['q_noise_max_abs_cosine'])-float(plan['noise_max_abs_cosine']))>2e-6:
            raise RuntimeError(f'Sealed RC13B rep {rep} Q max-cos replay mismatch after lock.')
        if float(meta['q_noise_max_abs_cosine'])>CORE.q_noise_max_abs_cosine:
            raise RuntimeError(f'Sealed RC13B rep {rep} violated Q support.')
        for target,pvals in meta['q_noise_pairwise_cosines'].items():
            for k,v in pvals.items():
                if abs(float(v)-float(plan['noise_pairwise_cosines'][target][k]))>2e-6:
                    raise RuntimeError(f'Sealed RC13B rep {rep} Q pairwise replay mismatch at {target}/{k}.')
    return True

def _verify_extracted_parent_bundle(parent_root):
    manifest=json.loads((parent_root/'manifests/artifact_manifest.json').read_text())
    for rec in manifest:
        p=parent_root/rec['path']
        if not p.exists() or int(p.stat().st_size)!=int(rec['bytes']) or file_sha256(p)!=str(rec['sha256']):
            raise RuntimeError(f'Extracted RC13B parent artifact drift: {rec["path"]}')
    return True

def _load_parent_discovery_bundle(parent_root):
    lock=json.loads((parent_root/'artifacts/stage06_discovery_lock.json').read_text())
    for p,h in ((parent_root/'artifacts/stage06_discovery_model.pkl',RC13B_DISCOVERY_MODEL_SHA256),
                (parent_root/'artifacts/stage06_discovery_atlas.parquet',RC13B_DISCOVERY_ATLAS_SHA256),
                (parent_root/'artifacts/stage06_discovery_dimension_curves.parquet',RC13B_DISCOVERY_CURVES_SHA256)):
        if file_sha256(p)!=h:raise RuntimeError(f'Imported RC13B discovery artifact drift: {p.name}')
    with open(parent_root/'artifacts/stage06_discovery_model.pkl','rb') as f:bundles=pickle.load(f)
    if int(bundles['between']['challenger_dim'])!=2 or int(bundles['within']['challenger_dim'])!=2:
        raise RuntimeError('Imported RC13B discovery challenger dimension drift.')
    if float(bundles['between']['jackknife_d1_agreement'])!=1.0 or float(bundles['within']['jackknife_d1_agreement'])!=1.0:
        raise RuntimeError('Imported RC13B discovery d=1 stability drift.')
    return bundles,lock

def _load_parent_competency(parent_root):
    df=pd.read_parquet(parent_root/'artifacts/stage06_confirmation_competency.parquet')
    if len(df)!=30 or set(df.replicate_id.astype(int))!=set(S6_CONFIRMATION):
        raise RuntimeError('RC13B confirmation competency artifact does not cover all frozen 30 tasks.')
    return df.sort_values('replicate_id').reset_index(drop=True)

def _load_sealed_parent_confirmation(parent_root):
    if not RC13C_Q_PLAN_LOCKED:raise RuntimeError('Sealed confirmation cannot be deserialized before Q-plan lock.')
    parts=[];metas=[]
    for rep in RC13B_SEALED_CONFIRMATION:
        stem=parent_root/'blocks/stage06/confirmation'/f'rep_{rep}'
        commit=json.loads(Path(str(stem)+'_commit.json').read_text());meta=json.loads(Path(str(stem)+'_meta.json').read_text())
        tp=Path(str(stem)+'.parquet');mp=Path(str(stem)+'_meta.json')
        if file_sha256(tp)!=commit['sha256_table'] or file_sha256(mp)!=commit['sha256_meta']:
            raise RuntimeError(f'Sealed parent confirmation hash drift at {rep}.')
        df=pd.read_parquet(tp)
        if len(df)!=7*264 or set(df.replicate_id.astype(int))!={rep}:raise RuntimeError(f'Sealed parent row identity mismatch at {rep}.')
        if float(df.q_noise_max_abs_cosine.max())>CORE.q_noise_max_abs_cosine:raise RuntimeError(f'Sealed parent Q support invalid at {rep}.')
        # RC13B predates the explicit candidate-index column. Candidate 0 is known exactly
        # from the frozen seed recipe and is replay-verified before these bytes are opened.
        df=df.copy();df['q_candidate_index']=0;df['rc13c_q_candidate_index']=0;df['rc13c_evidence_origin']='RC13B_sealed_valid_candidate0'
        parts.append(df);metas.append(meta)
    return pd.concat(parts,ignore_index=True),metas

def _run_completion_replicates(parent_root):
    comp=_load_parent_competency(parent_root).set_index('replicate_id')
    parts=[];metas=[]
    for rep in RC13C_COMPLETION_REPLICATES:
        df,meta=_load_or_run_rep(rep,'confirmation')
        plan=RC13C_Q_PLAN['replicates'][str(rep)]
        if int(meta.get('q_candidate_index',-1))!=int(plan['accepted_candidate']):
            raise RuntimeError(f'RC13C rep {rep} candidate index differs from frozen Q plan.')
        if abs(float(meta['q_noise_max_abs_cosine'])-float(plan['noise_max_abs_cosine']))>2e-6:
            raise RuntimeError(f'RC13C rep {rep} measured Q geometry differs from frozen plan.')
        # Deterministic base training must replay the already-observed pre-intervention competency values.
        if abs(float(meta['final_acc'])-float(comp.loc[rep,'final_acc']))>1e-8 or abs(float(meta['recipient_acc'])-float(comp.loc[rep,'recipient_acc']))>1e-8:
            raise RuntimeError(f'RC13C rep {rep} deterministic competency replay mismatch.')
        df=df.copy();df['rc13c_q_candidate_index']=int(plan['accepted_candidate']);df['rc13c_evidence_origin']='RC13C_prospective_completion'
        parts.append(df);metas.append(meta)
    return pd.concat(parts,ignore_index=True),metas

def _assemble_complete_confirmation(parent_root,new_df,new_metas):
    old_df,old_metas=_load_sealed_parent_confirmation(parent_root)
    out=pd.concat([old_df,new_df],ignore_index=True)
    expected=30*7*264
    if len(out)!=expected or set(out.replicate_id.astype(int))!=set(S6_CONFIRMATION):
        raise RuntimeError(f'RC13C complete confirmation atlas mismatch: rows={len(out)} expected={expected}.')
    branches=out[S6_STATE_COLS].drop_duplicates()
    if len(branches)!=30*7:raise RuntimeError('RC13C branch completeness mismatch.')
    ident=out[['replicate_id','task_hash','suite_hash','q_bank_sha256']].drop_duplicates()
    if len(ident)!=30 or ident.task_hash.nunique()!=30 or ident.suite_hash.nunique()!=30 or ident.q_bank_sha256.nunique()!=30:
        raise RuntimeError('RC13C confirmation independence/identity mismatch.')
    disc=pd.read_parquet(parent_root/'artifacts/stage06_discovery_atlas.parquet')
    if set(disc.task_hash.astype(str))&set(out.task_hash.astype(str)):raise RuntimeError('RC13C discovery/confirmation task overlap.')
    if set(disc.suite_hash.astype(str))&set(out.suite_hash.astype(str)):raise RuntimeError('RC13C discovery/confirmation suite overlap.')
    if float(out.q_noise_max_abs_cosine.max())>CORE.q_noise_max_abs_cosine:raise RuntimeError('RC13C complete atlas contains invalid Q geometry.')
    if out['q_candidate_index'].isna().any() or out['rc13c_q_candidate_index'].isna().any():
        raise RuntimeError('RC13C complete atlas has missing Q-candidate provenance.')
    if not np.array_equal(out['q_candidate_index'].to_numpy(int),out['rc13c_q_candidate_index'].to_numpy(int)):
        raise RuntimeError('RC13C Q-candidate provenance columns disagree.')
    for rep in S6_CONFIRMATION:
        got=set(out.loc[out.replicate_id.astype(int)==int(rep),'q_candidate_index'].astype(int).tolist())
        exp={int(RC13C_Q_PLAN['replicates'][str(rep)]['accepted_candidate'])}
        if got!=exp:raise RuntimeError(f'RC13C merged candidate provenance mismatch at rep {rep}: {got} != {exp}')
    out['rc13c_analysis_protocol_hash']=PROTOCOL_HASH
    p=save_table(out,BASE/'artifacts'/'stage06_rc13c_complete_confirmation_atlas')
    all_metas=old_metas+new_metas
    atomic_json(BASE/'artifacts'/'stage06_rc13c_complete_confirmation_meta.json',
        {'protocol_hash':PROTOCOL_HASH,'parent_zip_sha256':RC13B_SOURCE_ZIP_SHA256,'q_plan_sha256':RC13C_Q_PLAN['q_plan_sha256'],
         'replicates':all_metas,'atlas_sha256':file_sha256(p),'expected_rows':expected})
    adf=_anisotropy_table_from_metas(all_metas)
    if len(adf)!=30 or adf.replicate_id.nunique()!=30:raise RuntimeError('RC13C anisotropy panel task coverage mismatch.')
    save_table(adf,BASE/'artifacts'/'stage06_rc13c_complete_anisotropy_panel')
    return out,adf

def _verify_frozen_analysis_semantics(parent_audit):
    # RC13C may differ from RC13B only in the explicitly amended Q/completion plumbing.
    # Everything else that existed in the parent notebook must retain the parent's semantic hash.
    intentionally_changed={
        '_assert_runtime_integrity','_current_protocol_payload','build_q_bank','measure_branch',
        'run_s6_replicate','freeze_discovery_lock','run_stage06_campaign'
    }
    ph=parent_audit['parent_code_manifest'].get('function_hashes',{})
    allowed_missing={'launch_stage06_safely','stage06_workload_contract'}
    mism=[];checked=[];missing=[]
    for name,expected in sorted(ph.items()):
        obj=globals().get(name)
        if obj is None:
            if name not in allowed_missing:missing.append(name)
            continue
        if name in intentionally_changed:continue
        if not isinstance(obj,types.FunctionType):continue
        checked.append(name)
        if _callable_semantic_hash(obj)!=expected:mism.append(name)
    if missing:
        raise RuntimeError(f'RC13C unexpectedly omitted RC13B scientific helpers: {missing}')
    if mism:
        raise RuntimeError(f'RC13C unchanged scientific function differs from RC13B parent: {mism}')
    pclasses=parent_audit['parent_code_manifest'].get('class_hashes',{})
    class_mism=[];class_checked=[]
    for name,expected in sorted(pclasses.items()):
        obj=globals().get(name)
        if obj is None or not isinstance(obj,type):continue
        class_checked.append(name)
        if _class_semantic_hash(obj)!=expected:class_mism.append(name)
    if class_mism:
        raise RuntimeError(f'RC13C scientific class differs from RC13B parent: {class_mism}')
    parent_schema=parent_audit['parent_code_manifest'].get('schema')
    if parent_schema!=_current_schema_payload():
        raise RuntimeError('RC13C scientific analysis schema differs from RC13B parent.')
    # Core scientific thresholds/config must be identical to RC13B.
    parent_core=parent_audit['parent_core_config']
    current=json.loads(json.dumps(asdict(CORE)))
    if parent_core!=current:raise RuntimeError('RC13C CoreConfig differs from RC13B; amendment must not change scientific thresholds.')
    return {'unchanged_parent_functions_checked':checked,'unchanged_parent_classes_checked':class_checked,
            'intentionally_changed_parent_functions':sorted(intentionally_changed),
            'all_unchanged_semantic_hashes_match_parent':True,'analysis_schema_exact_match_parent':True,
            'core_config_exact_match_parent':True}

def run_rc13c_completion_campaign(source_path,parent_audit):
    qlock=prequalify_and_lock_rc13c_q_plan(parent_audit)
    parent_root=_extract_parent_after_q_lock(source_path)
    _verify_parent_q_replay_after_lock(parent_root)
    bundles,dlock=_load_parent_discovery_bundle(parent_root)
    semantics=_verify_frozen_analysis_semantics(parent_audit)
    atomic_json(BASE/'manifests'/'rc13c_frozen_estimator_verification.json',{**_identity_payload(),**semantics,
        'parent_discovery_model_sha256':RC13B_DISCOVERY_MODEL_SHA256,'q_plan_sha256':qlock['q_plan_sha256']})

    if not _done('RC13C_completion_replicates'):
        new_df,new_metas=_run_completion_replicates(parent_root)
        npth=save_table(new_df,BASE/'artifacts'/'stage06_rc13c_new_completion_rows')
        atomic_json(BASE/'artifacts'/'stage06_rc13c_new_completion_meta.json',
            {'replicates':new_metas,'table_sha256':file_sha256(npth),'q_plan_sha256':qlock['q_plan_sha256']})
        _mark('RC13C_completion_replicates',{'reps':list(RC13C_COMPLETION_REPLICATES),'rows':len(new_df)},[
            BASE/'manifests'/'rc13c_q_plan_lock.json',BASE/'artifacts'/'stage06_rc13c_new_completion_rows',
            BASE/'artifacts'/'stage06_rc13c_new_completion_meta.json'])
    else:
        new_df=load_table(BASE/'artifacts'/'stage06_rc13c_new_completion_rows')
        new_metas=json.loads((BASE/'artifacts'/'stage06_rc13c_new_completion_meta.json').read_text())['replicates']

    if not _done('RC13C_complete_confirmation_atlas'):
        complete,adf=_assemble_complete_confirmation(parent_root,new_df,new_metas)
        _mark('RC13C_complete_confirmation_atlas',{'rows':len(complete),'n_tasks':30},[
            BASE/'manifests'/'rc13c_q_plan_lock.json',BASE/'artifacts'/'stage06_rc13c_complete_confirmation_atlas',
            BASE/'artifacts'/'stage06_rc13c_complete_confirmation_meta.json',BASE/'artifacts'/'stage06_rc13c_complete_anisotropy_panel'])
    else:
        complete=load_table(BASE/'artifacts'/'stage06_rc13c_complete_confirmation_atlas')
        adf=load_table(BASE/'artifacts'/'stage06_rc13c_complete_anisotropy_panel')

    if not _done('RC13C_final_H6_analysis'):
        summary,bstate,wstate,contrast,atask=confirm_stage06(complete,bundles,adf)
        # The estimator is unchanged, but RC13C's preregistered reporting boundary explicitly
        # conditions the measurement family on valid-Q support geometry.
        summary['claim_boundary']=PREREG['claim_boundary']
        summary['completion_amendment']={
            'parent_protocol_hash':RC13B_PROTOCOL_HASH,'parent_zip_sha256':RC13B_SOURCE_ZIP_SHA256,
            'sealed_parent_replicates':list(RC13B_SEALED_CONFIRMATION),'new_completion_replicates':list(RC13C_COMPLETION_REPLICATES),
            'q_plan_sha256':qlock['q_plan_sha256'],'q_candidate_indices':{r:qlock['replicates'][r]['accepted_candidate'] for r in qlock['replicates']},
            'discovery_model_sha256':RC13B_DISCOVERY_MODEL_SHA256,
            'q_estimand_conditioning':'P(Q | max pairwise |cos(Q_A,Q_T,Q_B)| <= 0.15 at every frozen target)',
            'analysis_policy':'RC13B H6 estimator unchanged; n=30 evaluated once after completion.'
        }
        atomic_json(BASE/'artifacts'/'stage06_confirmation_summary.json',summary)
        save_table(bstate,BASE/'artifacts'/'stage06_confirmation_between_coordinates')
        save_table(wstate,BASE/'artifacts'/'stage06_confirmation_within_coordinates')
        save_table(contrast,BASE/'artifacts'/'stage06_confirmation_causal_contrasts')
        save_table(atask,BASE/'artifacts'/'stage06_confirmation_anisotropy_task_stats')
        _mark('RC13C_final_H6_analysis',{'claims':summary['claims'],'n_confirmation':30},[
            BASE/'manifests'/'rc13c_q_plan_lock.json',BASE/'manifests'/'rc13c_frozen_estimator_verification.json',
            BASE/'artifacts'/'stage06_rc13c_complete_confirmation_atlas',BASE/'artifacts'/'stage06_rc13c_complete_anisotropy_panel',
            BASE/'artifacts'/'stage06_confirmation_summary.json',BASE/'artifacts'/'stage06_confirmation_between_coordinates',
            BASE/'artifacts'/'stage06_confirmation_within_coordinates',BASE/'artifacts'/'stage06_confirmation_causal_contrasts',
            BASE/'artifacts'/'stage06_confirmation_anisotropy_task_stats'])
    else:
        summary=json.loads((BASE/'artifacts'/'stage06_confirmation_summary.json').read_text())
    final_export(summary)
    print(json.dumps(summary,indent=2))
    return summary


# Fail closed against accidentally invoking the obsolete full RC13B campaign path from this completion notebook.
def freeze_discovery_lock(*args,**kwargs):
    raise RuntimeError('RC13C imports the frozen RC13B discovery lock; discovery refitting is disabled by design.')

def run_stage06_campaign(*args,**kwargs):
    raise RuntimeError('Full Stage-06 rerun is disabled in RC13C. Use run_rc13c_completion_campaign().')


## RC13C zero-hour completion qualification

Before any sealed RC13B confirmation response table is deserialized, RC13C:

- locates the exact canonical RC13B results ZIP and verifies its SHA-256, CRC, member set, and every artifact-manifest payload hash;
- verifies RC13B stopped before H6 claim evaluation and contains exactly 27 committed confirmation replicates (3100–3126), with no final confirmation summary;
- verifies the imported discovery model/atlas/dimension-curve hashes and the RC13B discovery lock;
- recomputes the Q geometry for all 30 confirmation IDs using architecture shapes only;
- requires all sealed 3100–3126 tasks to replay as valid candidate 0 with matching pairwise cosines;
- requires 3127 candidate 0 to reproduce the documented RC13B support failure;
- deterministically rejection-samples a valid whole-bank candidate for each missing task and cryptographically locks the complete plan;
- verifies the frozen confirmatory estimator and CoreConfig are semantically identical to RC13B before final analysis.

Only after this firewall does the notebook deserialize the sealed confirmation tables and run 3127–3129.


In [ ]:
# 6. Zero-hour integrity, positive-control, and adversarial falsification tests

def _synthetic_coef(qrep,h,metric,sign,pos,latent_index=0):
    # QA latent 0: smooth shared response basis.
    if int(latent_index)==0:
        return float((1.0+0.11*qrep+0.07*h+0.13*metric+0.09*pos)*(1.0 if sign>=0 else -0.83))
    # QA latent 1: deliberately non-collinear response basis for the true-2D adversary.
    # Different signs/interactions across query replicate, horizon, metric, and target position
    # make it impossible to collapse to a constant multiple of latent 0.
    if int(latent_index)==1:
        # Alternating query/metric/position parity is intentionally nearly orthogonal to
        # the smooth latent-0 basis on both smoke and research query supports.
        parity=-1.0 if ((int(qrep)+int(metric)+int(pos))%2) else 1.0
        core=parity*(0.8+0.05*float(h))
        return float(core*(1.0 if sign>=0 else -1.0))
    raise ValueError(f'Unsupported synthetic latent_index={latent_index}')

def _synthetic_two_dim_basis_audit():
    v0=[];v1=[]
    for qrep in (CORE.qA_noise_replicate,CORE.qB_noise_replicate,CORE.qT_noise_replicate):
        for posi,_ in enumerate(CORE.target_positions):
            for sign in (-1,1):
                for h in CORE.horizons:
                    for mi,_ in enumerate(CORE.primary_response_metrics):
                        v0.append(_synthetic_coef(qrep,h,mi,sign,posi,0))
                        v1.append(_synthetic_coef(qrep,h,mi,sign,posi,1))
    a=np.asarray(v0,float);b=np.asarray(v1,float)
    cos=float(np.dot(a,b)/(max(np.linalg.norm(a),1e-30)*max(np.linalg.norm(b),1e-30)))
    X=np.column_stack([a,b])
    s=np.linalg.svd(X,compute_uv=False)
    ratio=float(s[1]/max(s[0],1e-30)) if len(s)>1 else 0.0
    if not (np.isfinite(cos) and np.isfinite(ratio) and abs(cos)<=0.90 and ratio>=0.20):
        raise AssertionError(f'Synthetic 2-D response bases are not sufficiently independent: cosine={cos:.4f}, s2/s1={ratio:.4f}')
    return {'basis_abs_cosine':abs(cos),'basis_s2_over_s1':ratio}

def _synthetic_stage06_df(reps,cohort,world,seed=0):
    rng=np.random.default_rng(stable_seed('S6_synthetic',world,cohort,seed))
    rows=[]
    hdoses=CORE.discovery_history_doses if cohort=='discovery' else CORE.confirmation_history_doses
    specs=[('baseline',0.0)]+[('history',float(x)) for x in hdoses]+[('random_a',float(x)) for x in CORE.random_a_doses]
    if cohort=='confirmation':specs += [('random_b',float(x)) for x in CORE.random_b_doses]
    qspec=[('noise',0),('noise',1),('noise',2),('gradient',0),('orthogonal',0),('optimizer_reset',0)]
    for rep in reps:
        b1=rng.normal();b2=rng.normal();b3=rng.normal()
        slopes={f:rng.normal(loc=1.0,scale=.25) for f in ('history','random_a','random_b')}
        for bi,(fam,dose) in enumerate(specs):
            if fam=='baseline':
                w1=w2=0.0
            elif world=='two_dim':
                # QA-only true 2-D intervention state. Discovery-visible history and random_a
                # span different latent axes; confirmation-only random_b uses a third mixture.
                # Small task-specific gain jitter prevents a brittle hand-constructed exact grid
                # while preserving two independent state coordinates.
                g1=1.0+0.08*rng.normal();g2=1.0+0.08*rng.normal()
                if fam=='history':
                    w1=g1*dose;w2=0.12*g2*dose
                elif fam=='random_a':
                    w1=0.10*g1*dose;w2=g2*dose
                elif fam=='random_b':
                    w1=0.82*g1*dose;w2=-0.78*g2*dose
                else:
                    raise ValueError(f'Unexpected two_dim family {fam}')
            else:
                w1=slopes[fam]*dose
                w2=(0.6+0.25*rng.normal())*dose
            # For pure-between, hidden-state branches have no coherent shared response beyond tiny view-specific noise.
            if world=='between_only':w1=w2=0.0
            # Dose-proxy breaks unseen random_b in confirmation.
            break_random_b=(world=='dose_proxy' and cohort=='confirmation' and fam=='random_b')
            for posi,pos in enumerate(CORE.target_positions):
                for kind,qrep in qspec:
                    signs=(-1,1) if kind in SIGNED_KINDS else (0,)
                    for sign in signs:
                        for h in CORE.horizons:
                            vals={}
                            for mi,metric in enumerate(CORE.primary_response_metrics):
                                eps=rng.normal(scale=.015)
                                if kind=='noise' and qrep in (CORE.qA_noise_replicate,CORE.qB_noise_replicate,CORE.qT_noise_replicate):
                                    # Between-task baseline latent.
                                    if world=='within_only':
                                        base_lat = b1 if qrep==CORE.qA_noise_replicate else (b2 if qrep==CORE.qB_noise_replicate else b3)
                                    else:
                                        base_lat=b1
                                    if world=='mismatched' and qrep==CORE.qB_noise_replicate:
                                        base_lat=b2
                                    # Within-task latent.
                                    if world=='two_dim':
                                        latent=_synthetic_coef(qrep,h,mi,sign,posi,0)*w1+_synthetic_coef(qrep,h,mi,sign,posi,1)*w2
                                    else:
                                        latent=_synthetic_coef(qrep,h,mi,sign,posi,0)*w1
                                    if world=='mismatched' and qrep==CORE.qB_noise_replicate:
                                        latent=_synthetic_coef(qrep,h,mi,sign,posi,0)*(rng.normal() if fam!='baseline' else 0.0)
                                    if break_random_b and qrep==CORE.qT_noise_replicate and h==CORE.primary_horizon:
                                        latent=rng.normal(scale=2.0)
                                    val=_synthetic_coef(qrep,h,mi,sign,posi,0)*base_lat+latent+eps
                                else:
                                    val=0.25*b1+0.1*w1+eps
                                vals[metric]=float(val)
                            base={
                              'family':'resnet','scale':1,'replicate_id':int(rep),'cohort':cohort,
                              'task_hash':f'synth_{rep}','suite_hash':'synthetic','task_K':32,'task_r':4,'label_noise':0.0,'task_mode':'fixedK',
                              'checkpoint':35,'checkpoint_frac':0.70,'branch_id':'baseline_0' if fam=='baseline' else f'{fam}_{dose:+.2f}',
                              'instrument_family':fam,'surgery_dose':float(dose),'surgery_alpha':0.1 if fam!='baseline' else 0.0,
                              'kind':kind,'target_pos':float(pos),'sign':int(sign),'replicate':int(qrep),'horizon':int(h),
                              'instantaneous_max_logit_diff':0.0,'function_invariance_passed':True,'optimizer_isolation_passed':True,
                              'exp_avg_changed':bool(fam!='baseline'),'exp_avg_sq_unchanged':True,'optimizer_step_unchanged':True,
                              'other_optimizer_state_unchanged':True,'param_groups_unchanged':True,
                              'q_calibration_passed':True,'min_secant_cosine':0.9999,'max_relative_secant_norm_drift':0.001,'q_noise_max_abs_cosine':0.01,
                              'instrument_hidden_min_secant_cosine':0.9999,'instrument_hidden_max_relative_secant_norm_drift':0.001,
                              'q_bank_sha256':f'q_{rep}','protocol_hash':PROTOCOL_HASH,'env_compat_hash':ENV_COMPAT_HASH,
                            }
                            for col in OBS_COLS:base[col]=0.0
                            rows.append({**base,**vals})
    return pd.DataFrame(rows)


def _synthetic_anisotropy_df(reps,positive=True):
    rows=[]
    mult=20.0 if positive else 1.0
    for i,r in enumerate(reps):
        hist=0.002*(1+0.05*np.sin(i))
        vals=[hist*mult*(1+.03*j) for j in range(CORE.anisotropy_random_directions)]
        rows.append({'replicate_id':int(r),'history_relative_m1_displacement':hist,
                     'history_target_relative_error':0.01,'history_calibration_capped':False,
                     'history_sign_asymmetry':0.01,'history_mean_one_step_rmse':CORE.surgery_target_one_step_rmse,
                     'history_hidden_min_secant_cosine':0.999,'history_hidden_max_relative_secant_norm_drift':0.001,
                     'random_axis_ids_json':json.dumps([f'random_{j:02d}' for j in range(CORE.anisotropy_random_directions)]),
                     'random_relative_m1_displacements_json':json.dumps(vals),
                     'random_valid_json':json.dumps([True]*CORE.anisotropy_random_directions),
                     'random_capped_json':json.dumps([False]*CORE.anisotropy_random_directions),
                     'random_hidden_min_secant_cosines_json':json.dumps([0.999]*CORE.anisotropy_random_directions),
                     'random_hidden_max_relative_secant_norm_drifts_json':json.dumps([0.001]*CORE.anisotropy_random_directions),
                     'random_target_relative_errors_json':json.dumps([0.01]*CORE.anisotropy_random_directions),
                     'random_sign_asymmetries_json':json.dumps([0.01]*CORE.anisotropy_random_directions),
                     'random_mean_one_step_rmse_json':json.dumps([CORE.surgery_target_one_step_rmse]*CORE.anisotropy_random_directions),
                     'random_errors_json':json.dumps(['']*CORE.anisotropy_random_directions),
                     'n_requested':CORE.anisotropy_random_directions,'protocol_hash':PROTOCOL_HASH})
    return pd.DataFrame(rows)

def _production_nested_cv_contract_qa():
    # Exercise the exact paid research nested-CV split contract on tiny matrices so
    # zero-hour QA verifies 5x4 grouping semantics without rerunning the whole
    # high-dimensional adversarial suite at production cost.
    if PROFILE!='research':
        return {'profile':'smoke','skipped_exact_research_contract':True}
    assert CORE.rcca_outer_folds==5 and CORE.rcca_inner_folds==4 and CORE.rcca_final_reg_folds==5
    assert tuple(CORE.candidate_dims)==(1,2,4,8)
    assert len(CORE.target_positions)==4 and int(CORE.primary_horizon)==16
    rng=np.random.default_rng(stable_seed('S6_production_nested_cv_contract'))
    groups=np.repeat(np.arange(20,dtype=int),2)
    z1=rng.normal(size=len(groups));z2=rng.normal(size=len(groups))
    A=np.column_stack([z1,z2,rng.normal(scale=.2,size=len(groups)),rng.normal(scale=.2,size=len(groups))])
    B=np.column_stack([z1+.03*rng.normal(size=len(groups)),z2+.03*rng.normal(size=len(groups)),rng.normal(scale=.2,size=len(groups)),rng.normal(scale=.2,size=len(groups))])
    T=np.column_stack([z1,z2,z1-z2,z1+z2])
    outer=list(GroupKFold(5).split(A,groups=groups))
    assert len(outer)==5
    for tr,va in outer:
        assert set(groups[tr]).isdisjoint(set(groups[va]))
        inner=list(GroupKFold(4).split(A[tr],groups=groups[tr]))
        assert len(inner)==4
        for itr,iva in inner:
            assert set(groups[tr][itr]).isdisjoint(set(groups[tr][iva]))
    curve=_rcca_cv_curve(A,B,T,groups)
    if set(curve.selection_method.astype(str))!={'nested_task_grouped_cv'}:
        raise AssertionError('Production nested-CV contract did not report nested task-grouped selection.')
    for js in curve.nested_outer_fold_mse_json:
        if len(json.loads(js))!=5:
            raise AssertionError('Production nested-CV contract did not execute five outer task folds.')
    return {'profile':'research','outer_folds':5,'inner_folds':4,'final_reg_folds':5,'group_disjoint':True,'curve_rows':int(len(curve))}

def _synthetic_full_pipeline_tests():
    global CORE,_QA_BOOTSTRAP_N,_QA_EXPECTED_CONFIRMATION_N
    old_core=CORE;oldb=globals().get('_QA_BOOTSTRAP_N');oldn=globals().get('_QA_EXPECTED_CONFIRMATION_N')
    try:
        production_cv_contract=_production_nested_cv_contract_qa()
        # Research-strength claim gates with a smaller, QA-only grouped-CV grid.
        # This changes only integrity-test runtime; the paid estimator keeps the
        # frozen production CORE (5x4 nested CV, candidate d=1/2/4/8).
        d=asdict(CORE)
        d.update({
          'rcca_outer_folds':3,'rcca_inner_folds':2,'rcca_final_reg_folds':3,'candidate_dims':(1,2,4),
          'between_min_query_corr':0.60,'between_min_query_corr_ci_lower':0.20,'min_between_predictive_improvement':0.03,
          'within_min_mean_query_corr':0.45,'within_min_query_corr_ci_lower':0.10,'within_min_positive_corr_fraction':0.65,
          'min_within_predictive_improvement':0.03,'min_unseen_instrument_improvement':0.03,
          'min_contrast_improvement_vs_zero':0.05,'min_contrast_improvement_vs_rich':0.02,
          'between_full_context_noninferiority_margin':0.20,'between_max_residual_gain_given_z':0.10,
          'within_full_context_noninferiority_margin':0.20,'within_max_residual_gain_given_z':0.10,
          'contrast_full_context_noninferiority_margin':0.25,'contrast_max_residual_gain_given_z':0.10,
          'anisotropy_median_ratio_min':10.0,'anisotropy_ratio_ci_lower_min':5.0,
          'anisotropy_fraction_above_floor':0.80,'anisotropy_task_ratio_floor':5.0,
          'anisotropy_best_random_median_min':3.0,'anisotropy_best_random_ci_lower_min':1.5,
        })
        CORE=CoreConfig(**d);_QA_BOOTSTRAP_N=192
        dreps=tuple(range(9300,9308));creps=tuple(range(9400,9410));_QA_EXPECTED_CONFIRMATION_N=len(creps)
        two_dim_basis_audit=_synthetic_two_dim_basis_audit()

        # 1. Shared between + within 1-D world must pass all three claims.
        dd=_synthetic_stage06_df(dreps,'discovery','positive');cc=_synthetic_stage06_df(creps,'confirmation','positive')
        bb=fit_stage06_discovery(dd)
        for _lab in ('between','within'):
            if set(bb[_lab]['dimension_curve'].selection_method.astype(str))!={'nested_task_grouped_cv'}:
                raise AssertionError('Stage-06 dimension curve did not use nested task-grouped CV.')
        ss,*_=confirm_stage06(cc,bb,_synthetic_anisotropy_df(creps,True))
        if not all(ss['claims'].values()):raise AssertionError(f'Known positive Stage-06 world failed: {ss["claims"]}')

        # 2. Pure between-task phenotype: B must pass while W fails.
        dd=_synthetic_stage06_df(dreps,'discovery','between_only');cc=_synthetic_stage06_df(creps,'confirmation','between_only')
        bb=fit_stage06_discovery(dd);ss,*_=confirm_stage06(cc,bb,_synthetic_anisotropy_df(creps,True))
        if not ss['claims']['H6_B_between_task_predictive_phenotype'] or ss['claims']['H6_W_within_task_causal_state']:
            raise AssertionError(f'Between-only adversary misclassified: {ss["claims"]}')

        # 3. Pure within-task state: W must pass while B fails.
        dd=_synthetic_stage06_df(dreps,'discovery','within_only');cc=_synthetic_stage06_df(creps,'confirmation','within_only')
        bb=fit_stage06_discovery(dd);ss,*_=confirm_stage06(cc,bb,_synthetic_anisotropy_df(creps,True))
        if ss['claims']['H6_B_between_task_predictive_phenotype'] or not ss['claims']['H6_W_within_task_causal_state']:
            raise AssertionError(f'Within-only adversary misclassified: {ss["claims"]}')

        # 4. True 2-D within-task system must not be accepted as a 1-D causal state.
        dd=_synthetic_stage06_df(dreps,'discovery','two_dim');cc=_synthetic_stage06_df(creps,'confirmation','two_dim')
        # Verify the generated centered matched-query data themselves contain a material second mode.
        _a2,_b2,_t2,_h2,_m2=prepare_stage06_views(dd,'within')
        _x2=np.concatenate([StandardScaler().fit_transform(_a2),StandardScaler().fit_transform(_b2),StandardScaler().fit_transform(_t2)],axis=1)
        _s2=np.linalg.svd(_x2,compute_uv=False)
        _data_ratio=float(_s2[1]/max(_s2[0],1e-30))
        if not np.isfinite(_data_ratio) or _data_ratio<0.20:
            raise AssertionError(f'True-2D QA data collapsed before fitting: s2/s1={_data_ratio:.4f}')
        bb=fit_stage06_discovery(dd);ss,*_=confirm_stage06(cc,bb,_synthetic_anisotropy_df(creps,True))
        if ss['claims']['H6_W_within_task_causal_state']:
            raise AssertionError(f'True 2-D within-task adversary incorrectly passed H6-W: N={ss["H6_W"]["N_dimension"]}, data_s2_s1={_data_ratio:.4f}')

        # 5. Mismatched physical view must fail query-invariant within-state identity.
        dd=_synthetic_stage06_df(dreps,'discovery','mismatched');cc=_synthetic_stage06_df(creps,'confirmation','mismatched')
        bb=fit_stage06_discovery(dd);ss,*_=confirm_stage06(cc,bb,_synthetic_anisotropy_df(creps,True))
        if ss['claims']['H6_W_within_task_causal_state']:
            raise AssertionError('Mismatched-query adversary incorrectly passed H6-W.')

        # 6. Discovery-visible dose proxy cannot rescue confirmation-only random_b transport.
        dd=_synthetic_stage06_df(dreps,'discovery','dose_proxy');cc=_synthetic_stage06_df(creps,'confirmation','dose_proxy')
        bb=fit_stage06_discovery(dd);ss,*_=confirm_stage06(cc,bb,_synthetic_anisotropy_df(creps,True))
        if ss['claims']['H6_W_within_task_causal_state']:
            raise AssertionError('Dose-proxy adversary incorrectly passed H6-W.')

        # 7. Null optimizer anisotropy must fail H6-A machinery.
        ap,_,_=analyze_anisotropy(_synthetic_anisotropy_df(creps,False))
        if ap:raise AssertionError('Null anisotropy adversary incorrectly passed H6-A.')

        return {'production_nested_cv_contract':production_cv_contract,
                'positive_all_claims':True,'between_only_classified':True,'within_only_classified':True,
                'two_dimensional_rejected':True,'two_dimensional_basis_audit':two_dim_basis_audit,
                'mismatched_query_rejected':True,'dose_proxy_rejected':True,
                'anisotropy_null_rejected':True}
    finally:
        CORE=old_core
        if oldb is None:globals().pop('_QA_BOOTSTRAP_N',None)
        else:_QA_BOOTSTRAP_N=oldb
        if oldn is None:globals().pop('_QA_EXPECTED_CONFIRMATION_N',None)
        else:_QA_EXPECTED_CONFIRMATION_N=oldn

def _real_branch_writer_qa():
    global CORE
    old_core=CORE
    try:
        d=asdict(CORE);d['horizons']=(0,1);d['context_horizons']=(1,);d['primary_horizon']=1;CORE=CoreConfig(**d)
        suite=make_suite(SuiteSpec(train_n=128,direction_n=16,calibration_n=16,response_n=16,evaluation_n=32,sealed_n=32,seed=stable_seed('S6_real_writer_suite')))
        task,_=make_task_variant(suite,16,4,0.0,'fixedK',99981)
        ts=TrainSpec(steps=4,batch_size=32,lr=CORE.lr,weight_decay=CORE.weight_decay,beta1=CORE.adam_beta1,beta2=CORE.adam_beta2,
            eps=CORE.adam_eps,amsgrad=CORE.adam_amsgrad,maximize=CORE.adam_maximize,foreach=CORE.adam_foreach,
            capturable=CORE.adam_capturable,differentiable=CORE.adam_differentiable,fused=CORE.adam_fused)
        bank,sch,dt,_=build_checkpoint_bank(task,'calib',1,99981,ts);rec=bank[int(round(CORE.recipient_frac*ts.steps))]
        snap=make_surgery_snapshot(task,dt,'calib',1,99981,ts,rec,'baseline',0.0,0.0,None,None);qbank=build_q_bank(task,dt,'calib',1,99981,ts,rec)
        df=measure_branch(task,dt,'calib',1,99981,ts,snap,sch,99981,'qa','baseline_0','baseline',0.0,0.0,None,None,q_bank=qbank)
        required=set(S6_STATE_COLS+S6_QUERY_BASE+list(CORE.primary_response_metrics)+OBS_COLS+['q_calibration_passed','q_bank_sha256','function_invariance_passed','optimizer_isolation_passed'])
        if not required.issubset(df.columns):raise AssertionError(f'Real branch writer missing columns: {sorted(required-set(df.columns))[:10]}')
        M=s6_response_matrix(df);qa,qb,qt=s6_query_partition(M.columns)
        if len(qa)!=len(qb) or not qt:raise AssertionError('Real writer matched-query partition invalid.')
        if not df.q_calibration_passed.astype(bool).all():raise AssertionError('Real branch writer produced failed Q calibration.')
        if df.q_bank_sha256.nunique()!=1:raise AssertionError('Real branch writer Q bank identity drift.')
        qcos=float(df.q_noise_max_abs_cosine.max())
        if qcos>CORE.q_noise_max_abs_cosine:raise AssertionError(f'Real branch writer Q directions insufficiently independent: {qcos}')
        return {'rows':int(len(df)),'columns':int(len(df.columns)),'q_pass_fraction':float(df.q_calibration_passed.mean()),
                'qA':len(qa),'qB':len(qb),'qT':len(qt),'q_noise_max_abs_cosine':qcos,
                'q_bank_sha256':str(df.q_bank_sha256.iloc[0])}
    finally:CORE=old_core

def _persistence_resume_regression():
    cohort='qa_persistence';rep=99998;stem=_rep_stem(cohort,rep);mp=_meta_path(cohort,rep);cp=_commit_path(cohort,rep)
    for p in (Path(str(stem)+'.parquet'),Path(str(stem)+'.csv'),mp,cp):
        try:p.unlink()
        except FileNotFoundError:pass
    calls={'n':0};real_runner=globals()['run_s6_replicate']
    def fake_runner(r,c):
        calls['n']+=1
        return pd.DataFrame({'qa_value':[1.0],'replicate_id':[int(r)],'cohort':[str(c)]}),{'replicate_id':int(r),'cohort':str(c),'qa':True}
    try:
        globals()['run_s6_replicate']=fake_runner
        _load_or_run_rep(rep,cohort)
        if calls['n']!=1 or _validate_rep_commit(cohort,rep) is None:raise RuntimeError('Fresh persistence QA write/verify failed.')
        _load_or_run_rep(rep,cohort)
        if calls['n']!=1:raise RuntimeError('Persistence QA resume recomputed an already committed replicate.')
        c=json.loads(cp.read_text());c['scientific_code_hash']='INTENTIONALLY_WRONG';atomic_json(cp,c)
        if _validate_rep_commit(cohort,rep) is not None:raise RuntimeError('Provenance QA accepted mismatched scientific-code fingerprint.')
        return {'fresh_write_verified':True,'resume_verified':True,'mismatched_code_rejected':True,'passed':True}
    finally:
        globals()['run_s6_replicate']=real_runner
        for p in (Path(str(stem)+'.parquet'),Path(str(stem)+'.csv'),mp,cp):
            try:p.unlink()
            except FileNotFoundError:pass

def _stage_marker_corruption_regression():
    qa=BASE/'artifacts'/'qa_stage_marker_payload.json';stage='QA_stage_marker_hash'
    atomic_json(qa,{'qa':1,'protocol_hash':PROTOCOL_HASH});_mark(stage,{'qa':True},[qa])
    if not _done(stage):raise RuntimeError('Stage-marker QA could not verify clean output.')
    original=qa.read_bytes()
    try:
        qa.write_bytes(original+b'\n')
        if _done(stage):raise RuntimeError('Stage-marker QA accepted corrupted output.')
    finally:
        qa.write_bytes(original)
        try:_stage_marker(stage).unlink()
        except FileNotFoundError:pass
        try:qa.unlink()
        except FileNotFoundError:pass
    return {'clean_verified':True,'corruption_rejected':True,'passed':True}

def _semantic_const(v):
    import types
    if isinstance(v,types.CodeType):return {'code':_semantic_code_payload(v)}
    if isinstance(v,bytes):return {'bytes':v.hex()}
    if isinstance(v,tuple):return {'tuple':[_semantic_const(x) for x in v]}
    if isinstance(v,frozenset):return {'frozenset':sorted([repr(x) for x in v])}
    if v is None or isinstance(v,(str,int,float,bool,complex)):return {'literal':repr(v)}
    return {'repr':repr(v)}

def _semantic_code_payload(co):
    return {'co_code':co.co_code.hex(),'co_consts':[_semantic_const(x) for x in co.co_consts],
            'co_names':list(co.co_names),'co_varnames':list(co.co_varnames),'co_freevars':list(co.co_freevars),'co_cellvars':list(co.co_cellvars),
            'co_argcount':int(co.co_argcount),'co_posonlyargcount':int(getattr(co,'co_posonlyargcount',0)),
            'co_kwonlyargcount':int(co.co_kwonlyargcount),'co_nlocals':int(co.co_nlocals),'co_stacksize':int(co.co_stacksize),
            'co_flags':int(co.co_flags),'co_exceptiontable':getattr(co,'co_exceptiontable',b'').hex()}

def _callable_semantic_hash(fn):
    payload={'code':_semantic_code_payload(fn.__code__),'defaults':repr(getattr(fn,'__defaults__',None)),'kwdefaults':repr(getattr(fn,'__kwdefaults__',None))}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(',',':')).encode()).hexdigest()

def _current_protocol_payload():
    return {
      'project':PROJECT,'version':VERSION,'protocol_revision':PROTOCOL_REVISION,'profile':PROFILE,'master_seed':MASTER_SEED,
      'core':asdict(CORE),
      'replicates':{'discovery':S6_DISCOVERY,'confirmation':S6_CONFIRMATION,
                    'sealed_parent_confirmation':RC13B_SEALED_CONFIRMATION,'completion':RC13C_COMPLETION_REPLICATES},
      'q_completion_rule':{'max_candidates':RC13C_Q_MAX_CANDIDATES,'namespace':RC13C_Q_REJECTION_NAMESPACE,
                           'candidate0':'exact RC13B seed recipe','acceptance_threshold':CORE.q_noise_max_abs_cosine},
      'parent_evidence':PARENT_EVIDENCE,'prereg':PREREG,
    }

def _current_schema_payload():
    return {'ROLE_NAMES':list(ROLE_NAMES),'SIGNED_KINDS':sorted(list(SIGNED_KINDS)),'S6_STATE_COLS':list(S6_STATE_COLS),
            'S6_QUERY_BASE':list(S6_QUERY_BASE),'OBS_COLS':list(OBS_COLS),'MICRO_TARGET_COLS':list(MICRO_TARGET_COLS),
            'primary_response_metrics':list(CORE.primary_response_metrics),'MASTER_SEED':int(MASTER_SEED)}

def _class_semantic_hash(cls):
    methods={}
    for k,v in sorted(cls.__dict__.items()):
        vv=v.__func__ if isinstance(v,(staticmethod,classmethod)) else v
        if callable(vv) and hasattr(vv,'__code__'):methods[k]=_callable_semantic_hash(vv)
    return hashlib.sha256(json.dumps(methods,sort_keys=True,separators=(',',':')).encode()).hexdigest()

def _assert_runtime_integrity(check_code=True):
    """Fail closed on mutable-notebook drift after the frozen protocol/code identities were created."""
    cur=_current_protocol_payload()
    h=hashlib.sha256(json.dumps(cur,sort_keys=True,separators=(',',':')).encode()).hexdigest()
    if h!=PROTOCOL_HASH:
        raise RuntimeError(f'Frozen RC13C protocol mutated in the live notebook runtime: {h} != {PROTOCOL_HASH}')
    if str(DEVICE)!=str(ENV_COMPAT['device']):raise RuntimeError('Runtime DEVICE drifted from the frozen environment identity.')
    if bool(torch.are_deterministic_algorithms_enabled())!=bool(STRICT_DETERMINISM):raise RuntimeError('PyTorch deterministic-algorithm state drifted after environment freeze.')
    if torch.cuda.is_available():
        if bool(torch.backends.cudnn.benchmark):raise RuntimeError('cuDNN benchmark unexpectedly enabled after deterministic freeze.')
        if not bool(torch.backends.cudnn.deterministic):raise RuntimeError('cuDNN deterministic mode unexpectedly disabled.')
        if bool(torch.backends.cuda.matmul.allow_tf32) or bool(torch.backends.cudnn.allow_tf32):raise RuntimeError('TF32 unexpectedly enabled after deterministic freeze.')
    man=globals().get('SCIENTIFIC_CODE_MANIFEST')
    if man:
        if _current_schema_payload()!=man.get('schema'):raise RuntimeError('Scientific schema mutated after code fingerprint freeze.')
        if check_code:
            ambient=globals().get('_S6_AMBIENT_MAIN_OBJECTS',{})
            for name,expected in man.get('function_hashes',{}).items():
                obj=globals().get(name)
                if not isinstance(obj,types.FunctionType) or getattr(obj,'__module__',None)!='__main__' or ambient.get(name) is obj:
                    raise RuntimeError(f'Frozen scientific function missing/rebound: {name}')
                got=_callable_semantic_hash(obj)
                if got!=expected:raise RuntimeError(f'Frozen scientific function body drifted: {name}')
            for name,expected in man.get('class_hashes',{}).items():
                cls=globals().get(name)
                if not isinstance(cls,type) or getattr(cls,'__module__',None)!='__main__' or ambient.get(name) is cls:
                    raise RuntimeError(f'Frozen scientific class missing/rebound: {name}')
                got=_class_semantic_hash(cls)
                if got!=expected:raise RuntimeError(f'Frozen scientific class body drifted: {name}')
    return True

def _scientific_code_manifest():
    """Hash every user-defined function/class loaded before the campaign.

    This avoids a manual-list provenance hole: changing a helper body must change the campaign
    fingerprint even when a top-level caller still references the same helper name.
    """
    import types
    entries={}
    ambient=globals().get('_S6_AMBIENT_MAIN_OBJECTS',{})
    for name,obj in sorted(globals().items()):
        if isinstance(obj,types.FunctionType) and getattr(obj,'__module__',None)=='__main__':
            if ambient.get(name) is obj:
                continue
            entries[name]=_callable_semantic_hash(obj)
    class_hashes={}
    for cname,cls in sorted(globals().items()):
        if not isinstance(cls,type) or getattr(cls,'__module__',None)!='__main__':continue
        if ambient.get(cname) is cls:continue
        class_hashes[cname]=_class_semantic_hash(cls)
    schema=_current_schema_payload()
    payload={'protocol_hash':PROTOCOL_HASH,'implementation_revision':IMPLEMENTATION_REVISION,
             'python_major_minor':[sys.version_info.major,sys.version_info.minor],'environment_compatibility_hash':ENV_COMPAT_HASH,
             'function_hashes':entries,'class_hashes':class_hashes,'schema':schema,
             'fingerprint_policy':'all functions/classes defined or overwritten by this notebook before campaign; pre-existing ambient __main__ objects excluded by identity; semantic bytecode/constants/names/defaults; excludes co_filename, firstlineno and line tables'}
    payload['campaign_code_hash']=hashlib.sha256(json.dumps({'functions':entries,'classes':class_hashes,'schema':schema},sort_keys=True,separators=(',',':')).encode()).hexdigest()
    atomic_json(BASE/'manifests'/'scientific_code_manifest.json',payload);return payload

def run_stage06_integrity_tests():
    global CORE
    # Structural anti-leakage proof from synthetic column labels.
    cols=[]
    for k,rps in [('noise',[0,1,2]),('gradient',[0]),('orthogonal',[0]),('optimizer_reset',[0])]:
        for rp in rps:
            for pos in CORE.target_positions[:1]:
                for sign in ((-1,1) if k in SIGNED_KINDS else (0,)):
                    for h in CORE.horizons:
                        for met in CORE.primary_response_metrics:cols.append((k,pos,sign,rp,h,met))
    qa,qb,qt=s6_query_partition(cols)
    assert all(c[0]=='noise' and int(c[3])==CORE.qA_noise_replicate and int(c[4]) in CORE.context_horizons for c in qa)
    assert all(c[0]=='noise' and int(c[3])==CORE.qB_noise_replicate and int(c[4]) in CORE.context_horizons for c in qb)
    assert all(c[0]=='noise' and int(c[3])==CORE.qT_noise_replicate and int(c[4])==CORE.primary_horizon for c in qt)
    assert not set(qa)&set(qb) and not set(qa)&set(qt) and not set(qb)&set(qt)
    # Physical-zero semantics: StandardScaler zero is not generally physical zero.
    _ys=StandardScaler().fit(np.array([[1.0,2.0],[3.0,6.0],[5.0,10.0]]))
    assert not np.allclose(_ys.transform(np.zeros((1,2))),np.zeros((1,2)))
    # Exercise the actual zero-change gate: a predictor at the discovery mean must beat a
    # physically nonzero zero-effect location in standardized coordinates.
    _zm=pd.DataFrame({'replicate_id':np.arange(6)})
    _zy=np.zeros((6,1));_zt={'Y_zero':np.full((6,1),-2.0),'predA_1':np.zeros((6,1)),'predB_1':np.zeros((6,1))}
    _zpass,_zstat=_zero_prediction_gate(_zm,_zy,_zt,0.05,'S6_QA_physical_zero')
    assert _zpass and all(v['coverage_passed'] for v in _zstat.values())
    # Finite Monte-Carlo sign-flip tests must never report impossible p=0.
    assert _exact_or_mc_signflip(np.ones(30),'greater',stable_seed('S6_QA_signflip'))>0.0
    # Non-finite task statistics must fail closed; they may never be silently dropped from n.
    _finite_rejected=0
    for _fn in (
        lambda:_bootstrap_replicates({1:1.0,2:float('nan')},n_boot=8,seed=1),
        lambda:_exact_or_mc_signflip([1.0,float('nan')],'greater',1),
        lambda:_mse(np.array([0.0,float('nan')]),np.array([0.0,0.0])),
    ):
        try:_fn()
        except RuntimeError:_finite_rejected+=1
    assert _finite_rejected==3
    # Degenerate per-task identity may not silently drop a task.
    _im=pd.DataFrame({'replicate_id':np.repeat([1,2],3)})
    _itr={'zA_1':np.array([[0.],[1.],[2.],[0.],[0.],[0.]]),
          'zB_1':np.array([[0.],[1.],[2.],[0.],[1.],[2.]])}
    _is=_within_identity(_im,_itr);assert not _is['all_tasks_valid'] and _is['expected_n']==2

    print('S6 QA structural query partition / physical-zero / inference-coverage semantics PASS')
    # Real hidden-state surgery identity/isolation and anisotropy-axis construction on a miniature network.
    suite=make_suite(SuiteSpec(train_n=512,direction_n=64,calibration_n=64,response_n=64,evaluation_n=128,sealed_n=128,seed=stable_seed('S6_selftest_suite')))
    task,_=make_task_variant(suite,16,4,0.0,'fixedK',99991)
    ts=TrainSpec(steps=4,batch_size=32,lr=CORE.lr,weight_decay=CORE.weight_decay,beta1=CORE.adam_beta1,beta2=CORE.adam_beta2,
        eps=CORE.adam_eps,amsgrad=CORE.adam_amsgrad,maximize=CORE.adam_maximize,foreach=CORE.adam_foreach,
        capturable=CORE.adam_capturable,differentiable=CORE.adam_differentiable,fused=CORE.adam_fused)
    bank,sch,dt,_=build_checkpoint_bank(task,'calib',1,99991,ts)
    don=bank[int(round(CORE.donor_frac*ts.steps))];rec=bank[int(round(CORE.recipient_frac*ts.steps))]
    rm,ro=load_snapshot(task,'calib',1,99991,ts,rec);dm,do=load_snapshot(task,'calib',1,99991,ts,don)
    axes,ameta=build_m1_axes(rm,ro,dm,do,99991,need_random_b=True)
    aa,aam=build_anisotropy_axes(rm,ro,axes['history'],99991,CORE.anisotropy_random_directions)
    assert len(aa)==CORE.anisotropy_random_directions
    del rm,ro,dm,do
    # History QA deliberately exercises the full frozen profile target. The miniature 4-step
    # network is not a valid scientific population for random-direction potency: an arbitrary
    # orthogonal direction can legitimately be too weak to reach the research 5e-4 target under
    # the 75% m1 displacement cap. For the random-axis *software/locality* QA only, use a fixed
    # small target that is independent of research outcomes. Paid H6-A calibration below remains
    # unchanged and uses CORE.surgery_target_one_step_rmse exactly.
    hcal=calibrate_hidden_axis(task,'calib',1,99991,ts,rec,sch,dt,axes['history'],'qa_history')
    _qa_core=CORE
    _qa_cfg=asdict(CORE);_qa_cfg['surgery_target_one_step_rmse']=1e-4;_qa_cfg['surgery_min_one_step_rmse']=1e-6
    try:
        CORE=CoreConfig(**_qa_cfg)
        rcal=calibrate_hidden_axis(task,'calib',1,99991,ts,rec,sch,dt,aa[sorted(aa)[0]],'qa_anisotropy_random_mechanics')
    finally:
        CORE=_qa_core
    assert hcal['hidden_min_secant_cosine']>=CORE.hidden_locality_min_secant_cosine
    assert hcal['hidden_max_relative_secant_norm_drift']<=CORE.hidden_locality_max_relative_norm_drift
    assert rcal['hidden_min_secant_cosine']>=CORE.hidden_locality_min_secant_cosine
    assert rcal['hidden_max_relative_secant_norm_drift']<=CORE.hidden_locality_max_relative_norm_drift
    snap=make_surgery_snapshot(task,dt,'calib',1,99991,ts,rec,'history',1.0,hcal['alpha'],axes['history'],ameta['history'])
    assert snap['function_invariance_passed'] and snap['optimizer_isolation_passed'] and snap['exp_avg_changed']
    assert snap['exp_avg_sq_unchanged'] and snap['step_unchanged'] and snap['param_groups_unchanged']

    print('S6 QA real surgery/hidden-locality/axis construction PASS')
    full=_synthetic_full_pipeline_tests()
    print('S6 QA synthetic discrimination suite PASS')
    writer=_real_branch_writer_qa()
    print('S6 QA real branch writer PASS')
    persistence=_persistence_resume_regression()
    marker=_stage_marker_corruption_regression()
    out={'protocol_hash':PROTOCOL_HASH,'matched_query_counts':{'A':len(qa),'B':len(qb),'T':len(qt)},
         'real_surgery_function_invariant':True,'real_optimizer_field_isolation':True,
         'real_hidden_history_locality':{'min_cosine':float(hcal['hidden_min_secant_cosine']),'max_norm_drift':float(hcal['hidden_max_relative_secant_norm_drift'])},
         'real_hidden_random_locality':{'qa_target_one_step_rmse':1e-4,'min_cosine':float(rcal['hidden_min_secant_cosine']),'max_norm_drift':float(rcal['hidden_max_relative_secant_norm_drift'])},
         'anisotropy_axes_constructed':len(aa),'real_branch_writer':writer,'persistence_resume':persistence,
         'stage_marker_corruption':marker,'synthetic_full_pipeline':full,
         'scientific_code_hash':SCIENTIFIC_CODE_MANIFEST['campaign_code_hash'],'passed':True}
    atomic_json(BASE/'manifests'/'stage06_integrity_tests.json',out)
    print('STAGE06 RC13B INTEGRITY TESTS PASSED',json.dumps(out,indent=2))
    return out

# Scientific fingerprint and integrity execution are intentionally deferred until the final
# pre-launch cell so every user-defined helper (including workload/launch handling) is bound.

def _rc13c_nonzero_q_candidate_replay_qa():
    """Exercise a real nonzero candidate through build_q_bank before paid completion.

    Uses a short ResNet-18-GN checkpoint and reduced horizons only for software qualification.
    The paid completion retains the frozen research CORE and prelocked production candidate.
    """
    global CORE,CONFIG_HASH
    old_core=CORE;old_config=CONFIG_HASH
    try:
        d=asdict(CORE);d['horizons']=(0,1);d['context_horizons']=(1,);d['primary_horizon']=1;d['target_positions']=(0.0,)
        CORE=CoreConfig(**d)
        CONFIG_HASH=hashlib.sha256(json.dumps(_current_protocol_payload(),sort_keys=True,separators=(',',':')).encode()).hexdigest()
        suite=make_suite(SuiteSpec(train_n=128,direction_n=16,calibration_n=16,response_n=16,evaluation_n=32,sealed_n=32,
                                   seed=stable_seed('RC13C_nonzero_q_replay_qa')))
        rep=99881
        task,_=make_task_variant(suite,16,4,0.0,'fixedK',rep)
        ts=TrainSpec(steps=4,batch_size=32,lr=CORE.lr,weight_decay=CORE.weight_decay,beta1=CORE.adam_beta1,beta2=CORE.adam_beta2,
            eps=CORE.adam_eps,amsgrad=CORE.adam_amsgrad,maximize=CORE.adam_maximize,foreach=CORE.adam_foreach,
            capturable=CORE.adam_capturable,differentiable=CORE.adam_differentiable,fused=CORE.adam_fused)
        bank,sch,dt,_=build_checkpoint_bank(task,'resnet',1,rep,ts);rec=bank[int(round(CORE.recipient_frac*ts.steps))]
        m=build_model('resnet',4,1,0,in_ch=1)
        accepted=None
        try:
            for c in range(1,RC13C_Q_MAX_CANDIDATES):
                # Generic geometry equivalent to production prequalification, but for the small QA model.
                pairs=target_panel_positions(m,CORE.target_positions);noise_cosines={};noise_ids={};gmax=0.0
                for target,tpos,tactual in pairs:
                    vec=[];ids={}
                    for qrep in (0,1,2):
                        did=_rc13c_noise_direction_id('resnet',1,rep,rec['step'],float(tpos),qrep,c)
                        v=unit_direction(m,target,'noise',None,None,did).detach().reshape(-1).float()
                        vec.append(v);ids[str(qrep)]=int(did)
                    pv={}
                    for i,j,label in ((0,1,'A_T'),(0,2,'A_B'),(1,2,'T_B')):
                        cc=float(torch.dot(vec[i],vec[j])/(torch.linalg.vector_norm(vec[i])*torch.linalg.vector_norm(vec[j])+1e-30))
                        pv[label]=cc;gmax=max(gmax,abs(cc))
                    noise_cosines[str(target)]=pv;noise_ids[str(target)]=ids
                if gmax<=CORE.q_noise_max_abs_cosine:
                    accepted={'candidate':c,'accepted_candidate':c,'noise_max_abs_cosine':float(gmax),
                              'noise_pairwise_cosines':noise_cosines,'noise_direction_ids':noise_ids}
                    break
        finally:
            del m
        if accepted is None:raise RuntimeError('RC13C nonzero-candidate QA found no valid candidate.')
        qb=build_q_bank(task,dt,'resnet',1,rep,ts,rec,q_plan_entry=accepted)
        if int(qb['q_candidate_index'])<=0 or int(qb['q_candidate_index'])!=int(accepted['accepted_candidate']):
            raise RuntimeError('RC13C nonzero-candidate QA failed candidate-index replay.')
        if qb['noise_direction_ids']!=accepted['noise_direction_ids']:
            raise RuntimeError('RC13C nonzero-candidate QA failed direction-ID replay.')
        return {'candidate':int(qb['q_candidate_index']),'noise_max_abs_cosine':float(qb['noise_max_abs_cosine']),
                'q_bank_sha256':str(qb['q_bank_sha256']),'passed':True}
    finally:
        CORE=old_core;CONFIG_HASH=old_config
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()


def run_rc13c_zero_hour_integrity():
    _assert_runtime_integrity()
    # Core structural semantics remain the frozen RC13B semantics.
    cols=[]
    for k,rps in [('noise',[0,1,2]),('gradient',[0]),('orthogonal',[0]),('optimizer_reset',[0])]:
        for rp in rps:
            for pos in CORE.target_positions[:1]:
                for sign in ((-1,1) if k in SIGNED_KINDS else (0,)):
                    for h in CORE.horizons:
                        for met in CORE.primary_response_metrics:cols.append((k,pos,sign,rp,h,met))
    qa,qb,qt=s6_query_partition(cols)
    if set(qa)&set(qb) or set(qa)&set(qt) or set(qb)&set(qt):raise RuntimeError('RC13C Q partition overlap.')
    ys=StandardScaler().fit(np.array([[1.,2.],[3.,6.],[5.,10.]]))
    if np.allclose(ys.transform(np.zeros((1,2))),0):raise RuntimeError('Physical-zero semantic regression.')

    source=_locate_rc13b_source_zip()
    parent=validate_rc13b_source_zip(source)
    # This call freezes the plan before any parent Parquet read.
    qplan=prequalify_and_lock_rc13c_q_plan(parent)
    if not bool(qplan.get('created_before_parent_confirmation_deserialization',False)):
        raise RuntimeError('RC13C Q-plan lock ordering assertion failed.')
    # The original failure must be repaired prospectively, not threshold-relaxed.
    if float(qplan['threshold'])!=float(CORE.q_noise_max_abs_cosine) or float(qplan['threshold'])!=0.15:
        raise RuntimeError('RC13C changed the frozen Q-independence threshold.')
    if int(qplan['replicates']['3127']['accepted_candidate'])<=0:
        raise RuntimeError('RC13C 3127 did not require a prospective replacement candidate as expected.')
    for rep in RC13B_SEALED_CONFIRMATION:
        if int(qplan['replicates'][str(rep)]['accepted_candidate'])!=0:
            raise RuntimeError(f'RC13C sealed rep {rep} unexpectedly changed Q candidate.')

    # Exercise both ordinary candidate-0 and explicit nonzero-candidate replay before paid work.
    writer=_real_branch_writer_qa()
    nonzero_writer=_rc13c_nonzero_q_candidate_replay_qa()
    _assert_runtime_integrity()
    out={'protocol_hash':PROTOCOL_HASH,'parent_zip_sha256':RC13B_SOURCE_ZIP_SHA256,
         'parent_bundle_verified':True,'sealed_parent_reps':len(RC13B_SEALED_CONFIRMATION),
         'q_plan_sha256':qplan['q_plan_sha256'],
         'completion_candidates':{str(r):int(qplan['replicates'][str(r)]['accepted_candidate']) for r in RC13C_COMPLETION_REPLICATES},
         'real_branch_writer':writer,'nonzero_q_candidate_replay':nonzero_writer,'passed':True}
    atomic_json(BASE/'manifests'/'rc13c_integrity_tests.json',out)
    print('RC13C ZERO-HOUR INTEGRITY PASSED',json.dumps(out,indent=2))
    return source,parent,qplan,out


## Launch policy and completion workload

RC13C is a **completion-only** campaign. It does not rerun discovery or the 27 already committed confirmation replicates.

Expected new paid work:

- 3 missing confirmation tasks: 3127–3129;
- 7 branches per task;
- 264 physical response rows per branch;
- **5,544 new response rows**;
- approximately **16,620 counted optimizer updates** (1,500 base-training + 15,120 Q-continuation updates), excluding calibration forward passes.

After the three tasks are durably committed, RC13C assembles the complete 55,440-row confirmation atlas and evaluates the frozen RC13B H6-B/H6-W/H6-A analysis exactly once.


In [ ]:
# 7. RC13C completion workload and direct launch

def rc13c_workload_contract():
    branches=1+len(CORE.confirmation_history_doses)+len(CORE.random_a_doses)+len(CORE.random_b_doses)
    q_per_target=(2*3)+2+2+1
    rows_per_branch=len(CORE.target_positions)*q_per_target*len(CORE.horizons)
    new_rows=len(RC13C_COMPLETION_REPLICATES)*branches*rows_per_branch
    # One untreated control continuation plus every physical query continuation advances to max(horizon).
    signed_queries_per_target=(2*3)+2+2+1
    q_updates_per_branch=int(max(CORE.horizons)*(1+len(CORE.target_positions)*signed_queries_per_target))
    out={
      'protocol_hash':PROTOCOL_HASH,'profile':PROFILE,
      'parent_sealed_confirmation_replicates':len(RC13B_SEALED_CONFIRMATION),
      'new_completion_replicates':len(RC13C_COMPLETION_REPLICATES),
      'completion_ids':list(RC13C_COMPLETION_REPLICATES),
      'branches_per_completion_replicate':branches,'rows_per_branch':rows_per_branch,
      'new_response_rows':new_rows,'final_confirmation_rows':30*branches*rows_per_branch,
      'new_base_training_updates':len(RC13C_COMPLETION_REPLICATES)*CORE.train_steps,
      'new_q_continuation_updates':len(RC13C_COMPLETION_REPLICATES)*branches*q_updates_per_branch,
      'new_total_counted_optimizer_updates':len(RC13C_COMPLETION_REPLICATES)*CORE.train_steps+len(RC13C_COMPLETION_REPLICATES)*branches*q_updates_per_branch,
      'exclusions':'Calibration forward passes are not counted. Parent 3100-3126 compute is preserved, not rerun.'
    }
    atomic_json(BASE/'manifests'/'rc13c_workload_contract.json',out);return out

def launch_rc13c_safely(source,parent):
    try:return run_rc13c_completion_campaign(source,parent)
    except ScientificGateStop as e:
        payload={'status':'scientific_stop_before_complete_claim_evaluation','claims_evaluated':False,
                 'claims':{'H6_B_between_task_predictive_phenotype':None,'H6_W_within_task_causal_state':None,'H6_A_optimizer_anisotropy':None},
                 'message':str(e),'protocol_hash':PROTOCOL_HASH,'implementation_revision':IMPLEMENTATION_REVISION,'time':time.time()}
        atomic_json(BASE/'artifacts'/'STAGE06_RC13C_SCIENTIFIC_STOP.json',payload);_durability_barrier()
        print('RC13C SCIENTIFIC GATE STOP:',e)
        try:final_export(payload)
        except Exception as export_error:print('RC13C scientific-stop export warning:',type(export_error).__name__,export_error)
        return payload
    except Exception as e:
        payload={'status':'software_failure','type':type(e).__name__,'message':str(e),
                 'protocol_hash':PROTOCOL_HASH,'implementation_revision':IMPLEMENTATION_REVISION,'time':time.time()}
        try:atomic_json(BASE/'artifacts'/'STAGE06_RC13C_SOFTWARE_FAILURE.json',payload);_durability_barrier()
        except Exception:pass
        print('RC13C SOFTWARE/INFRASTRUCTURE FAILURE:',type(e).__name__,e);raise

# Bind every helper before source validation / Q-plan generation / paid work.
SCIENTIFIC_CODE_MANIFEST=_scientific_code_manifest()
print('SCIENTIFIC_CODE_HASH',SCIENTIFIC_CODE_MANIFEST['campaign_code_hash'])

SOURCE_ZIP,PARENT_AUDIT,Q_PLAN,INTEGRITY=run_rc13c_zero_hour_integrity()
_assert_runtime_integrity()
print('RC13C QA runtime protocol/code/determinism restoration PASS')

WORKLOAD_CONTRACT=rc13c_workload_contract()
print('RC13C WORKLOAD CONTRACT',json.dumps(WORKLOAD_CONTRACT,indent=2))

RUN_RC13C_NOW=bool(int(os.environ.get('NEXUS_RUN_RC13C','1' if (PROFILE=='research' and torch.cuda.is_available()) else '0')))
if RUN_RC13C_NOW:
    RC13C_RESULT=launch_rc13c_safely(SOURCE_ZIP,PARENT_AUDIT)
else:
    print('RC13C paid completion not launched. Parent verification and Q-plan lock are complete.')
